<a href="https://colab.research.google.com/github/cleophasmashiri/ai-jupter-notebooks/blob/main/llm_from_scratch/gpt_dev_tutorial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Build a Large Language Model (From Scratch) — Tutorial Edition

This is a fill-in-the-blank companion notebook to Sebastian Raschka's book **_Build a Large Language Model (From Scratch)_**. It follows the book chapter by chapter — same class names, same function signatures, same variable names — so you can read the book and this notebook side by side.

Every code cell that introduces a new piece of the book's implementation has been turned into an exercise: the surrounding scaffolding is filled in, the core logic is replaced with `...`, and a test cell right after checks your implementation against the behavior described in the book.

If you've worked through [`gpt_dev_tutorial.ipynb`](https://colab.research.google.com/github/cleophasmashiri/ai-jupter-notebooks/blob/main/gpt_dev_tutorial.ipynb) (the Karpathy "let's build GPT" notebook) some of this will look familiar — both books arrive at the same decoder-only Transformer. This notebook is more methodical about the software-engineering side: a real `Dataset`/`DataLoader` pipeline, a `GPTModel` class you can load OpenAI's actual GPT-2 weights into, and two full fine-tuning pipelines (classification, instruction-following) on top of it.

### How this notebook works

1. Cells come in groups: **instructions** (markdown) → **your code** (with `...` blanks) → **tests** (a `%%ipytest` cell) → a collapsed **"Show solution"** block.
2. Fill in the blanks in the code cell, run it, then run the test cell immediately after.
3. A green pytest summary (`X passed`) means your implementation matches the book's; move on. A failure tells you which behavior is wrong.
4. Stuck? Every exercise has a "Show solution" block right after its test cell — click to expand it, not before you've genuinely tried.
5. `# Understanding X()` cells are reference material — read them when a call is unfamiliar, skip them if it isn't.
6. Run cells top to bottom — later exercises depend on variables and classes defined by earlier ones, exactly like the book's running example does.

### Gamify mode (optional)

Every test cell is worth points (1 per `def test_...`, 200 total across the notebook). Click **"Mark passed (+N)"** under a test cell once its tests are actually green -- this is honor-system, not verified automatically, so only claim it once you've genuinely got it working. Opening a **"Show solution"** block costs points equal to that exercise's test count, whether or not you'd already claimed them, since peeking is peeking regardless of what you'd already earned.

**Platform note:** the live score badge below only updates in classic Jupyter, JupyterLab, and VS Code's notebook viewer, which render a notebook as one continuous page. Google Colab sandboxes each cell's output in its own iframe, and GitHub's notebook preview strips JavaScript entirely (it's a static page) -- in both, the buttons and reveals still work individually, they just can't reach back to update a shared badge elsewhere on the page. This is cosmetic only: nothing about following the exercises depends on the score.

<div id="gamify-badge" style="display:inline-block;padding:10px 18px;border:1.5px solid #2563eb;border-radius:10px;background:#eff6ff;font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Helvetica,Arial,sans-serif;font-size:15px;color:#1e3a8a;font-weight:600">
Score: <span id="gamify-score">0</span> / <span id="gamify-total">200</span> (<span id="gamify-pct">0.0</span>%)
</div>

<script>
window.gamifyState = window.gamifyState || {score: 0, total: 200, claimed: new Set(), revealed: new Set()};
window.gamifyUpdate = function() {
  var s = document.getElementById("gamify-score");
  var p = document.getElementById("gamify-pct");
  if (s) { s.textContent = window.gamifyState.score; }
  if (p) { p.textContent = (100 * window.gamifyState.score / window.gamifyState.total).toFixed(1); }
};
window.gamifyClaim = function(id, n, btn) {
  if (window.gamifyState.claimed.has(id)) { return; }
  window.gamifyState.claimed.add(id);
  window.gamifyState.score += n;
  window.gamifyUpdate();
  if (btn) { btn.textContent = "+" + n + " claimed"; btn.disabled = true; btn.style.opacity = "0.55"; btn.style.cursor = "default"; }
};
window.gamifyReveal = function(detailsEl, id, n) {
  if (!detailsEl.open) { return; }
  if (window.gamifyState.revealed.has(id)) { return; }
  window.gamifyState.revealed.add(id);
  window.gamifyState.score -= n;
  window.gamifyUpdate();
};
window.gamifyUpdate();
</script>

### Reset — clear all your solutions

Every exercise cell you type into stays part of *this* file the moment you run it -- there's no autosave-to-blank, and no built-in Jupyter/Colab button that clears only the exercise cells and leaves everything else alone. The reliable way to get a completely fresh copy (blanks restored, gamify score back to zero since it's just in-browser state anyway): run the cell below. It downloads whatever's currently on GitHub to a new file next to this one -- open *that* file to start over. Nothing in this notebook is modified, so you can keep this copy around (e.g. for its "Show solution" reveals) while you redo the exercises in the fresh one.

In [ ]:
import urllib.request

RESET_URL = "https://raw.githubusercontent.com/cleophasmashiri/ai-jupter-notebooks/main/llm_from_scratch/gpt_dev_tutorial.ipynb"
RESET_FILENAME = "gpt_dev_tutorial_fresh.ipynb"

urllib.request.urlretrieve(RESET_URL, RESET_FILENAME)
print(f"Downloaded a fresh, all-blanks copy as '{RESET_FILENAME}' in the current directory.")
print()
print("Open THAT file to actually start over -- this file is left untouched:")
print("  - Colab: File > Open notebook > Upload, and pick the downloaded file")
print("           (or File > Revert notebook, if you opened this one straight from GitHub)")
print("  - Jupyter/JupyterLab: open it from the file browser, or replace this file with it")
print("           and reload the page")
print("  - VS Code: open it from the Explorer, or replace this file with it and reopen the tab")

### Roadmap — chapters 2 through 7, plus appendices A, D, and E

Chapter 1 ("Understanding large language models") is scene-setting with no code, so this notebook starts at chapter 2. From there, each chapter builds directly on the last:

1. **Ch. 2 — Working with text data**: turn raw text into token ID tensors a neural net can consume, with a real `Dataset`/`DataLoader` pipeline.
2. **Ch. 3 — Coding attention mechanisms**: derive self-attention from a plain dot product up to a batched, causal, multi-head `nn.Module`.
3. **Ch. 4 — Implementing a GPT model from scratch**: assemble `LayerNorm`, `GELU`, `FeedForward`, and the multi-head attention from ch. 3 into a full `GPTModel`, and generate text from it (untrained).
4. **Ch. 5 — Pretraining on unlabeled data**: define the loss, write the training loop, and add temperature/top-k sampling so generation stops being deterministic.
5. **Ch. 6 — Fine-tuning for classification**: repurpose a pretrained `GPTModel` as a spam classifier by swapping its output head.
6. **Ch. 7 — Fine-tuning to follow instructions**: format instruction/response pairs, mask padding out of the loss, and fine-tune the same model to follow Alpaca-style instructions.

Three appendices follow chapter 7, treated here as chapters in their own right:

7. **Appendix A — Introduction to PyTorch**: the tensor/autograd/`nn.Module`/`Dataset`/training-loop fundamentals every chapter above already relied on, named and exercised explicitly.
8. **Appendix D — Bells and whistles for the training loop**: learning rate warmup, cosine decay, and gradient clipping, layered onto chapter 5's training loop.
9. **Appendix E — Parameter-efficient fine-tuning with LoRA**: fine-tune orders of magnitude fewer parameters than chapters 6–7's freeze-and-unfreeze approach, by learning a low-rank approximation of each weight update instead of touching the weights directly.

Appendices B (references) and C (exercise solutions) aren't reproduced here — neither has code to exercise.

## Chapter 1 (background) — Understanding large language models

No code in this chapter, but three ideas from it are worth carrying forward explicitly since every later chapter assumes them.

**What "pretraining then fine-tuning" means.** An LLM first learns general language competence from a *next-word prediction* task over a huge, unlabeled text corpus (**pretraining**) — this alone produces a *base model* that can complete text plausibly but doesn't reliably follow instructions or perform a specific task. That base model is then adapted with a much smaller, labeled or curated dataset for a specific purpose (**fine-tuning**). This notebook builds one small base model from scratch (chapters 2–5) and fine-tunes copies of it two different ways (chapters 6–7) — mirroring exactly this two-stage recipe, just at a scale that runs on a laptop instead of a data center.

**Why next-word prediction is enough to bootstrap everything else.** Predicting the next word doesn't require any human-labeled data — the label at every position is just "whatever word actually came next" in real text, so *any* text corpus is automatically a training set (this is **self-supervised learning**). Despite being a narrow-sounding task, doing it well at scale forces a model to implicitly learn grammar, facts, reasoning patterns, and style — capabilities the book's chapter 6/7 fine-tuning steps then redirect toward specific ends, rather than building from nothing.

**GPT is a decoder-only Transformer.** The original 2017 Transformer (*Attention Is All You Need*) has two halves: an **encoder** (reads a full input, bidirectionally) and a **decoder** (generates output one token at a time, autoregressively, only allowed to look backward). GPT keeps only the decoder half — appropriate for a model whose entire job is "given everything so far, predict what comes next."

<img src="https://raw.githubusercontent.com/cleophasmashiri/ai-jupter-notebooks/main/llm_from_scratch/images/01-encoder-decoder-transformer-vs-gpt-decoder-only.png" alt="Encoder-decoder Transformer vs. GPT decoder-only" style="max-width:100%;height:auto;display:block;margin:1em auto" width="652" height="362"/>

Chapter 3 builds the causal-attention mechanism this decoder-only design depends on; chapter 4 stacks it into the full architecture pictured above as one `GPTModel` class.

**The three-stage build, as a roadmap.** The book returns to this figure at the start of every chapter — it's worth having once, up front, as a map of where each chapter's code fits:

<img src="https://raw.githubusercontent.com/cleophasmashiri/ai-jupter-notebooks/main/llm_from_scratch/images/02-three-stage-llm-build-roadmap.png" alt="Three-stage LLM build roadmap" style="max-width:100%;height:auto;display:block;margin:1em auto" width="812" height="472"/>

Every remaining chapter in this notebook is one box (or a few boxes) in this picture.

In [ ]:
!pip install -q ipytest tiktoken
import ipytest
ipytest.autoconfig()

## Chapter 2 — Working with text data

We start with the same dataset the book uses throughout chapters 2–5: Edith Wharton's short story *"The Verdict"* (public domain, ~20k characters — small enough to iterate on quickly, large enough to be a real tokenization/data-loading exercise).

In [ ]:
!wget -q https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/main/ch02/01_main-chapter-code/the-verdict.txt
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()
print("Total number of characters:", len(raw_text))
print(raw_text[:99])

### 2.1 — Understanding word embeddings

Everything a neural net computes is arithmetic on vectors of numbers — but text is discrete symbols. **Embedding** is the general name for "map a discrete thing (a word, a token, a category) to a continuous vector," and the whole rest of this chapter is the concrete pipeline that does it for GPT: raw text → tokens → token IDs → embedding vectors.

The book contrasts a few embedding schemes to place GPT's choice in context:
- **word2vec-style embeddings** are trained *standalone*, on a separate objective (predict a word from its neighbors), then plugged into whatever downstream model needs them. Nearby vectors end up meaning similar things (`"cat"` close to `"dog"`), a well-known and useful property.
- **GPT's token embeddings**, by contrast, are just another `nn.Embedding` layer *inside* the model, trained end-to-end alongside every other weight, purely to minimize next-token-prediction loss — no separate embedding-training stage, no separate objective. This is simpler to implement (one training loop, not two) and lets the embeddings specialize for exactly the task the rest of the network needs them for.
- Embeddings can have any number of dimensions — small models might use 256, GPT-3 uses 12,288. More dimensions can capture more nuanced relationships, at the cost of more parameters and compute; the book uses small values (e.g. 256) for runnable examples, real GPT-2 uses 768–1600 depending on size.

The full pipeline this chapter builds, end to end:

<img src="https://raw.githubusercontent.com/cleophasmashiri/ai-jupter-notebooks/main/llm_from_scratch/images/03-tokenization-to-embedding-pipeline.png" alt="Tokenization to embedding pipeline" style="max-width:100%;height:auto;display:block;margin:1em auto" width="1072" height="132"/>

Exercises 1–4 below build the left half (text → token IDs) two ways: first a from-scratch word-level tokenizer (to see exactly what a tokenizer does), then GPT-2's real byte pair encoding tokenizer. Exercises 6–8 build the right half (token IDs → input embeddings).

### 2.2 — Tokenizing text

The first step of any tokenizer is splitting raw text into a list of substrings — words and punctuation. The book builds this up with Python's `re.split()`, first splitting on whitespace, then refining the regex to also split out commas, periods, and other punctuation as their own tokens (so `"Hello, world."` becomes `["Hello", ",", "world", "."]`, not `["Hello,", "world."]`).

### Exercise 1 — A simple word tokenizer

Implement `preprocess`, a function that splits `text` into a list of tokens. Book's listing (section 2.2): split on `r'([,.:;?_!"()\']|--|\s)'`, then strip whitespace from each piece and drop empty strings.

In [ ]:
# TODO: implement this exercise
import re

def preprocess(text):
    result = ...  # split `text` on punctuation/whitespace, strip each piece, drop empties
    return result

preprocessed = preprocess(raw_text)
print(len(preprocessed))
print(preprocessed[:30])

In [ ]:
%%ipytest -qq

def test_preprocess_basic():
    assert preprocess("Hello, world. Is this-- a test?") == [
        "Hello", ",", "world", ".", "Is", "this", "--", "a", "test", "?"
    ]

def test_preprocess_drops_whitespace_only_tokens():
    assert "" not in preprocess("Hello,   world.")
    assert " " not in preprocess("Hello,   world.")

def test_preprocessed_full_text():
    assert len(preprocessed) == 4690
    assert preprocessed[:10] == [
        "I", "HAD", "always", "thought", "Jack", "Gisburn", "rather", "a", "cheap", "genius"
    ]

<button id="claim-ex1" onclick="window.gamifyClaim('ex1', 3, this)" style="padding:6px 14px;border-radius:8px;border:1.5px solid #16a34a;background:#f0fdf4;color:#15803d;font-weight:600;font-size:13px;cursor:pointer;font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Helvetica,Arial,sans-serif">
✅ Mark passed (+3 pts, 3 tests)
</button>

<details ontoggle="window.gamifyReveal(this, 'ex1', 3)"><summary>Show solution (3 tests, −3 pts if opened)</summary>

```python
import re

def preprocess(text):
    result = re.split(r'([,.:;?_!"()\']|--|\s)', text)
    result = [item.strip() for item in result if item.strip()]
    return result

preprocessed = preprocess(raw_text)
print(len(preprocessed))
print(preprocessed[:30])
```

</details>

### 2.3 — Converting tokens into token IDs

Next we build a vocabulary: every unique token, sorted alphabetically, mapped to an integer id. This `str_to_int` mapping (the book calls it `vocab`) is exactly the `stoi` idea — a neural net can't consume strings, so every token needs a stable integer id before anything else can happen.

### Exercise 2 — Build the vocabulary

Build `vocab`, a dict mapping each unique token in `preprocessed` to an integer id, assigned in sorted order (so `vocab[sorted_unique_tokens[0]] == 0`, etc.).

In [ ]:
# TODO: implement this exercise
all_words = ...       # sorted list of unique tokens in `preprocessed`
vocab_size = ...      # how many unique tokens there are
vocab = ...            # dict: token -> integer id, assigned in sorted order

print(vocab_size)
print(list(vocab.items())[:5])

In [ ]:
%%ipytest -qq

def test_vocab_size():
    assert vocab_size == 1130, f"expected 1130 unique tokens in the verdict, got {vocab_size}"

def test_vocab_is_sorted_assignment():
    assert all_words == sorted(set(preprocessed))
    assert vocab["!"] == 0
    assert vocab["a"] == 115  # sanity check specific to this dataset's sorted vocabulary
    assert vocab[all_words[-1]] == vocab_size - 1

def test_vocab_covers_every_token():
    assert set(vocab.keys()) == set(preprocessed)
    assert len(vocab) == len(set(preprocessed))

<button id="claim-ex2" onclick="window.gamifyClaim('ex2', 3, this)" style="padding:6px 14px;border-radius:8px;border:1.5px solid #16a34a;background:#f0fdf4;color:#15803d;font-weight:600;font-size:13px;cursor:pointer;font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Helvetica,Arial,sans-serif">
✅ Mark passed (+3 pts, 3 tests)
</button>

<details ontoggle="window.gamifyReveal(this, 'ex2', 3)"><summary>Show solution (3 tests, −3 pts if opened)</summary>

```python
all_words = sorted(set(preprocessed))
vocab_size = len(all_words)
vocab = {token: integer for integer, token in enumerate(all_words)}

print(vocab_size)
print(list(vocab.items())[:5])
```

</details>

Note: `test_vocab_is_sorted_assignment` checks `vocab["a"] == 115` as a spot check specific to *this* dataset's sorted vocabulary — the exact index depends on Python's default string sort (uppercase before lowercase, so all-caps words like `"HAD"` sort before `"a"`), which is what `sorted()` gives you here.

### Exercise 3 — SimpleTokenizerV1

Wrap `vocab` in a tokenizer class with `encode`/`decode` methods, per the book's listing 2.3. `decode` also has to undo the tokenizer's habit of putting a space before every token, by removing whitespace immediately before punctuation.

In [ ]:
# TODO: implement this exercise
class SimpleTokenizerV1:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = ...  # inverse mapping: id -> token

    def encode(self, text):
        preprocessed = preprocess(text)
        ids = ...  # list of ids, one per token, via self.str_to_int
        return ids

    def decode(self, ids):
        text = ...  # " ".join the tokens for `ids` via self.int_to_str
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)  # remove space before punctuation
        return text

tokenizer = SimpleTokenizerV1(vocab)
sample_text = """"It\'s the last he painted, you know," Mrs. Gisburn said with pardonable pride."""
ids = tokenizer.encode(sample_text)
print(ids[:10])
print(tokenizer.decode(ids))

In [ ]:
%%ipytest -qq

def test_encode_decode_roundtrip():
    t = SimpleTokenizerV1(vocab)
    # avoid words with an internal apostrophe here: the tokenizer splits "It's" into
    # ["It", "'", "s"], and decode's punctuation-spacing fix only handles the space
    # *before* punctuation, so contractions don't round-trip exactly -- a known
    # limitation of this simple word-level tokenizer, not a bug to fix.
    text = "Jack Gisburn rather a cheap genius, you know, said with pardonable pride."
    ids = t.encode(text)
    assert all(isinstance(i, int) for i in ids)
    assert t.decode(ids) == text

def test_int_to_str_is_inverse():
    t = SimpleTokenizerV1(vocab)
    for tok, idx in list(vocab.items())[:20]:
        assert t.int_to_str[idx] == tok

def test_encode_matches_vocab_lookup():
    t = SimpleTokenizerV1(vocab)
    text = "Jack Gisburn rather a cheap genius"
    assert t.encode(text) == [vocab[w] for w in preprocess(text)]

<button id="claim-ex3" onclick="window.gamifyClaim('ex3', 3, this)" style="padding:6px 14px;border-radius:8px;border:1.5px solid #16a34a;background:#f0fdf4;color:#15803d;font-weight:600;font-size:13px;cursor:pointer;font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Helvetica,Arial,sans-serif">
✅ Mark passed (+3 pts, 3 tests)
</button>

<details ontoggle="window.gamifyReveal(this, 'ex3', 3)"><summary>Show solution (3 tests, −3 pts if opened)</summary>

```python
class SimpleTokenizerV1:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i: s for s, i in vocab.items()}

    def encode(self, text):
        preprocessed = preprocess(text)
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids

    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text

tokenizer = SimpleTokenizerV1(vocab)
sample_text = """"It's the last he painted, you know," Mrs. Gisburn said with pardonable pride."""
ids = tokenizer.encode(sample_text)
print(ids[:10])
print(tokenizer.decode(ids))
```

</details>

### 2.4 — Adding special context tokens

`SimpleTokenizerV1` breaks the moment it sees a word that isn't in `vocab` — `KeyError`. Real tokenizers need a way to represent "a token I've never seen" (`<|unk|>`) and, for training on multiple documents concatenated together, a way to mark document boundaries (`<|endoftext|>`) so the model doesn't learn spurious connections across unrelated texts.

### Exercise 4 — Extend the vocabulary and SimpleTokenizerV2

First extend `vocab` with the two special tokens, appended after all the existing ones (so their ids are `vocab_size` and `vocab_size + 1`). Then implement `SimpleTokenizerV2`: `encode` should replace any token not in `str_to_int` with `<|unk|>` before mapping to ids.

In [ ]:
# TODO: implement this exercise
all_tokens = ...  # sorted(all_words) plus ["<|endoftext|>", "<|unk|>"], in that order
vocab_v2 = ...    # dict: token -> id, assigned in order over `all_tokens`

class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i: s for s, i in vocab.items()}

    def encode(self, text):
        preprocessed = preprocess(text)
        preprocessed = ...  # replace any token not in self.str_to_int with "<|unk|>"
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids

    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text

tokenizer_v2 = SimpleTokenizerV2(vocab_v2)
text1 = "Hello, do you like tea?"
text2 = "In the sunlit terraces of the palace."
joined = " <|endoftext|> ".join((text1, text2))
print(tokenizer_v2.encode(joined))
print(tokenizer_v2.decode(tokenizer_v2.encode(joined)))

In [ ]:
%%ipytest -qq

def test_vocab_v2_size():
    assert len(vocab_v2) == vocab_size + 2

def test_special_tokens_appended_last():
    assert vocab_v2["<|endoftext|>"] == vocab_size
    assert vocab_v2["<|unk|>"] == vocab_size + 1

def test_unknown_word_maps_to_unk():
    t = SimpleTokenizerV2(vocab_v2)
    ids = t.encode("Hello, do you like tea?")
    assert vocab_v2["<|unk|>"] in ids
    assert "Hello" not in vocab_v2

def test_known_words_pass_through():
    t = SimpleTokenizerV2(vocab_v2)
    text = "Jack Gisburn rather a cheap genius"
    assert vocab_v2["<|unk|>"] not in t.encode(text)

<button id="claim-ex4" onclick="window.gamifyClaim('ex4', 4, this)" style="padding:6px 14px;border-radius:8px;border:1.5px solid #16a34a;background:#f0fdf4;color:#15803d;font-weight:600;font-size:13px;cursor:pointer;font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Helvetica,Arial,sans-serif">
✅ Mark passed (+4 pts, 4 tests)
</button>

<details ontoggle="window.gamifyReveal(this, 'ex4', 4)"><summary>Show solution (4 tests, −4 pts if opened)</summary>

```python
all_tokens = sorted(all_words) + ["<|endoftext|>", "<|unk|>"]
vocab_v2 = {token: integer for integer, token in enumerate(all_tokens)}

class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i: s for s, i in vocab.items()}

    def encode(self, text):
        preprocessed = preprocess(text)
        preprocessed = [item if item in self.str_to_int else "<|unk|>" for item in preprocessed]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids

    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text

tokenizer_v2 = SimpleTokenizerV2(vocab_v2)
text1 = "Hello, do you like tea?"
text2 = "In the sunlit terraces of the palace."
joined = " <|endoftext|> ".join((text1, text2))
print(tokenizer_v2.encode(joined))
print(tokenizer_v2.decode(tokenizer_v2.encode(joined)))
```

</details>

### 2.5 — Byte pair encoding

A word-level vocabulary like the one above still hits `<|unk|>` constantly on real text — any typo, rare name, or word outside the training corpus is unrepresentable. **Byte pair encoding (BPE)** solves this differently: it builds a vocabulary of frequently-occurring *subword* chunks (learned from a huge corpus), so any string — no matter how unusual — can be built out of some sequence of known chunks, down to individual bytes/characters as a fallback. This is what GPT-2, GPT-3, and GPT-4 actually use, so the book switches to it here and doesn't look back.

Rather than reimplement BPE's training algorithm from scratch, the book uses OpenAI's own `tiktoken` library, loaded with the exact GPT-2 vocabulary (50,257 tokens).

BPE is built bottom-up: start from individual bytes/characters, then repeatedly merge the *most frequently co-occurring* pair into a new single token, thousands of times, until the vocabulary reaches its target size (50,257 for GPT-2). Common whole words end up as single tokens; rare or unfamiliar words fall back to smaller, more common pieces:

<img src="https://raw.githubusercontent.com/cleophasmashiri/ai-jupter-notebooks/main/llm_from_scratch/images/04-byte-pair-encoding-common-words-vs-unfamiliar-word.png" alt="Byte pair encoding: common words vs. unfamiliar words" style="max-width:100%;height:auto;display:block;margin:1em auto" width="732" height="312"/>

No word is ever "unrepresentable" — in the worst case BPE just falls back to more, smaller pieces, which is exactly what eliminates the need for `<|unk|>`.

# Understanding `tiktoken.get_encoding()`

`tiktoken` is OpenAI's BPE tokenizer library. `tiktoken.get_encoding(name)` loads a pretrained BPE vocabulary by name — no training required, since the vocabulary was already learned once (by OpenAI, on a huge web-scale corpus) and shipped as a fixed table.

### Syntax
```python
tiktoken.get_encoding("gpt2")
```
Returns a tokenizer object with `.encode(text, allowed_special=set())` and `.decode(ids)` methods.

### Example
```python
import tiktoken
tokenizer = tiktoken.get_encoding("gpt2")
ids = tokenizer.encode("Akwirw ier", allowed_special={"<|endoftext|>"})
print(ids)
```
```
[33901, 86, 343, 86, 220, 959]
```
Note there's no `<|unk|>` handling needed at all — `allowed_special` only matters for tokens like `<|endoftext|>` that BPE treats specially rather than splitting into bytes; every other string, however unusual, always maps to *some* sequence of subword ids. Decoding a sub-slice of those ids independently (e.g. just `[86]`) reveals what the pieces actually are: BPE routinely splits unfamiliar words into fragments smaller than a whole word, sometimes down to individual letters, which is exactly the fallback that eliminates `<|unk|>`.

### In this notebook
```python
tokenizer = tiktoken.get_encoding("gpt2")
```
Every dataset/dataloader exercise from here on uses this `tokenizer` instead of `SimpleTokenizerV2` — same `.encode()`/`.decode()` shape, a much larger and more robust vocabulary (50,257 vs. ~1,130 tokens).

### Exercise 5 — Use the BPE tokenizer

Instantiate the GPT-2 BPE tokenizer from `tiktoken`, and encode/decode a piece of text that includes the `<|endoftext|>` special token (which requires passing `allowed_special`).

In [ ]:
# TODO: implement this exercise
import tiktoken

bpe_tokenizer = ...  # tiktoken.get_encoding for "gpt2"

text = (
    "Hello, do you like tea? <|endoftext|> In the sunlit terraces"
    " of someunknownPlace."
)
bpe_ids = ...  # encode `text`, allowing the "<|endoftext|>" special token
bpe_decoded = ...  # decode `bpe_ids` back to text

print(bpe_ids)
print(bpe_decoded)

In [ ]:
%%ipytest -qq

def test_bpe_tokenizer_type():
    assert bpe_tokenizer.__class__.__name__ == "Encoding"

def test_bpe_roundtrip():
    assert bpe_decoded == text

def test_bpe_ids_are_ints():
    assert all(isinstance(i, int) for i in bpe_ids)
    assert len(bpe_ids) > 10  # more tokens than words, since "someunknownPlace" splits into pieces

def test_bpe_handles_unknown_word_without_unk_token():
    unk_ids = bpe_tokenizer.encode("Akwirw ier")
    assert unk_ids == [33901, 86, 343, 86, 220, 959]

<button id="claim-ex5" onclick="window.gamifyClaim('ex5', 4, this)" style="padding:6px 14px;border-radius:8px;border:1.5px solid #16a34a;background:#f0fdf4;color:#15803d;font-weight:600;font-size:13px;cursor:pointer;font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Helvetica,Arial,sans-serif">
✅ Mark passed (+4 pts, 4 tests)
</button>

<details ontoggle="window.gamifyReveal(this, 'ex5', 4)"><summary>Show solution (4 tests, −4 pts if opened)</summary>

```python
import tiktoken

bpe_tokenizer = tiktoken.get_encoding("gpt2")

text = (
    "Hello, do you like tea? <|endoftext|> In the sunlit terraces"
    " of someunknownPlace."
)
bpe_ids = bpe_tokenizer.encode(text, allowed_special={"<|endoftext|>"})
bpe_decoded = bpe_tokenizer.decode(bpe_ids)

print(bpe_ids)
print(bpe_decoded)
```

</details>

### 2.6 — Data sampling with a sliding window

Training data for next-word prediction is generated by sliding a fixed-size window over the token stream: each window of `max_length` tokens is an input, and the same window shifted one position to the right is the target. `stride` controls how far the window moves each step — `stride == max_length` means no overlap between consecutive training examples; `stride < max_length` means examples overlap.

For a token stream `[40, 367, 2885, 1464, 1807, ...]` with `max_length=4, stride=1`, each window overlaps the last by all but one token:

<img src="https://raw.githubusercontent.com/cleophasmashiri/ai-jupter-notebooks/main/llm_from_scratch/images/05-sliding-window-input-target-shift.png" alt="Sliding window: input/target shift" style="max-width:100%;height:auto;display:block;margin:1em auto" width="652" height="232"/>

Every target is just the input shifted one position to the right — the same "predict the next token" framing every chapter from here on relies on, whether the "sequence" is raw pretraining text (this chapter), a formatted instruction (chapter 7), or anything else.

### Exercise 6 — GPTDatasetV1

Implement the book's `GPTDatasetV1` (listing 2.5), a PyTorch `Dataset` that pre-tokenizes `txt` once in `__init__`, then slices out every `(input_chunk, target_chunk)` pair via a sliding window of size `max_length` and step `stride`.

In [ ]:
# TODO: implement this exercise
import torch
from torch.utils.data import Dataset, DataLoader

class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []
        token_ids = tokenizer.encode(txt)

        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = ...   # token_ids[i : i+max_length]
            target_chunk = ...  # token_ids[i+1 : i+max_length+1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return ...  # number of (input, target) pairs

    def __getitem__(self, idx):
        return ...  # (self.input_ids[idx], self.target_ids[idx])

_ds = GPTDatasetV1(raw_text, bpe_tokenizer, max_length=4, stride=4)
print(len(_ds))
print(_ds[0])

In [ ]:
%%ipytest -qq

def test_dataset_pair_shapes():
    x0, y0 = _ds[0]
    assert x0.shape == (4,)
    assert y0.shape == (4,)

def test_dataset_target_is_shifted_input():
    x0, y0 = _ds[0]
    x1, y1 = _ds[1]
    # with stride == max_length, chunk 1's input starts where chunk 0's target ends... not quite:
    # instead verify the shift-by-one relationship within a single window using stride=1
    ds_overlap = GPTDatasetV1(raw_text, bpe_tokenizer, max_length=4, stride=1)
    xi, yi = ds_overlap[0]
    all_ids = bpe_tokenizer.encode(raw_text)
    assert xi.tolist() == all_ids[0:4]
    assert yi.tolist() == all_ids[1:5]

def test_dataset_length_matches_sliding_window_count():
    all_ids = bpe_tokenizer.encode(raw_text)
    expected = len(range(0, len(all_ids) - 4, 4))
    assert len(_ds) == expected

<button id="claim-ex6" onclick="window.gamifyClaim('ex6', 3, this)" style="padding:6px 14px;border-radius:8px;border:1.5px solid #16a34a;background:#f0fdf4;color:#15803d;font-weight:600;font-size:13px;cursor:pointer;font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Helvetica,Arial,sans-serif">
✅ Mark passed (+3 pts, 3 tests)
</button>

<details ontoggle="window.gamifyReveal(this, 'ex6', 3)"><summary>Show solution (3 tests, −3 pts if opened)</summary>

```python
import torch
from torch.utils.data import Dataset, DataLoader

class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []
        token_ids = tokenizer.encode(txt)

        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i+max_length]
            target_chunk = token_ids[i+1:i+max_length+1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return (self.input_ids[idx], self.target_ids[idx])

_ds = GPTDatasetV1(raw_text, bpe_tokenizer, max_length=4, stride=4)
print(len(_ds))
print(_ds[0])
```

</details>

### Exercise 7 — create_dataloader_v1

Wrap `GPTDatasetV1` in the book's `create_dataloader_v1` helper (listing 2.6): build the GPT-2 tokenizer, construct the dataset, and hand it to a PyTorch `DataLoader`.

In [ ]:
# TODO: implement this exercise
def create_dataloader_v1(txt, batch_size=4, max_length=256,
                          stride=128, shuffle=True, drop_last=True,
                          num_workers=0):
    tokenizer = ...  # tiktoken gpt2 encoding
    dataset = ...    # GPTDatasetV1(txt, tokenizer, max_length, stride)
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )
    return dataloader

dataloader = create_dataloader_v1(raw_text, batch_size=1, max_length=4, stride=1, shuffle=False)
data_iter = iter(dataloader)
first_batch = next(data_iter)
print(first_batch)

In [ ]:
%%ipytest -qq

def test_first_batch_shapes():
    x, y = first_batch
    assert x.shape == (1, 4)
    assert y.shape == (1, 4)

def test_first_batch_matches_dataset():
    all_ids = bpe_tokenizer.encode(raw_text)
    x, y = first_batch
    assert x[0].tolist() == all_ids[0:4]
    assert y[0].tolist() == all_ids[1:5]

def test_batched_dataloader_shapes():
    dl = create_dataloader_v1(raw_text, batch_size=8, max_length=4, stride=4, shuffle=False)
    x, y = next(iter(dl))
    assert x.shape == (8, 4)
    assert y.shape == (8, 4)

<button id="claim-ex7" onclick="window.gamifyClaim('ex7', 3, this)" style="padding:6px 14px;border-radius:8px;border:1.5px solid #16a34a;background:#f0fdf4;color:#15803d;font-weight:600;font-size:13px;cursor:pointer;font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Helvetica,Arial,sans-serif">
✅ Mark passed (+3 pts, 3 tests)
</button>

<details ontoggle="window.gamifyReveal(this, 'ex7', 3)"><summary>Show solution (3 tests, −3 pts if opened)</summary>

```python
def create_dataloader_v1(txt, batch_size=4, max_length=256,
                          stride=128, shuffle=True, drop_last=True,
                          num_workers=0):
    tokenizer = tiktoken.get_encoding("gpt2")
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )
    return dataloader

dataloader = create_dataloader_v1(raw_text, batch_size=1, max_length=4, stride=1, shuffle=False)
data_iter = iter(dataloader)
first_batch = next(data_iter)
print(first_batch)
```

</details>

### 2.7 & 2.8 — Token embeddings and positional encoding

A batch of token *ids* is still just integers — `nn.Embedding` turns each id into a learned dense vector (`token_embeddings`). But embeddings alone are **permutation-blind**: the same token id always produces the same vector no matter where it sits in the sequence, so `nn.Embedding` by itself throws away word order entirely. GPT fixes this with a second embedding table, `pos_embeddings`, indexed not by token id but by *position* (`0, 1, 2, ..., context_length-1`), and adds the two together — this is an **absolute positional embedding**, learned jointly with everything else during training.

### Exercise 8 — Combine token and positional embeddings

Given `token_embedding_layer` and `pos_embedding_layer` (both provided), compute `input_embeddings` for a batch `inputs` of token ids, shape `(batch_size, max_length)`.

In [ ]:
# TODO: implement this exercise
import torch.nn as nn

vocab_size_bpe = 50257
output_dim = 256
context_length = 4

torch.manual_seed(123)
token_embedding_layer = nn.Embedding(vocab_size_bpe, output_dim)
pos_embedding_layer = nn.Embedding(context_length, output_dim)

emb_dataloader = create_dataloader_v1(
    raw_text, batch_size=8, max_length=context_length, stride=context_length, shuffle=False
)
emb_inputs, emb_targets = next(iter(emb_dataloader))

token_embeddings = ...     # token_embedding_layer(emb_inputs) -> (8, 4, 256)
pos_embeddings = ...       # pos_embedding_layer(torch.arange(context_length)) -> (4, 256)
input_embeddings = ...     # token_embeddings + pos_embeddings, broadcasting over the batch

print(input_embeddings.shape)

In [ ]:
%%ipytest -qq

def test_token_embeddings_shape():
    assert token_embeddings.shape == (8, 4, 256)

def test_pos_embeddings_shape():
    assert pos_embeddings.shape == (4, 256)

def test_input_embeddings_shape_and_value():
    assert input_embeddings.shape == (8, 4, 256)
    expected = token_embeddings + pos_embeddings
    assert torch.allclose(input_embeddings, expected)

def test_positional_embedding_breaks_permutation_invariance():
    # two identical tokens at different positions must get different final embeddings
    same_id = emb_inputs[0, 0].item()
    idx_matches = (emb_inputs == same_id).nonzero()
    if idx_matches.shape[0] >= 2:
        (b1, t1), (b2, t2) = idx_matches[0].tolist(), idx_matches[1].tolist()
        if t1 != t2:
            assert not torch.allclose(input_embeddings[b1, t1], input_embeddings[b2, t2])

<button id="claim-ex8" onclick="window.gamifyClaim('ex8', 4, this)" style="padding:6px 14px;border-radius:8px;border:1.5px solid #16a34a;background:#f0fdf4;color:#15803d;font-weight:600;font-size:13px;cursor:pointer;font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Helvetica,Arial,sans-serif">
✅ Mark passed (+4 pts, 4 tests)
</button>

<details ontoggle="window.gamifyReveal(this, 'ex8', 4)"><summary>Show solution (4 tests, −4 pts if opened)</summary>

```python
import torch.nn as nn

vocab_size_bpe = 50257
output_dim = 256
context_length = 4

torch.manual_seed(123)
token_embedding_layer = nn.Embedding(vocab_size_bpe, output_dim)
pos_embedding_layer = nn.Embedding(context_length, output_dim)

emb_dataloader = create_dataloader_v1(
    raw_text, batch_size=8, max_length=context_length, stride=context_length, shuffle=False
)
emb_inputs, emb_targets = next(iter(emb_dataloader))

token_embeddings = token_embedding_layer(emb_inputs)
pos_embeddings = pos_embedding_layer(torch.arange(context_length))
input_embeddings = token_embeddings + pos_embeddings

print(input_embeddings.shape)
```

</details>

### Checkpoint: chapter 2 — working with text data

1. Why does BPE (`tiktoken`) never need an `<|unk|>` token, unlike `SimpleTokenizerV1`/`V2`?

<details><summary>Show answer</summary>

BPE's vocabulary is built bottom-up from bytes/characters, so every possible string can always be represented as *some* sequence of known subword pieces — worst case, it falls back to spelling a word out piece by piece (or even byte by byte). A word-level vocabulary like `SimpleTokenizerV2`'s only has entries for whole words it happened to see during vocabulary construction, so anything else is unrepresentable and needs a catch-all `<|unk|>`.

</details>

2. In `GPTDatasetV1`, why is `target_chunk` just `input_chunk` shifted by one position, rather than something else?

<details><summary>Show answer</summary>

The training task is next-token prediction: at every position `t` in the input, the label is "what token actually comes next," i.e. the token at position `t+1`. Shifting the whole window by one position produces exactly that label for every position simultaneously, so one `(input_chunk, target_chunk)` pair supervises `max_length` separate next-token predictions per training example.

</details>

3. Why do `token_embedding_layer` and `pos_embedding_layer` get *added* together rather than concatenated?

<details><summary>Show answer</summary>

Addition keeps the embedding dimension fixed at `output_dim` (same width the rest of the network expects) while still letting the model recover "what" and "where" separately if it needs to — the two embedding tables are learned independently and can occupy different subspaces of the same vector. Concatenation would double the width and require every downstream layer to be sized differently just to accommodate position information.

</details>

If any of these aren't automatic, revisit the corresponding section above before continuing — chapter 3 builds directly on `input_embeddings`.

## Chapter 3 — Coding attention mechanisms

### 3.1 — The problem with modeling long sequences

Before attention, sequence-to-sequence tasks (e.g. translation) were dominated by **RNN encoder-decoder** architectures: an RNN reads the entire input sequence one token at a time, compressing everything it's seen into a single fixed-size **hidden state** vector, then a second RNN unpacks that one vector back into an output sequence, one token at a time.

The flaw is structural, not a matter of tuning: no matter how long the input is, the encoder has to squeeze it into that *same-sized* hidden vector — a 5-word sentence and a 500-word paragraph both get compressed to one vector of, say, 512 numbers. Information from early in a long sequence increasingly gets overwritten or diluted by the time the encoder finishes reading, so the decoder ends up working from a lossy summary, not the full input.

<img src="https://raw.githubusercontent.com/cleophasmashiri/ai-jupter-notebooks/main/llm_from_scratch/images/06-rnn-encoder-decoder-bottleneck.png" alt="RNN encoder-decoder bottleneck" style="max-width:100%;height:auto;display:block;margin:1em auto" width="652" height="262"/>

### 3.2 — Capturing data dependencies with attention mechanisms

The fix, introduced for machine translation in 2014 (Bahdanau et al.) before *Attention Is All You Need* generalized it: instead of forcing the whole input through one bottleneck vector, let the decoder **look back at every input position** at every output step, and *learn how much to weight each one* for the token it's currently generating. Different output words end up attending to different input words — exactly the intuition "self-attention" (this chapter) and "cross-attention" (used in encoder-decoder Transformers, briefly noted later in this chapter) both formalize.

<img src="https://raw.githubusercontent.com/cleophasmashiri/ai-jupter-notebooks/main/llm_from_scratch/images/07-attention-weighting-every-input-position.png" alt="Attention: weighting every input position" style="max-width:100%;height:auto;display:block;margin:1em auto" width="652" height="262"/>

`Attention Is All You Need` (2017) took this one step further and asked: what if attention isn't just a *helper* bolted onto an RNN, but the *entire* mechanism, with no recurrence at all? That's the **Transformer**, and self-attention — attention from a sequence to *itself*, rather than from a decoder to a separate encoder — is the mechanism the rest of this chapter builds from scratch.

### 3.3 — Attending to different parts of the input with self-attention

The book derives self-attention in the same order Karpathy's "let's build GPT" does — starting from a plain, weight-free dot product, then adding trainable weights, then causality, then multiple heads — but on a small worked example you can trace by hand: a 6-token sentence embedded (by hand, not learned) into 3-dim vectors.

```python
inputs = torch.tensor(
  [[0.43, 0.15, 0.89],   # Your      (x^1)
   [0.55, 0.87, 0.66],   # journey   (x^2)
   [0.57, 0.85, 0.64],   # starts    (x^3)
   [0.22, 0.58, 0.33],   # with      (x^4)
   [0.77, 0.25, 0.10],   # one       (x^5)
   [0.05, 0.80, 0.55]]   # step      (x^6)
)
```

This tiny example runs in your head as easily as in PyTorch, which is exactly the point — every exercise below scales up unchanged to real, learned embeddings of dimension 768+.

In [ ]:
import torch
import torch.nn as nn

inputs = torch.tensor(
  [[0.43, 0.15, 0.89],
   [0.55, 0.87, 0.66],
   [0.57, 0.85, 0.64],
   [0.22, 0.58, 0.33],
   [0.77, 0.25, 0.10],
   [0.05, 0.80, 0.55]]
)
inputs.shape

### 3.3.1 — A simple self-attention mechanism without trainable weights

For a single query token (say `x^2`, "journey"), attention weights measure how relevant every token — including itself — is to it. The book computes this the most direct way possible: a dot product between the query and every input vector, then a `softmax` to turn the scores into a probability distribution.

### Exercise 1 — Attention weights for a single query

Using `query = inputs[1]`, compute `attn_scores_2` (dot product of `query` against every row of `inputs`), `attn_weights_2` (softmax of the scores), and `context_vec_2` (the weighted sum of `inputs` using those weights).

In [ ]:
# TODO: implement this exercise
query = inputs[1]

attn_scores_2 = torch.empty(inputs.shape[0])
for i, x_i in enumerate(inputs):
    attn_scores_2[i] = ...  # torch.dot(x_i, query)

attn_weights_2 = ...  # torch.softmax(attn_scores_2, dim=0)

context_vec_2 = torch.zeros(query.shape)
for i, x_i in enumerate(inputs):
    context_vec_2 += ...  # attn_weights_2[i] * x_i

print(attn_weights_2)
print(context_vec_2)

In [ ]:
%%ipytest -qq

def test_attn_scores_2_is_dot_product():
    expected = torch.tensor([torch.dot(x_i, inputs[1]) for x_i in inputs])
    assert torch.allclose(attn_scores_2, expected)

def test_attn_weights_2_sums_to_one():
    assert torch.isclose(attn_weights_2.sum(), torch.tensor(1.0))
    assert torch.all(attn_weights_2 >= 0)

def test_attn_weights_2_highest_on_query_itself():
    # token 1 attending to itself should get a large share since dot(x,x) is typically largest
    assert attn_weights_2.argmax() == 1

def test_context_vec_2_is_weighted_sum():
    expected = (attn_weights_2.unsqueeze(-1) * inputs).sum(dim=0)
    assert torch.allclose(context_vec_2, expected, atol=1e-5)

<button id="claim-ex9" onclick="window.gamifyClaim('ex9', 4, this)" style="padding:6px 14px;border-radius:8px;border:1.5px solid #16a34a;background:#f0fdf4;color:#15803d;font-weight:600;font-size:13px;cursor:pointer;font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Helvetica,Arial,sans-serif">
✅ Mark passed (+4 pts, 4 tests)
</button>

<details ontoggle="window.gamifyReveal(this, 'ex9', 4)"><summary>Show solution (4 tests, −4 pts if opened)</summary>

```python
query = inputs[1]

attn_scores_2 = torch.empty(inputs.shape[0])
for i, x_i in enumerate(inputs):
    attn_scores_2[i] = torch.dot(x_i, query)

attn_weights_2 = torch.softmax(attn_scores_2, dim=0)

context_vec_2 = torch.zeros(query.shape)
for i, x_i in enumerate(inputs):
    context_vec_2 += attn_weights_2[i] * x_i

print(attn_weights_2)
print(context_vec_2)
```

</details>

### 3.3.2 — Computing attention weights for all input tokens

The single-query loop above generalizes to "every token attends to every token" by replacing the dot-product loop with one matmul: `inputs @ inputs.T` gives every pairwise dot product at once — row `i` is exactly `attn_scores_2`-style scores for query `i`.

### Exercise 2 — Attention weights and context vectors, vectorized

Compute `attn_weights` (row-wise softmax of `inputs @ inputs.T`) and `all_context_vecs` (`attn_weights @ inputs`) for the whole 6-token sequence in one shot — no loops.

In [ ]:
# TODO: implement this exercise
attn_scores = ...        # inputs @ inputs.T          -> (6, 6)
attn_weights = ...       # row-wise softmax of attn_scores, dim=-1
all_context_vecs = ...   # attn_weights @ inputs       -> (6, 3)

print(attn_weights)
print(all_context_vecs)

In [ ]:
%%ipytest -qq

def test_attn_scores_shape():
    assert attn_scores.shape == (6, 6)
    assert torch.allclose(attn_scores, inputs @ inputs.T)

def test_attn_weights_rows_sum_to_one():
    assert torch.allclose(attn_weights.sum(dim=-1), torch.ones(6))

def test_all_context_vecs_shape():
    assert all_context_vecs.shape == (6, 3)

def test_row_1_matches_single_query_result():
    # row 1 of this vectorized computation must equal the by-hand exercise 1 result
    assert torch.allclose(all_context_vecs[1], context_vec_2, atol=1e-5)

<button id="claim-ex10" onclick="window.gamifyClaim('ex10', 4, this)" style="padding:6px 14px;border-radius:8px;border:1.5px solid #16a34a;background:#f0fdf4;color:#15803d;font-weight:600;font-size:13px;cursor:pointer;font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Helvetica,Arial,sans-serif">
✅ Mark passed (+4 pts, 4 tests)
</button>

<details ontoggle="window.gamifyReveal(this, 'ex10', 4)"><summary>Show solution (4 tests, −4 pts if opened)</summary>

```python
attn_scores = inputs @ inputs.T
attn_weights = torch.softmax(attn_scores, dim=-1)
all_context_vecs = attn_weights @ inputs

print(attn_weights)
print(all_context_vecs)
```

</details>

### 3.4 — Implementing self-attention with trainable weights

Real self-attention doesn't operate directly on `inputs` — it first projects every token through three separate learned matrices, `W_query`, `W_key`, `W_value`, producing `queries`, `keys`, `values`. This is the actual mechanism from *Attention Is All You Need* ("scaled dot-product attention"): scores come from `queries @ keys.T`, scaled by `1/sqrt(d_out)` before the softmax (this keeps the softmax's input variance roughly constant regardless of `d_out`, preventing it from saturating into a near-one-hot distribution), and the context vector is `attn_weights @ values` — note `values`, not the raw `inputs` this time.

The query/key/value naming comes from information retrieval: a **query** is "what am I looking for," a **key** is "what does this item advertise about itself" (compared against the query to compute relevance), and a **value** is "what does this item actually contain" (returned, weighted by relevance). Every token plays all three roles simultaneously — it issues a query, offers a key, and carries a value — which is exactly what makes this *self*-attention rather than attention between two different sequences.

<img src="https://raw.githubusercontent.com/cleophasmashiri/ai-jupter-notebooks/main/llm_from_scratch/images/08-query-key-value-attention-flow.png" alt="Query, key, value attention flow" style="max-width:100%;height:auto;display:block;margin:1em auto" width="632" height="482"/>

### Exercise 3 — SelfAttention_v1

Implement `SelfAttention_v1` (book listing 3.1): three `nn.Parameter` weight matrices, initialized with `torch.rand(d_in, d_out)`, and a `forward` that computes scaled dot-product attention.

In [ ]:
# TODO: implement this exercise
class SelfAttention_v1(nn.Module):
    def __init__(self, d_in, d_out):
        super().__init__()
        self.W_query = nn.Parameter(torch.rand(d_in, d_out))
        self.W_key = nn.Parameter(torch.rand(d_in, d_out))
        self.W_value = nn.Parameter(torch.rand(d_in, d_out))

    def forward(self, x):
        keys = ...     # x @ self.W_key
        queries = ...  # x @ self.W_query
        values = ...   # x @ self.W_value
        attn_scores = ...    # queries @ keys.T
        attn_weights = ...   # softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        context_vec = ...    # attn_weights @ values
        return context_vec

torch.manual_seed(123)
d_in, d_out = 3, 2
sa_v1 = SelfAttention_v1(d_in, d_out)
print(sa_v1(inputs))

In [ ]:
%%ipytest -qq

def test_sa_v1_output_shape():
    torch.manual_seed(123)
    sa = SelfAttention_v1(3, 2)
    out = sa(inputs)
    assert out.shape == (6, 2)

def test_sa_v1_uses_scaled_dot_product():
    torch.manual_seed(123)
    sa = SelfAttention_v1(3, 2)
    keys = inputs @ sa.W_key
    queries = inputs @ sa.W_query
    values = inputs @ sa.W_value
    expected_scores = queries @ keys.T
    expected_weights = torch.softmax(expected_scores / keys.shape[-1]**0.5, dim=-1)
    expected_context = expected_weights @ values
    out = sa(inputs)
    assert torch.allclose(out, expected_context, atol=1e-5)

def test_sa_v1_is_deterministic_given_seed():
    torch.manual_seed(123)
    out1 = SelfAttention_v1(3, 2)(inputs)
    torch.manual_seed(123)
    out2 = SelfAttention_v1(3, 2)(inputs)
    assert torch.allclose(out1, out2)

<button id="claim-ex11" onclick="window.gamifyClaim('ex11', 3, this)" style="padding:6px 14px;border-radius:8px;border:1.5px solid #16a34a;background:#f0fdf4;color:#15803d;font-weight:600;font-size:13px;cursor:pointer;font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Helvetica,Arial,sans-serif">
✅ Mark passed (+3 pts, 3 tests)
</button>

<details ontoggle="window.gamifyReveal(this, 'ex11', 3)"><summary>Show solution (3 tests, −3 pts if opened)</summary>

```python
class SelfAttention_v1(nn.Module):
    def __init__(self, d_in, d_out):
        super().__init__()
        self.W_query = nn.Parameter(torch.rand(d_in, d_out))
        self.W_key = nn.Parameter(torch.rand(d_in, d_out))
        self.W_value = nn.Parameter(torch.rand(d_in, d_out))

    def forward(self, x):
        keys = x @ self.W_key
        queries = x @ self.W_query
        values = x @ self.W_value
        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        context_vec = attn_weights @ values
        return context_vec

torch.manual_seed(123)
d_in, d_out = 3, 2
sa_v1 = SelfAttention_v1(d_in, d_out)
print(sa_v1(inputs))
```

</details>

### Exercise 4 — SelfAttention_v2

Same computation, cleaner weights: replace the raw `nn.Parameter` matrices with `nn.Linear(d_in, d_out, bias=qkv_bias)` layers (book listing 3.2). `nn.Linear` has a better default weight-initialization scheme than `torch.rand`, which matters for training stability once this is stacked into a real model.

In [ ]:
# TODO: implement this exercise
class SelfAttention_v2(nn.Module):
    def __init__(self, d_in, d_out, qkv_bias=False):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

    def forward(self, x):
        keys = ...      # self.W_key(x)
        queries = ...   # self.W_query(x)
        values = ...    # self.W_value(x)
        attn_scores = ...    # queries @ keys.T
        attn_weights = ...   # softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        context_vec = ...    # attn_weights @ values
        return context_vec

torch.manual_seed(789)
sa_v2 = SelfAttention_v2(d_in, d_out)
print(sa_v2(inputs))

In [ ]:
%%ipytest -qq

def test_sa_v2_output_shape():
    torch.manual_seed(789)
    sa = SelfAttention_v2(3, 2)
    out = sa(inputs)
    assert out.shape == (6, 2)

def test_sa_v2_uses_linear_layers_not_raw_matmul_params():
    torch.manual_seed(789)
    sa = SelfAttention_v2(3, 2)
    assert isinstance(sa.W_query, nn.Linear)
    assert isinstance(sa.W_key, nn.Linear)
    assert isinstance(sa.W_value, nn.Linear)

def test_sa_v2_matches_manual_computation():
    torch.manual_seed(789)
    sa = SelfAttention_v2(3, 2)
    keys = sa.W_key(inputs)
    queries = sa.W_query(inputs)
    values = sa.W_value(inputs)
    expected_weights = torch.softmax((queries @ keys.T) / keys.shape[-1]**0.5, dim=-1)
    expected_context = expected_weights @ values
    assert torch.allclose(sa(inputs), expected_context, atol=1e-5)

<button id="claim-ex12" onclick="window.gamifyClaim('ex12', 3, this)" style="padding:6px 14px;border-radius:8px;border:1.5px solid #16a34a;background:#f0fdf4;color:#15803d;font-weight:600;font-size:13px;cursor:pointer;font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Helvetica,Arial,sans-serif">
✅ Mark passed (+3 pts, 3 tests)
</button>

<details ontoggle="window.gamifyReveal(this, 'ex12', 3)"><summary>Show solution (3 tests, −3 pts if opened)</summary>

```python
class SelfAttention_v2(nn.Module):
    def __init__(self, d_in, d_out, qkv_bias=False):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

    def forward(self, x):
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)
        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        context_vec = attn_weights @ values
        return context_vec

torch.manual_seed(789)
sa_v2 = SelfAttention_v2(d_in, d_out)
print(sa_v2(inputs))
```

</details>

### 3.5 — Hiding future words with causal attention

Everything so far lets every token see every *other* token, including ones that come later in the sequence — fine for understanding a whole fixed passage, but nonsensical for a model that must generate text one token at a time (attending to "the future" during training would let the model cheat by looking at the very answer it's supposed to predict). **Causal attention** fixes this by zeroing out attention weights for any position `j > i` relative to query position `i`.

The book shows two equivalent ways to get there:
1. **Mask-then-renormalize**: compute normal softmax attention weights, zero out the upper triangle with `torch.tril`, then divide each row by its new sum so rows still sum to 1.
2. **Mask-then-softmax** (the one actually used going forward): set the *scores* (not weights) at masked positions to `-inf` using `torch.triu(..., diagonal=1)` before the softmax — softmax naturally sends `exp(-inf) = 0`, so no separate renormalization step is needed, and it's numerically cleaner.

For a 4-token sequence, the resulting attention-weight matrix is lower-triangular — row *i* only ever has nonzero weight in columns `≤ i`:

<img src="https://raw.githubusercontent.com/cleophasmashiri/ai-jupter-notebooks/main/llm_from_scratch/images/09-causal-attention-masking-before-and-after.png" alt="Causal attention masking, before and after" style="max-width:100%;height:auto;display:block;margin:1em auto" width="692" height="227"/>

Row 1 can only attend to itself; row 4 (the last token) can attend to everything — matching generation, where by the time you're predicting token 5 you have tokens 1–4 to look back on, but nothing further ahead exists yet to peek at.

### Exercise 5 — Both causal masking approaches, and their equivalence

Given `attn_scores` and `attn_weights` freshly computed from `sa_v2`, build the causal mask both ways and confirm they produce the same result.

In [ ]:
# TODO: implement this exercise
queries = sa_v2.W_query(inputs)
keys = sa_v2.W_key(inputs)
attn_scores = queries @ keys.T
attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
context_length = attn_scores.shape[0]

# Approach 1: mask the *weights*, then renormalize
mask_simple = ...           # torch.tril(torch.ones(context_length, context_length))
masked_simple = ...         # attn_weights * mask_simple
row_sums = masked_simple.sum(dim=-1, keepdim=True)
masked_simple_norm = ...    # masked_simple / row_sums

# Approach 2: mask the *scores* with -inf, then softmax
mask = ...                  # torch.triu(torch.ones(context_length, context_length), diagonal=1)
masked = ...                # attn_scores.masked_fill(mask.bool(), -torch.inf)
attn_weights_causal = ...   # softmax(masked / keys.shape[-1]**0.5, dim=-1)

print(masked_simple_norm)
print(attn_weights_causal)

In [ ]:
%%ipytest -qq

def test_masks_are_triangular_opposites():
    assert torch.equal(mask_simple, torch.tril(torch.ones(6, 6)))
    assert torch.equal(mask, torch.triu(torch.ones(6, 6), diagonal=1))

def test_masked_simple_norm_rows_sum_to_one():
    assert torch.allclose(masked_simple_norm.sum(dim=-1), torch.ones(6), atol=1e-5)

def test_upper_triangle_is_zero_in_both_approaches():
    triu_idx = torch.triu_indices(6, 6, offset=1)
    assert torch.all(masked_simple_norm[triu_idx[0], triu_idx[1]] == 0)
    assert torch.all(attn_weights_causal[triu_idx[0], triu_idx[1]] == 0)

def test_both_causal_masking_approaches_agree():
    assert torch.allclose(masked_simple_norm, attn_weights_causal, atol=1e-4)

def test_first_row_only_attends_to_itself():
    assert torch.isclose(attn_weights_causal[0, 0], torch.tensor(1.0), atol=1e-5)

<button id="claim-ex13" onclick="window.gamifyClaim('ex13', 5, this)" style="padding:6px 14px;border-radius:8px;border:1.5px solid #16a34a;background:#f0fdf4;color:#15803d;font-weight:600;font-size:13px;cursor:pointer;font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Helvetica,Arial,sans-serif">
✅ Mark passed (+5 pts, 5 tests)
</button>

<details ontoggle="window.gamifyReveal(this, 'ex13', 5)"><summary>Show solution (5 tests, −5 pts if opened)</summary>

```python
queries = sa_v2.W_query(inputs)
keys = sa_v2.W_key(inputs)
attn_scores = queries @ keys.T
attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
context_length = attn_scores.shape[0]

mask_simple = torch.tril(torch.ones(context_length, context_length))
masked_simple = attn_weights * mask_simple
row_sums = masked_simple.sum(dim=-1, keepdim=True)
masked_simple_norm = masked_simple / row_sums

mask = torch.triu(torch.ones(context_length, context_length), diagonal=1)
masked = attn_scores.masked_fill(mask.bool(), -torch.inf)
attn_weights_causal = torch.softmax(masked / keys.shape[-1]**0.5, dim=-1)

print(masked_simple_norm)
print(attn_weights_causal)
```

</details>

# Understanding `nn.Dropout()` in attention

`nn.Dropout(p)` randomly zeroes elements of its input (each independently, with probability `p`) during training, and does nothing at all during `.eval()`. Applied *directly to the attention weights matrix* (as the book does, right after the softmax), it means each training step, a token effectively ignores a random subset of the positions it would otherwise attend to — a regularizer that prevents the model from overfitting to specific attention patterns in the training data. The book uses relatively small rates (`0.1`–`0.2`) for this. Since dropout is randomized, exercises below default to `dropout=0.0` so their outputs stay deterministic and testable — in a real training run you'd use a nonzero rate.

### 3.5.3 — A compact causal attention class

### Exercise 6 — CausalAttention

Implement the book's `CausalAttention` class (listing 3.3): batched (input `x` has shape `(b, num_tokens, d_in)`), with the causal mask stored via `self.register_buffer` (so it moves with `.to(device)` automatically, without being treated as a trainable parameter) and dropout applied to the attention weights.

In [ ]:
# TODO: implement this exercise
class CausalAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, qkv_bias=False):
        super().__init__()
        self.d_out = d_out
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            'mask',
            ...  # torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        attn_scores = ...  # queries @ keys.transpose(1, 2)   -- transpose *within* each batch element
        attn_scores.masked_fill_(
            self.mask.bool()[:num_tokens, :num_tokens], -torch.inf
        )
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)
        context_vec = ...  # attn_weights @ values
        return context_vec

batch = torch.stack((inputs, inputs), dim=0)
print(batch.shape)

torch.manual_seed(123)
context_length = batch.shape[1]
ca = CausalAttention(d_in, d_out, context_length, 0.0)
context_vecs = ca(batch)
print("context_vecs.shape:", context_vecs.shape)

In [ ]:
%%ipytest -qq

def test_batch_shape():
    assert batch.shape == (2, 6, 3)

def test_causal_attention_output_shape():
    torch.manual_seed(123)
    ca_t = CausalAttention(3, 2, 6, 0.0)
    out = ca_t(batch)
    assert out.shape == (2, 6, 2)

def test_causal_attention_uses_register_buffer_for_mask():
    torch.manual_seed(123)
    ca_t = CausalAttention(3, 2, 6, 0.0)
    assert "mask" in dict(ca_t.named_buffers())
    assert "mask" not in dict(ca_t.named_parameters())

def test_causal_attention_matches_across_batch_elements():
    # inputs is duplicated across the batch dim, so both rows must produce identical output
    torch.manual_seed(123)
    ca_t = CausalAttention(3, 2, 6, 0.0)
    out = ca_t(batch)
    assert torch.allclose(out[0], out[1])

<button id="claim-ex14" onclick="window.gamifyClaim('ex14', 4, this)" style="padding:6px 14px;border-radius:8px;border:1.5px solid #16a34a;background:#f0fdf4;color:#15803d;font-weight:600;font-size:13px;cursor:pointer;font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Helvetica,Arial,sans-serif">
✅ Mark passed (+4 pts, 4 tests)
</button>

<details ontoggle="window.gamifyReveal(this, 'ex14', 4)"><summary>Show solution (4 tests, −4 pts if opened)</summary>

```python
class CausalAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, qkv_bias=False):
        super().__init__()
        self.d_out = d_out
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            'mask',
            torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        attn_scores = queries @ keys.transpose(1, 2)
        attn_scores.masked_fill_(
            self.mask.bool()[:num_tokens, :num_tokens], -torch.inf
        )
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)
        context_vec = attn_weights @ values
        return context_vec

batch = torch.stack((inputs, inputs), dim=0)
print(batch.shape)

torch.manual_seed(123)
context_length = batch.shape[1]
ca = CausalAttention(d_in, d_out, context_length, 0.0)
context_vecs = ca(batch)
print("context_vecs.shape:", context_vecs.shape)
```

</details>

### 3.6 — Extending single-head attention to multi-head attention

A single attention head learns one notion of "relevance." **Multi-head attention** runs several attention heads in parallel on the same input, each with its own `W_query`/`W_key`/`W_value`, then concatenates their outputs — letting different heads specialize (one might track local syntax, another long-range references) instead of forcing everything into one fixed pattern.

<img src="https://raw.githubusercontent.com/cleophasmashiri/ai-jupter-notebooks/main/llm_from_scratch/images/10-multi-head-attention-split-project-concatenate.png" alt="Multi-head attention: split, project, concatenate" style="max-width:100%;height:auto;display:block;margin:1em auto" width="952" height="272"/>

Each head sees the *same* input but learns *different* projections, so `num_heads * head_dim == d_out` splits one representation into several independent "points of view" that get recombined at the end — exercise 7 below does this the direct way (a literal list of separate `CausalAttention` modules); exercise 8 reimplements the identical math as one fused module, which is what real GPT implementations actually run.

### Exercise 7 — MultiHeadAttentionWrapper

The most direct implementation: a `nn.ModuleList` of independent `CausalAttention` heads, concatenated along the last dimension.

In [ ]:
# TODO: implement this exercise
class MultiHeadAttentionWrapper(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        self.heads = nn.ModuleList(
            [CausalAttention(d_in, d_out, context_length, dropout, qkv_bias) for _ in range(num_heads)]
        )

    def forward(self, x):
        return ...  # concatenate each head's output along the last dimension

torch.manual_seed(123)
context_length = batch.shape[1]
d_in, d_out = 3, 2
mha_wrapper = MultiHeadAttentionWrapper(d_in, d_out, context_length, 0.0, num_heads=2)
context_vecs_mhw = mha_wrapper(batch)
print(context_vecs_mhw.shape)

In [ ]:
%%ipytest -qq

def test_mhw_num_heads():
    torch.manual_seed(123)
    m = MultiHeadAttentionWrapper(3, 2, 6, 0.0, num_heads=2)
    assert len(m.heads) == 2
    assert all(isinstance(h, CausalAttention) for h in m.heads)

def test_mhw_output_shape_is_concatenated():
    torch.manual_seed(123)
    m = MultiHeadAttentionWrapper(3, 2, 6, 0.0, num_heads=2)
    out = m(batch)
    assert out.shape == (2, 6, 4)  # num_heads * d_out

def test_mhw_matches_manual_concat():
    torch.manual_seed(123)
    m = MultiHeadAttentionWrapper(3, 2, 6, 0.0, num_heads=2)
    out = m(batch)
    expected = torch.cat([h(batch) for h in m.heads], dim=-1)
    assert torch.allclose(out, expected)

<button id="claim-ex15" onclick="window.gamifyClaim('ex15', 3, this)" style="padding:6px 14px;border-radius:8px;border:1.5px solid #16a34a;background:#f0fdf4;color:#15803d;font-weight:600;font-size:13px;cursor:pointer;font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Helvetica,Arial,sans-serif">
✅ Mark passed (+3 pts, 3 tests)
</button>

<details ontoggle="window.gamifyReveal(this, 'ex15', 3)"><summary>Show solution (3 tests, −3 pts if opened)</summary>

```python
class MultiHeadAttentionWrapper(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        self.heads = nn.ModuleList(
            [CausalAttention(d_in, d_out, context_length, dropout, qkv_bias) for _ in range(num_heads)]
        )

    def forward(self, x):
        return torch.cat([head(x) for head in self.heads], dim=-1)

torch.manual_seed(123)
context_length = batch.shape[1]
d_in, d_out = 3, 2
mha_wrapper = MultiHeadAttentionWrapper(d_in, d_out, context_length, 0.0, num_heads=2)
context_vecs_mhw = mha_wrapper(batch)
print(context_vecs_mhw.shape)
```

</details>

### Exercise 8 — MultiHeadAttention (the efficient, single-class version)

`MultiHeadAttentionWrapper` computes each head with a *separate* set of `nn.Linear` layers — `num_heads` separate matmuls per q/k/v. The book's final `MultiHeadAttention` class (listing 3.5) does one big projection to `d_out` and *reshapes* it into `num_heads` heads of `head_dim = d_out // num_heads` each — mathematically equivalent, but one matmul instead of `num_heads` of them, which is what every real GPT implementation actually does. This is also the exact class chapter 4 imports to build the full model.

In [ ]:
# TODO: implement this exercise
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert (d_out % num_heads == 0), "d_out must be divisible by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = ...  # d_out // num_heads
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        # split d_out into (num_heads, head_dim), then move num_heads next to the batch dim
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

        keys = ...      # keys.transpose(1, 2)      -> (b, num_heads, num_tokens, head_dim)
        queries = ...   # queries.transpose(1, 2)
        values = ...    # values.transpose(1, 2)

        attn_scores = queries @ keys.transpose(2, 3)
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]
        attn_scores.masked_fill_(mask_bool, -torch.inf)

        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        context_vec = (attn_weights @ values).transpose(1, 2)
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        context_vec = ...  # self.out_proj(context_vec) -- the optional output projection
        return context_vec

torch.manual_seed(123)
batch_size, context_length, d_in = batch.shape
d_out = 2
mha = MultiHeadAttention(d_in, d_out, context_length, 0.0, num_heads=2)
context_vecs_mha = mha(batch)
print(context_vecs_mha.shape)
print(context_vecs_mha)

In [ ]:
%%ipytest -qq

def test_mha_output_shape_equals_d_out_not_multiplied():
    torch.manual_seed(123)
    m = MultiHeadAttention(3, 2, 6, 0.0, num_heads=2)
    out = m(batch)
    assert out.shape == (2, 6, 2)  # note: d_out, NOT num_heads * d_out like the wrapper

def test_mha_head_dim_computed_correctly():
    torch.manual_seed(123)
    m = MultiHeadAttention(3, 6, 6, 0.0, num_heads=3)
    assert m.head_dim == 2

def test_mha_rejects_non_divisible_num_heads():
    with __import__("pytest").raises(AssertionError):
        MultiHeadAttention(3, 5, 6, 0.0, num_heads=2)

def test_mha_has_output_projection():
    torch.manual_seed(123)
    m = MultiHeadAttention(3, 2, 6, 0.0, num_heads=2)
    assert isinstance(m.out_proj, nn.Linear)
    assert m.out_proj.in_features == 2 and m.out_proj.out_features == 2

def test_mha_identical_batch_rows_produce_identical_output():
    torch.manual_seed(123)
    m = MultiHeadAttention(3, 2, 6, 0.0, num_heads=2)
    out = m(batch)
    assert torch.allclose(out[0], out[1])

<button id="claim-ex16" onclick="window.gamifyClaim('ex16', 5, this)" style="padding:6px 14px;border-radius:8px;border:1.5px solid #16a34a;background:#f0fdf4;color:#15803d;font-weight:600;font-size:13px;cursor:pointer;font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Helvetica,Arial,sans-serif">
✅ Mark passed (+5 pts, 5 tests)
</button>

<details ontoggle="window.gamifyReveal(this, 'ex16', 5)"><summary>Show solution (5 tests, −5 pts if opened)</summary>

```python
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert (d_out % num_heads == 0), "d_out must be divisible by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        attn_scores = queries @ keys.transpose(2, 3)
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]
        attn_scores.masked_fill_(mask_bool, -torch.inf)

        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        context_vec = (attn_weights @ values).transpose(1, 2)
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        context_vec = self.out_proj(context_vec)
        return context_vec

torch.manual_seed(123)
batch_size, context_length, d_in = batch.shape
d_out = 2
mha = MultiHeadAttention(d_in, d_out, context_length, 0.0, num_heads=2)
context_vecs_mha = mha(batch)
print(context_vecs_mha.shape)
print(context_vecs_mha)
```

</details>

### Checkpoint: chapter 3 — attention mechanisms

1. Why does the score get divided by `sqrt(head_dim)` before the softmax?

<details><summary>Show answer</summary>

If `q` and `k` entries have roughly unit variance, their dot product's variance grows proportionally to the dimension they're summed over (`head_dim`) — larger dimensions produce larger-magnitude scores purely as an artifact of dimensionality, not because the underlying relevance is actually stronger. Left unscaled, this pushes softmax's input to extreme magnitudes, saturating it toward a near-one-hot distribution and shrinking gradients. Dividing by `sqrt(head_dim)` keeps the scores' variance roughly constant regardless of dimension.

</details>

2. `register_buffer` vs. `nn.Parameter` — what's the practical difference for the causal mask?

<details><summary>Show answer</summary>

Both move with the module when you call `.to(device)` and get saved/loaded with `state_dict()`. The difference is `nn.Parameter` is registered as trainable — it shows up in `.parameters()` and gets a gradient update every optimizer step. The causal mask is a fixed, non-learned constant (it should never change during training), so `register_buffer` gives you device/checkpoint management without accidentally making the mask trainable.

</details>

3. `MultiHeadAttentionWrapper` and `MultiHeadAttention` compute the same thing — why does the book bother with the second, more complex version?

<details><summary>Show answer</summary>

Efficiency. The wrapper launches `num_heads` separate small matmuls (one full `nn.Linear` per head, per q/k/v) — three GPU kernel launches times `num_heads`. The single-class version does exactly *one* matmul to `d_out` for each of q/k/v, then reshapes into heads with `.view()`/`.transpose()` (cheap, no computation) before batched-matmul attention. Same math, far fewer, larger kernel launches — which is what actually matters for GPU throughput at scale.

</details>

4. If you wanted this attention module to work for a bidirectional task (e.g. classifying a whole known sentence) instead of autoregressive generation, what would you change?

<details><summary>Show answer</summary>

Drop the causal mask entirely (skip the `masked_fill_` step, or don't build `self.mask` at all) — bidirectional attention just needs every token to be able to attend to every other token, since the whole input is available upfront and there's no "future" being illegitimately peeked at.

</details>

Chapter 4 imports `MultiHeadAttention` exactly as implemented in exercise 8 — make sure it's solid before moving on.

## Chapter 4 — Implementing a GPT model from scratch to generate text

Chapter 3 built one piece — attention — in isolation. This chapter assembles the rest of the architecture around it: normalization, a per-token feed-forward network, residual connections, and the embedding/output layers that turn all of it into an actual language model. Every exercise here reuses `MultiHeadAttention` from chapter 3 exactly as written.

The whole model is configured by one dictionary — this is the book's `GPT_CONFIG_124M`, matching the smallest GPT-2 checkpoint OpenAI released:

In [ ]:
GPT_CONFIG_124M = {
    "vocab_size": 50257,    # Vocabulary size (GPT-2 BPE, from chapter 2)
    "context_length": 1024, # Max sequence length the model was trained on
    "emb_dim": 768,         # Embedding dimension
    "n_heads": 12,          # Number of attention heads
    "n_layers": 12,         # Number of transformer blocks
    "drop_rate": 0.1,       # Dropout rate
    "qkv_bias": False       # Whether Q/K/V linear layers have a bias term
}

# A much smaller config, used below to keep exercise/test runtimes fast.
TEST_CFG = {
    "vocab_size": 50,
    "context_length": 6,
    "emb_dim": 12,
    "n_heads": 3,
    "n_layers": 2,
    "drop_rate": 0.0,
    "qkv_bias": False
}

### 4.1 — Coding an LLM architecture, at a glance

Before the individual pieces (normalization, feed-forward, shortcuts) each get their own exercise below, here's the complete assembled picture they're building toward — the book calls this same diagram out at the start of the chapter for exactly this reason, so you know where each piece slots in before seeing its code:

<img src="https://raw.githubusercontent.com/cleophasmashiri/ai-jupter-notebooks/main/llm_from_scratch/images/11-full-gptmodel-architecture.png" alt="Full GPTModel architecture" style="max-width:100%;height:auto;display:block;margin:1em auto" width="572" height="632"/>

Everything from here to exercise 5 (`GPTModel`) is building the `TransformerBlock` box in the middle — one instance of which is shown expanded in the diagram right before exercise 4.

The book briefly first codes a `DummyGPTModel` — the same skeleton above, but with `TransformerBlock` and `LayerNorm` replaced by no-op placeholders that just pass their input straight through unchanged. The point of that intermediate step is purely organizational: get the overall shape (embeddings in, logits out, right tensor shapes throughout) working and testable *before* filling in what each block actually computes. This notebook skips straight to the real implementations below, but the reasoning is worth keeping in mind if you ever scaffold a new architecture yourself — get the skeleton's shapes right first, then fill in real logic one piece at a time.

### 4.2 — Normalizing activations with layer normalization

Deep networks are notoriously sensitive to how activations are scaled as they flow through many stacked layers — values can drift toward very large or very small magnitudes, destabilizing training (vanishing/exploding gradients). **Layer normalization** rescales every token's feature vector independently to zero mean and unit variance, then applies two small *learned* parameters (`scale`, `shift`) so the network can still recover whatever distribution actually works best — it's not forced to stay at exactly mean-0/variance-1, just anchored there as a starting point.

### Exercise 1 — LayerNorm from scratch

Implement the book's `LayerNorm` module (listing 4.2): normalize over the last dimension (`emb_dim`), then scale and shift.

In [ ]:
# TODO: implement this exercise
class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x):
        mean = ...   # x.mean(dim=-1, keepdim=True)
        var = ...    # x.var(dim=-1, keepdim=True, unbiased=False)
        norm_x = ...  # (x - mean) / sqrt(var + self.eps)
        return self.scale * norm_x + self.shift

torch.manual_seed(123)
batch_example = torch.randn(2, 5)
ln = LayerNorm(emb_dim=5)
out_ln = ln(batch_example)
print("Mean:", out_ln.mean(dim=-1))
print("Variance:", out_ln.var(dim=-1, unbiased=False))

In [ ]:
%%ipytest -qq

def test_layernorm_output_is_zero_mean_unit_var():
    torch.manual_seed(123)
    x = torch.randn(2, 5)
    ln_t = LayerNorm(5)
    out = ln_t(x)
    assert torch.allclose(out.mean(dim=-1), torch.zeros(2), atol=1e-5)
    assert torch.allclose(out.var(dim=-1, unbiased=False), torch.ones(2), atol=1e-4)

def test_layernorm_normalizes_last_dim_independently_per_token():
    torch.manual_seed(0)
    x = torch.randn(3, 4, 8)  # (batch, tokens, emb_dim)
    ln_t = LayerNorm(8)
    out = ln_t(x)
    assert out.shape == x.shape
    assert torch.allclose(out.mean(dim=-1), torch.zeros(3, 4), atol=1e-5)

def test_layernorm_uses_biased_variance_not_bessel_corrected():
    torch.manual_seed(0)
    x = torch.randn(4, 6)
    ln_t = LayerNorm(6)
    mean = x.mean(dim=-1, keepdim=True)
    biased_var = x.var(dim=-1, keepdim=True, unbiased=False)
    expected = ln_t.scale * (x - mean) / torch.sqrt(biased_var + 1e-5) + ln_t.shift
    assert torch.allclose(ln_t(x), expected, atol=1e-5)

def test_layernorm_scale_and_shift_are_trainable_parameters():
    ln_t = LayerNorm(5)
    names = dict(ln_t.named_parameters())
    assert "scale" in names and "shift" in names
    assert torch.equal(names["scale"], torch.ones(5))
    assert torch.equal(names["shift"], torch.zeros(5))

<button id="claim-ex17" onclick="window.gamifyClaim('ex17', 4, this)" style="padding:6px 14px;border-radius:8px;border:1.5px solid #16a34a;background:#f0fdf4;color:#15803d;font-weight:600;font-size:13px;cursor:pointer;font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Helvetica,Arial,sans-serif">
✅ Mark passed (+4 pts, 4 tests)
</button>

<details ontoggle="window.gamifyReveal(this, 'ex17', 4)"><summary>Show solution (4 tests, −4 pts if opened)</summary>

```python
class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        norm_x = (x - mean) / torch.sqrt(var + self.eps)
        return self.scale * norm_x + self.shift

torch.manual_seed(123)
batch_example = torch.randn(2, 5)
ln = LayerNorm(emb_dim=5)
out_ln = ln(batch_example)
print("Mean:", out_ln.mean(dim=-1))
print("Variance:", out_ln.var(dim=-1, unbiased=False))
```

</details>

### 4.3 — Implementing a feed forward network with GELU activations

GPT uses **GELU** (Gaussian Error Linear Unit) instead of the more classic ReLU. Unlike ReLU's hard cutoff at 0, GELU is smooth everywhere — including a small negative "dip" just below zero — which empirically trains better in deep transformer stacks. The book uses the same tanh-based approximation the original GPT-2 implementation used (an exact GELU requires the Gaussian CDF, which has no closed form).

<img src="https://raw.githubusercontent.com/cleophasmashiri/ai-jupter-notebooks/main/llm_from_scratch/images/12-gelu-vs-relu-activation-curves.png" alt="GELU vs. ReLU activation curves" style="max-width:100%;height:auto;display:block;margin:1em auto" width="520"/>

### Exercise 2 — GELU

Implement the book's tanh approximation: `GELU(x) = 0.5x · (1 + tanh[√(2/π) · (x + 0.044715x³)])`.

In [ ]:
# TODO: implement this exercise
class GELU(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        return ...  # 0.5 * x * (1 + tanh(sqrt(2/pi) * (x + 0.044715 * x**3)))

gelu, relu = GELU(), nn.ReLU()
sample = torch.linspace(-3, 3, 7)
print("GELU:", gelu(sample))
print("ReLU:", relu(sample))

In [ ]:
%%ipytest -qq

def test_gelu_matches_formula():
    x = torch.linspace(-5, 5, 50)
    expected = 0.5 * x * (1 + torch.tanh(
        torch.sqrt(torch.tensor(2.0 / torch.pi)) * (x + 0.044715 * torch.pow(x, 3))
    ))
    assert torch.allclose(GELU()(x), expected, atol=1e-6)

def test_gelu_is_approximately_zero_at_zero():
    assert torch.isclose(GELU()(torch.tensor(0.0)), torch.tensor(0.0), atol=1e-6)

def test_gelu_approaches_identity_for_large_positive_x():
    x = torch.tensor(10.0)
    assert torch.isclose(GELU()(x), x, atol=1e-3)

def test_gelu_is_negative_for_some_negative_input_unlike_relu():
    # GELU's small negative dip is the key qualitative difference from ReLU
    x = torch.tensor(-0.5)
    assert GELU()(x).item() < 0
    assert nn.ReLU()(x).item() == 0

<button id="claim-ex18" onclick="window.gamifyClaim('ex18', 4, this)" style="padding:6px 14px;border-radius:8px;border:1.5px solid #16a34a;background:#f0fdf4;color:#15803d;font-weight:600;font-size:13px;cursor:pointer;font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Helvetica,Arial,sans-serif">
✅ Mark passed (+4 pts, 4 tests)
</button>

<details ontoggle="window.gamifyReveal(this, 'ex18', 4)"><summary>Show solution (4 tests, −4 pts if opened)</summary>

```python
class GELU(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(
            torch.sqrt(torch.tensor(2.0 / torch.pi)) * (x + 0.044715 * torch.pow(x, 3))
        ))

gelu, relu = GELU(), nn.ReLU()
sample = torch.linspace(-3, 3, 7)
print("GELU:", gelu(sample))
print("ReLU:", relu(sample))
```

</details>

### Exercise 3 — FeedForward

The per-token feed-forward sub-layer: expand `emb_dim` to `4 * emb_dim`, apply `GELU`, project back down to `emb_dim`. This runs identically and independently on every token position — it's where each token "thinks" about what attention just handed it, as opposed to attention itself, which is the only place information moves *between* token positions.

In [ ]:
# TODO: implement this exercise
class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.layers = nn.Sequential(
            ...,  # nn.Linear(emb_dim, 4*emb_dim)
            ...,  # GELU()
            ...,  # nn.Linear(4*emb_dim, emb_dim)
        )

    def forward(self, x):
        return self.layers(x)

ffn = FeedForward(TEST_CFG)
x = torch.rand(2, 3, TEST_CFG["emb_dim"])
print(ffn(x).shape)

In [ ]:
%%ipytest -qq

def test_feedforward_preserves_shape():
    ffn_t = FeedForward(TEST_CFG)
    x = torch.rand(2, 3, TEST_CFG["emb_dim"])
    assert ffn_t(x).shape == x.shape

def test_feedforward_expands_to_4x_internally():
    ffn_t = FeedForward(TEST_CFG)
    linears = [m for m in ffn_t.layers if isinstance(m, nn.Linear)]
    assert len(linears) == 2
    assert linears[0].out_features == 4 * TEST_CFG["emb_dim"]
    assert linears[1].in_features == 4 * TEST_CFG["emb_dim"]
    assert linears[1].out_features == TEST_CFG["emb_dim"]

def test_feedforward_uses_gelu():
    ffn_t = FeedForward(TEST_CFG)
    assert any(isinstance(m, GELU) for m in ffn_t.layers)

<button id="claim-ex19" onclick="window.gamifyClaim('ex19', 3, this)" style="padding:6px 14px;border-radius:8px;border:1.5px solid #16a34a;background:#f0fdf4;color:#15803d;font-weight:600;font-size:13px;cursor:pointer;font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Helvetica,Arial,sans-serif">
✅ Mark passed (+3 pts, 3 tests)
</button>

<details ontoggle="window.gamifyReveal(this, 'ex19', 3)"><summary>Show solution (3 tests, −3 pts if opened)</summary>

```python
class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]),
            GELU(),
            nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"]),
        )

    def forward(self, x):
        return self.layers(x)

ffn = FeedForward(TEST_CFG)
x = torch.rand(2, 3, TEST_CFG["emb_dim"])
print(ffn(x).shape)
```

</details>

### 4.4 — Adding shortcut connections

Stacking many layers naively makes gradients shrink (or blow up) as they backpropagate through each one — by the time a gradient signal reaches an early layer in a 12+ layer network, it can be too small to meaningfully update those weights. **Shortcut (residual) connections** — `x = x + sublayer(x)` instead of `x = sublayer(x)` — give gradients a direct additive path back to every earlier layer, bypassing the multiplicative shrinkage. The book demonstrates this concretely: a 5-layer deep network with and without shortcuts, comparing gradient magnitudes layer by layer.

This demonstration is given directly (not an exercise) — the mechanics (`nn.Sequential`, `.backward()`, inspecting `.grad`) aren't new; what matters is *reading the result*.

In [ ]:
class ExampleDeepNeuralNetwork(nn.Module):
    def __init__(self, layer_sizes, use_shortcut):
        super().__init__()
        self.use_shortcut = use_shortcut
        self.layers = nn.ModuleList([
            nn.Sequential(nn.Linear(layer_sizes[0], layer_sizes[1]), GELU()),
            nn.Sequential(nn.Linear(layer_sizes[1], layer_sizes[2]), GELU()),
            nn.Sequential(nn.Linear(layer_sizes[2], layer_sizes[3]), GELU()),
            nn.Sequential(nn.Linear(layer_sizes[3], layer_sizes[4]), GELU()),
            nn.Sequential(nn.Linear(layer_sizes[4], layer_sizes[5]), GELU()),
        ])

    def forward(self, x):
        for layer in self.layers:
            layer_output = layer(x)
            if self.use_shortcut and x.shape == layer_output.shape:
                x = x + layer_output
            else:
                x = layer_output
        return x

def print_gradients(model, x):
    output = model(x)
    target = torch.tensor([[0.]])
    loss = nn.MSELoss()(output, target)
    loss.backward()
    for name, param in model.named_parameters():
        if "weight" in name:
            print(f"{name} has gradient mean of {param.grad.abs().mean().item():.6f}")

layer_sizes = [3, 3, 3, 3, 3, 1]
sample_input = torch.tensor([[1., 0., -1.]])

torch.manual_seed(123)
model_without_shortcut = ExampleDeepNeuralNetwork(layer_sizes, use_shortcut=False)
print("Without shortcut connections:")
print_gradients(model_without_shortcut, sample_input)

torch.manual_seed(123)
model_with_shortcut = ExampleDeepNeuralNetwork(layer_sizes, use_shortcut=True)
print("\nWith shortcut connections:")
print_gradients(model_with_shortcut, sample_input)

Notice the *without*-shortcut gradients shrink sharply toward the earliest layer (`layers.0`) — classic vanishing gradients. With shortcuts, gradient magnitudes stay comparable across all five layers. This is why `TransformerBlock` below wraps both attention and the feed-forward network in `x = x + sublayer(x)`, not just `x = sublayer(x)`.

### 4.5 — Connecting attention and linear layers in a transformer block

`TransformerBlock` is where every earlier piece meets: `MultiHeadAttention` (ch. 3), `FeedForward` and `LayerNorm` (above), and shortcut connections around each. The book uses **pre-norm** — `LayerNorm` applied *before* each sublayer, not after — which is what GPT-2 onward adopted for training stability (the original 2017 Transformer paper used post-norm).

<img src="https://raw.githubusercontent.com/cleophasmashiri/ai-jupter-notebooks/main/llm_from_scratch/images/13-transformerblock-internals-two-pre-norm-residual-s.png" alt="TransformerBlock internals: two pre-norm residual sub-layers" style="max-width:100%;height:auto;display:block;margin:1em auto" width="612" height="652"/>

Two sub-layers, each wrapped the same way: normalize → transform → dropout → add back the *pre-normalization* input. `MultiHeadAttention` is the only place tokens exchange information with each other (`x[t]` reads from `x[<=t]`); `FeedForward` runs identically and independently on every position — no cross-token mixing happens there at all. Stack `n_layers` of this one block (exercise 5, next) and that's the entire "reasoning" engine of a GPT model.

### Exercise 4 — TransformerBlock

Implement `x = x + Dropout(Attention(LayerNorm(x)))`, then `x = x + Dropout(FeedForward(LayerNorm(x)))`.

In [ ]:
# TODO: implement this exercise
class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = MultiHeadAttention(
            d_in=cfg["emb_dim"],
            d_out=cfg["emb_dim"],
            context_length=cfg["context_length"],
            num_heads=cfg["n_heads"],
            dropout=cfg["drop_rate"],
            qkv_bias=cfg["qkv_bias"])
        self.ff = FeedForward(cfg)
        self.norm1 = LayerNorm(cfg["emb_dim"])
        self.norm2 = LayerNorm(cfg["emb_dim"])
        self.drop_shortcut = nn.Dropout(cfg["drop_rate"])

    def forward(self, x):
        shortcut = x
        x = self.norm1(x)
        x = self.att(x)
        x = self.drop_shortcut(x)
        x = ...  # add the shortcut back

        shortcut = x
        x = self.norm2(x)
        x = self.ff(x)
        x = self.drop_shortcut(x)
        x = ...  # add the shortcut back
        return x

torch.manual_seed(123)
x = torch.rand(2, 4, TEST_CFG["emb_dim"])
block = TransformerBlock(TEST_CFG)
output = block(x)
print("Input shape:", x.shape)
print("Output shape:", output.shape)

In [ ]:
%%ipytest -qq

def test_transformer_block_preserves_shape():
    torch.manual_seed(123)
    x = torch.rand(2, 4, TEST_CFG["emb_dim"])
    block_t = TransformerBlock(TEST_CFG)
    assert block_t(x).shape == x.shape

def test_transformer_block_uses_two_shortcut_connections():
    # a zeroed-attention-and-ff block (via monkeypatched submodules) should return x unchanged
    torch.manual_seed(123)
    block_t = TransformerBlock(TEST_CFG)
    block_t.att.forward = lambda x: torch.zeros_like(x)
    block_t.ff.forward = lambda x: torch.zeros_like(x)
    x = torch.rand(2, 4, TEST_CFG["emb_dim"])
    # with shortcuts, LayerNorm(x) is discarded and x passes through unchanged since sublayers output 0
    out = block_t(x)
    assert torch.allclose(out, x, atol=1e-5)

def test_transformer_block_has_expected_submodules():
    block_t = TransformerBlock(TEST_CFG)
    assert isinstance(block_t.att, MultiHeadAttention)
    assert isinstance(block_t.ff, FeedForward)
    assert isinstance(block_t.norm1, LayerNorm) and isinstance(block_t.norm2, LayerNorm)

<button id="claim-ex20" onclick="window.gamifyClaim('ex20', 3, this)" style="padding:6px 14px;border-radius:8px;border:1.5px solid #16a34a;background:#f0fdf4;color:#15803d;font-weight:600;font-size:13px;cursor:pointer;font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Helvetica,Arial,sans-serif">
✅ Mark passed (+3 pts, 3 tests)
</button>

<details ontoggle="window.gamifyReveal(this, 'ex20', 3)"><summary>Show solution (3 tests, −3 pts if opened)</summary>

```python
class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = MultiHeadAttention(
            d_in=cfg["emb_dim"],
            d_out=cfg["emb_dim"],
            context_length=cfg["context_length"],
            num_heads=cfg["n_heads"],
            dropout=cfg["drop_rate"],
            qkv_bias=cfg["qkv_bias"])
        self.ff = FeedForward(cfg)
        self.norm1 = LayerNorm(cfg["emb_dim"])
        self.norm2 = LayerNorm(cfg["emb_dim"])
        self.drop_shortcut = nn.Dropout(cfg["drop_rate"])

    def forward(self, x):
        shortcut = x
        x = self.norm1(x)
        x = self.att(x)
        x = self.drop_shortcut(x)
        x = x + shortcut

        shortcut = x
        x = self.norm2(x)
        x = self.ff(x)
        x = self.drop_shortcut(x)
        x = x + shortcut
        return x

torch.manual_seed(123)
x = torch.rand(2, 4, TEST_CFG["emb_dim"])
block = TransformerBlock(TEST_CFG)
output = block(x)
print("Input shape:", x.shape)
print("Output shape:", output.shape)
```

</details>

# Understanding `nn.Sequential()`

`nn.Sequential(*modules)` chains modules together: calling the resulting object runs its input through each module in order, feeding each one's output to the next. It's just a convenience wrapper — equivalent to writing `x = m1(x); x = m2(x); ...` by hand — but it lets a stack of identical layers be built and indexed like a list.

### In this notebook
```python
self.layers = nn.Sequential(
    nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]), GELU(), nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"])
)
...
self.trf_blocks = nn.Sequential(*[TransformerBlock(cfg) for _ in range(cfg["n_layers"])])
```
The second usage is the important one for `GPTModel` below: `[TransformerBlock(cfg) for _ in range(cfg["n_layers"])]` builds a Python list of `n_layers` *independent* `TransformerBlock` instances (each with its own randomly-initialized weights), and `nn.Sequential(*...)` unpacks that list into one module that runs them back-to-back — exactly "stack `n_layers` transformer blocks," in one line.

### 4.6 — Coding the GPT model

Everything assembles here: token + positional embeddings (chapter 2) feed into a stack of `TransformerBlock`s (above), followed by a final `LayerNorm` and a linear `out_head` that projects back up to `vocab_size` — one logit per possible next token, at every position.

### Exercise 5 — GPTModel

Implement the book's `GPTModel` class (the architecture diagram's every box, assembled).

In [ ]:
# TODO: implement this exercise
class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])
        self.trf_blocks = nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])])
        self.final_norm = LayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)

    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx)
        pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))
        x = ...  # tok_embeds + pos_embeds
        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        logits = ...  # self.out_head(x)
        return logits

torch.manual_seed(123)
model = GPTModel(TEST_CFG)
sample_batch = torch.tensor([[1, 2, 3, 4], [5, 6, 7, 8]])
logits = model(sample_batch)
print("Output shape:", logits.shape)

total_params = sum(p.numel() for p in model.parameters())
print("Total parameters (test config):", total_params)

In [ ]:
%%ipytest -qq

def test_gpt_model_output_shape():
    torch.manual_seed(123)
    m = GPTModel(TEST_CFG)
    x = torch.tensor([[1, 2, 3, 4], [5, 6, 7, 8]])
    out = m(x)
    assert out.shape == (2, 4, TEST_CFG["vocab_size"])

def test_gpt_model_adds_positional_info():
    # shuffling token order must change the output (attention + position embeddings are order-sensitive)
    torch.manual_seed(123)
    m = GPTModel(TEST_CFG)
    x = torch.tensor([[1, 2, 3, 4]])
    x_shuffled = torch.tensor([[4, 3, 2, 1]])
    out = m(x)
    out_shuffled = m(x_shuffled)
    assert not torch.allclose(out, out_shuffled)

def test_gpt_model_has_expected_submodules():
    m = GPTModel(TEST_CFG)
    assert isinstance(m.tok_emb, nn.Embedding) and m.tok_emb.num_embeddings == TEST_CFG["vocab_size"]
    assert isinstance(m.pos_emb, nn.Embedding) and m.pos_emb.num_embeddings == TEST_CFG["context_length"]
    assert len(m.trf_blocks) == TEST_CFG["n_layers"]
    assert isinstance(m.final_norm, LayerNorm)
    assert m.out_head.out_features == TEST_CFG["vocab_size"]

def test_full_124m_config_param_count():
    torch.manual_seed(123)
    real_model = GPTModel(GPT_CONFIG_124M)
    total = sum(p.numel() for p in real_model.parameters())
    assert total == 163_009_536

<button id="claim-ex21" onclick="window.gamifyClaim('ex21', 4, this)" style="padding:6px 14px;border-radius:8px;border:1.5px solid #16a34a;background:#f0fdf4;color:#15803d;font-weight:600;font-size:13px;cursor:pointer;font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Helvetica,Arial,sans-serif">
✅ Mark passed (+4 pts, 4 tests)
</button>

<details ontoggle="window.gamifyReveal(this, 'ex21', 4)"><summary>Show solution (4 tests, −4 pts if opened)</summary>

```python
class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])
        self.trf_blocks = nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])])
        self.final_norm = LayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)

    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx)
        pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))
        x = tok_embeds + pos_embeds
        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        logits = self.out_head(x)
        return logits

torch.manual_seed(123)
model = GPTModel(TEST_CFG)
sample_batch = torch.tensor([[1, 2, 3, 4], [5, 6, 7, 8]])
logits = model(sample_batch)
print("Output shape:", logits.shape)

total_params = sum(p.numel() for p in model.parameters())
print("Total parameters (test config):", total_params)
```

</details>

The last test instantiates the *real* `GPT_CONFIG_124M` — 163,009,536 parameters. That's not 124M despite the name: GPT-2's own implementation **ties** `out_head`'s weights to `tok_emb`'s (reusing the same 768×50,257 matrix for both "token → embedding" and "embedding → logits" — a common trick that shaves off ~38M redundant parameters). This notebook's `GPTModel` keeps them separate, matching how the book first presents it in this chapter; weight tying gets applied later in chapter 5 when loading OpenAI's actual pretrained weights.

### 4.7 — Generating text

The final piece: turn `GPTModel`'s output logits into actual next-token predictions, one token at a time, feeding each prediction back in as input for the next step. This is **greedy decoding** — always picking the single most probable next token — the simplest possible generation strategy (chapter 5 adds randomness via temperature and top-k sampling).

### Exercise 6 — generate_text_simple

At each step: crop `idx` to the model's `context_size` (the model was never trained on positions beyond that), run a forward pass, take only the logits at the *last* position, and append the argmax token to `idx`.

In [ ]:
# TODO: implement this exercise
def generate_text_simple(model, idx, max_new_tokens, context_size):
    for _ in range(max_new_tokens):
        idx_cond = ...  # idx[:, -context_size:]  -- crop to the model's max context
        with torch.no_grad():
            logits = model(idx_cond)
        logits = ...  # logits[:, -1, :]  -- only the last position's logits: (batch, vocab_size)
        probas = torch.softmax(logits, dim=-1)
        idx_next = ...  # torch.argmax(probas, dim=-1, keepdim=True)
        idx = torch.cat((idx, idx_next), dim=1)
    return idx

torch.manual_seed(123)
model = GPTModel(TEST_CFG)
start_idx = torch.tensor([[1, 2]])
out = generate_text_simple(model, start_idx, max_new_tokens=5, context_size=TEST_CFG["context_length"])
print(out)

In [ ]:
%%ipytest -qq

def test_generate_appends_correct_number_of_tokens():
    torch.manual_seed(123)
    m = GPTModel(TEST_CFG)
    start = torch.tensor([[1, 2]])
    out = generate_text_simple(m, start, max_new_tokens=5, context_size=TEST_CFG["context_length"])
    assert out.shape == (1, 7)

def test_generate_preserves_original_tokens():
    torch.manual_seed(123)
    m = GPTModel(TEST_CFG)
    start = torch.tensor([[1, 2]])
    out = generate_text_simple(m, start, max_new_tokens=3, context_size=TEST_CFG["context_length"])
    assert torch.equal(out[:, :2], start)

def test_generate_is_deterministic_greedy():
    torch.manual_seed(123)
    m = GPTModel(TEST_CFG)
    start = torch.tensor([[1, 2]])
    out1 = generate_text_simple(m, start.clone(), max_new_tokens=4, context_size=TEST_CFG["context_length"])
    out2 = generate_text_simple(m, start.clone(), max_new_tokens=4, context_size=TEST_CFG["context_length"])
    assert torch.equal(out1, out2)

def test_generate_crops_context_beyond_context_size():
    torch.manual_seed(123)
    m = GPTModel(TEST_CFG)
    long_start = torch.randint(0, TEST_CFG["vocab_size"], (1, TEST_CFG["context_length"] + 3))
    # should not raise, despite input already exceeding context_size before generation starts
    out = generate_text_simple(m, long_start, max_new_tokens=2, context_size=TEST_CFG["context_length"])
    assert out.shape == (1, TEST_CFG["context_length"] + 5)

<button id="claim-ex22" onclick="window.gamifyClaim('ex22', 4, this)" style="padding:6px 14px;border-radius:8px;border:1.5px solid #16a34a;background:#f0fdf4;color:#15803d;font-weight:600;font-size:13px;cursor:pointer;font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Helvetica,Arial,sans-serif">
✅ Mark passed (+4 pts, 4 tests)
</button>

<details ontoggle="window.gamifyReveal(this, 'ex22', 4)"><summary>Show solution (4 tests, −4 pts if opened)</summary>

```python
def generate_text_simple(model, idx, max_new_tokens, context_size):
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -context_size:]
        with torch.no_grad():
            logits = model(idx_cond)
        logits = logits[:, -1, :]
        probas = torch.softmax(logits, dim=-1)
        idx_next = torch.argmax(probas, dim=-1, keepdim=True)
        idx = torch.cat((idx, idx_next), dim=1)
    return idx

torch.manual_seed(123)
model = GPTModel(TEST_CFG)
start_idx = torch.tensor([[1, 2]])
out = generate_text_simple(model, start_idx, max_new_tokens=5, context_size=TEST_CFG["context_length"])
print(out)
```

</details>

Untrained, this produces gibberish token ids — expected, since every weight is still randomly initialized. Chapter 5 is entirely about fixing that.

### Checkpoint: chapter 4 — the GPT architecture

1. Why pre-norm (`LayerNorm` before each sublayer) instead of post-norm (after, like the original 2017 Transformer)?

<details><summary>Show answer</summary>

Pre-norm keeps the residual/shortcut path completely "clean" — the raw, unnormalized `x` flows through every addition unchanged, so gradients have an unobstructed path back to early layers regardless of how deep the stack is. Post-norm applies normalization *after* the addition, which interferes with that clean path. Empirically, pre-norm trains noticeably more stably in deep transformer stacks, which is why GPT-2 onward switched to it.

</details>

2. `GPTModel`'s embedding dimension (`emb_dim`) never changes anywhere inside `trf_blocks` — every block takes `(batch, seq_len, emb_dim)` in and returns the exact same shape out. Why is that a deliberate design choice?

<details><summary>Show answer</summary>

It's what makes `nn.Sequential(*blocks)` possible at all — stacking modules back-to-back only works if each one's output shape matches the next one's expected input shape. This fixed-width "residual stream" is a standard Transformer design pattern: shape only changes twice in the whole model, once at the embedding layer (going in) and once at `out_head` (going out).

</details>

3. `generate_text_simple` recomputes the *entire* forward pass from scratch at every new token, even though most of the sequence hasn't changed since the last step. Why not cache and reuse those earlier computations?

<details><summary>Show answer</summary>

You can, and production inference code does (this is called a "KV cache" — caching each layer's computed keys/values for previous positions so they don't need recomputing every step). The book keeps `generate_text_simple` uncached because it's clearer pedagogically, at the cost of being `O(n²)` in sequence length instead of `O(n)` per generation call. Appendix D covers some of these production concerns.

</details>

## Chapter 5 — Pretraining on unlabeled data

`GPTModel` from chapter 4 can already run a forward pass and generate text — it just generates garbage, because every weight is still randomly initialized. This chapter is entirely about fixing that: define a loss, run gradient descent, and watch the generated text stop being random.

Like chapter 4, exercises here use a small custom config (`PRETRAIN_CFG`) so training runs in seconds rather than minutes — but note it keeps the **real** GPT-2 vocabulary size (50,257), since we're training on real BPE-tokenized text this time.

In [ ]:
PRETRAIN_CFG = dict(GPT_CONFIG_124M)
PRETRAIN_CFG.update({"context_length": 6, "emb_dim": 12, "n_heads": 3, "n_layers": 2, "drop_rate": 0.0})
print(PRETRAIN_CFG)

### Utility functions for token id <-> text conversion

Small but load-bearing: `generate_text_simple` operates on integer id tensors, not strings, so every training/generation exercise from here on needs to convert back and forth.

### Exercise 1 — text_to_token_ids and token_ids_to_text

`text_to_token_ids` encodes text into ids and adds a batch dimension (`unsqueeze(0)`) since the model always expects `(batch, seq_len)`. `token_ids_to_text` reverses both steps.

In [ ]:
# TODO: implement this exercise
def text_to_token_ids(text, tokenizer):
    encoded = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
    encoded_tensor = ...  # torch.tensor(encoded), then add a batch dimension
    return encoded_tensor

def token_ids_to_text(token_ids, tokenizer):
    flat = ...  # remove the batch dimension from token_ids
    return tokenizer.decode(flat.tolist())

gpt2_tokenizer = tiktoken.get_encoding("gpt2")
demo_ids = text_to_token_ids("Every effort moves you", gpt2_tokenizer)
print(demo_ids.shape)
print(token_ids_to_text(demo_ids, gpt2_tokenizer))

In [ ]:
%%ipytest -qq

def test_text_to_token_ids_adds_batch_dim():
    ids = text_to_token_ids("Every effort moves you", gpt2_tokenizer)
    assert ids.dim() == 2
    assert ids.shape[0] == 1

def test_text_to_token_ids_matches_raw_encode():
    ids = text_to_token_ids("Every effort moves you", gpt2_tokenizer)
    expected = gpt2_tokenizer.encode("Every effort moves you")
    assert ids[0].tolist() == expected

def test_roundtrip():
    text = "Every effort moves you"
    assert token_ids_to_text(text_to_token_ids(text, gpt2_tokenizer), gpt2_tokenizer) == text

def test_token_ids_to_text_removes_batch_dim():
    ids = torch.tensor([[464, 2068, 7586]])
    result = token_ids_to_text(ids, gpt2_tokenizer)
    assert isinstance(result, str)

<button id="claim-ex23" onclick="window.gamifyClaim('ex23', 4, this)" style="padding:6px 14px;border-radius:8px;border:1.5px solid #16a34a;background:#f0fdf4;color:#15803d;font-weight:600;font-size:13px;cursor:pointer;font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Helvetica,Arial,sans-serif">
✅ Mark passed (+4 pts, 4 tests)
</button>

<details ontoggle="window.gamifyReveal(this, 'ex23', 4)"><summary>Show solution (4 tests, −4 pts if opened)</summary>

```python
def text_to_token_ids(text, tokenizer):
    encoded = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
    encoded_tensor = torch.tensor(encoded).unsqueeze(0)
    return encoded_tensor

def token_ids_to_text(token_ids, tokenizer):
    flat = token_ids.squeeze(0)
    return tokenizer.decode(flat.tolist())

gpt2_tokenizer = tiktoken.get_encoding("gpt2")
demo_ids = text_to_token_ids("Every effort moves you", gpt2_tokenizer)
print(demo_ids.shape)
print(token_ids_to_text(demo_ids, gpt2_tokenizer))
```

</details>

### 5.1.2 — Calculating the training and validation set losses

The loss is cross-entropy between predicted next-token logits and the actual next tokens — exactly the classification loss you'd use for any multi-class problem, just applied at every position of every sequence in the batch simultaneously. `logits.flatten(0, 1)` collapses `(batch, seq_len, vocab_size)` into `(batch*seq_len, vocab_size)` and `target_batch.flatten()` collapses targets to match, so `F.cross_entropy` sees one big batch of independent predictions.

**Perplexity**, a number you'll see quoted alongside loss in the book and in most LLM papers, is just `exp(cross_entropy_loss)` — converting the loss back out of log-space into "the model's effective uncertainty, in units of vocabulary choices." A perplexity of 1,000 roughly means "the model is about as unsure as if it were guessing uniformly among 1,000 equally likely next tokens" — more interpretable than a raw loss number, and directly comparable across models trained on different vocabularies. This notebook reports raw loss throughout (matching the book's own training-loop prints), but converting is one line if you want it: `perplexity = torch.exp(loss)`.

### Exercise 2 — calc_loss_batch and calc_loss_loader

Implement `calc_loss_batch` (loss for one batch) and `calc_loss_loader` (average loss over up to `num_batches` batches from a `DataLoader`, or all of them if `num_batches is None`).

In [ ]:
# TODO: implement this exercise
import torch.nn.functional as F

def calc_loss_batch(input_batch, target_batch, model, device):
    input_batch = input_batch.to(device)
    target_batch = target_batch.to(device)
    logits = model(input_batch)
    loss = ...  # F.cross_entropy(logits.flatten(0, 1), target_batch.flatten())
    return loss

def calc_loss_loader(data_loader, model, device, num_batches=None):
    total_loss = 0.
    if len(data_loader) == 0:
        return float("nan")
    elif num_batches is None:
        num_batches = len(data_loader)
    else:
        num_batches = min(num_batches, len(data_loader))
    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i < num_batches:
            loss = ...  # calc_loss_batch(input_batch, target_batch, model, device)
            total_loss += loss.item()
        else:
            break
    return total_loss / num_batches

device = "cpu"
train_ratio = 0.90
split_idx = int(train_ratio * len(raw_text))
train_data = raw_text[:split_idx]
val_data = raw_text[split_idx:]

torch.manual_seed(123)
pretrain_train_loader = create_dataloader_v1(
    train_data, batch_size=2, max_length=PRETRAIN_CFG["context_length"],
    stride=PRETRAIN_CFG["context_length"], drop_last=True, shuffle=True, num_workers=0
)
pretrain_val_loader = create_dataloader_v1(
    val_data, batch_size=2, max_length=PRETRAIN_CFG["context_length"],
    stride=PRETRAIN_CFG["context_length"], drop_last=False, shuffle=False, num_workers=0
)

torch.manual_seed(123)
pretrain_model = GPTModel(PRETRAIN_CFG)
pretrain_model.to(device)

initial_train_loss = calc_loss_loader(pretrain_train_loader, pretrain_model, device)
initial_val_loss = calc_loss_loader(pretrain_val_loader, pretrain_model, device)
print("Train loss:", initial_train_loss)
print("Val loss:", initial_val_loss)

In [ ]:
%%ipytest -qq
import math

def test_calc_loss_batch_returns_scalar_tensor():
    x, y = next(iter(pretrain_train_loader))
    loss = calc_loss_batch(x, y, pretrain_model, device)
    assert loss.dim() == 0

def test_calc_loss_batch_matches_manual_cross_entropy():
    x, y = next(iter(pretrain_train_loader))
    logits = pretrain_model(x.to(device))
    expected = F.cross_entropy(logits.flatten(0, 1), y.to(device).flatten())
    assert torch.isclose(calc_loss_batch(x, y, pretrain_model, device), expected)

def test_untrained_model_loss_near_random_guess_baseline():
    # cross-entropy of a uniform random guess over vocab_size classes is ln(vocab_size)
    baseline = math.log(PRETRAIN_CFG["vocab_size"])
    assert abs(initial_train_loss - baseline) < 1.0

def test_calc_loss_loader_averages_over_num_batches():
    # use a non-shuffled loader here: pretrain_train_loader reshuffles on every fresh
    # iter(), so two separate manual iterations wouldn't see the same batches in the
    # same order -- unrelated to calc_loss_loader itself, purely a fair-comparison issue
    fixed_loader = create_dataloader_v1(
        train_data, batch_size=2, max_length=PRETRAIN_CFG["context_length"],
        stride=PRETRAIN_CFG["context_length"], drop_last=True, shuffle=False, num_workers=0
    )
    single_batch_losses = []
    for i, (x, y) in enumerate(fixed_loader):
        if i >= 3:
            break
        single_batch_losses.append(calc_loss_batch(x, y, pretrain_model, device).item())
    manual_avg = sum(single_batch_losses) / 3
    assert math.isclose(calc_loss_loader(fixed_loader, pretrain_model, device, num_batches=3), manual_avg, rel_tol=1e-5)

<button id="claim-ex24" onclick="window.gamifyClaim('ex24', 4, this)" style="padding:6px 14px;border-radius:8px;border:1.5px solid #16a34a;background:#f0fdf4;color:#15803d;font-weight:600;font-size:13px;cursor:pointer;font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Helvetica,Arial,sans-serif">
✅ Mark passed (+4 pts, 4 tests)
</button>

<details ontoggle="window.gamifyReveal(this, 'ex24', 4)"><summary>Show solution (4 tests, −4 pts if opened)</summary>

```python
import torch.nn.functional as F

def calc_loss_batch(input_batch, target_batch, model, device):
    input_batch = input_batch.to(device)
    target_batch = target_batch.to(device)
    logits = model(input_batch)
    loss = F.cross_entropy(logits.flatten(0, 1), target_batch.flatten())
    return loss

def calc_loss_loader(data_loader, model, device, num_batches=None):
    total_loss = 0.
    if len(data_loader) == 0:
        return float("nan")
    elif num_batches is None:
        num_batches = len(data_loader)
    else:
        num_batches = min(num_batches, len(data_loader))
    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i < num_batches:
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            total_loss += loss.item()
        else:
            break
    return total_loss / num_batches

device = "cpu"
train_ratio = 0.90
split_idx = int(train_ratio * len(raw_text))
train_data = raw_text[:split_idx]
val_data = raw_text[split_idx:]

torch.manual_seed(123)
pretrain_train_loader = create_dataloader_v1(
    train_data, batch_size=2, max_length=PRETRAIN_CFG["context_length"],
    stride=PRETRAIN_CFG["context_length"], drop_last=True, shuffle=True, num_workers=0
)
pretrain_val_loader = create_dataloader_v1(
    val_data, batch_size=2, max_length=PRETRAIN_CFG["context_length"],
    stride=PRETRAIN_CFG["context_length"], drop_last=False, shuffle=False, num_workers=0
)

torch.manual_seed(123)
pretrain_model = GPTModel(PRETRAIN_CFG)
pretrain_model.to(device)

initial_train_loss = calc_loss_loader(pretrain_train_loader, pretrain_model, device)
initial_val_loss = calc_loss_loader(pretrain_val_loader, pretrain_model, device)
print("Train loss:", initial_train_loss)
print("Val loss:", initial_val_loss)
```

</details>

### 5.1.3 — The training loop

Standard PyTorch training loop shape (`zero_grad` → forward → loss → `backward` → `step`), plus periodic evaluation and a text sample after every epoch so you can watch generation quality improve qualitatively, not just watch the loss number drop.

<img src="https://raw.githubusercontent.com/cleophasmashiri/ai-jupter-notebooks/main/llm_from_scratch/images/14-the-training-loop-with-periodic-evaluation.png" alt="The training loop, with periodic evaluation" style="max-width:100%;height:auto;display:block;margin:1em auto" width="532" height="672"/>

Four lines (`zero_grad` → forward-via-`calc_loss_batch` → `backward` → `step`) are the entire learning algorithm; everything else in `train_model_simple` is bookkeeping (tracking losses, periodic evaluation, printing a sample) around that core.

`evaluate_model` and `generate_and_print_sample` are given directly below — both are direct reuses of things already implemented above (`calc_loss_loader` and `generate_text_simple` + the token/text helpers), not new concepts.

### Exercise 3 — train_model_simple

Implement the book's main training function (listing 5.3): the four-line training step, wrapped in an epoch loop, with periodic evaluation.

In [ ]:
def evaluate_model(model, train_loader, val_loader, device, eval_iter):
    model.eval()
    with torch.no_grad():
        train_loss = calc_loss_loader(train_loader, model, device, num_batches=eval_iter)
        val_loss = calc_loss_loader(val_loader, model, device, num_batches=eval_iter)
    model.train()
    return train_loss, val_loss

def generate_and_print_sample(model, tokenizer, device, start_context):
    model.eval()
    context_size = model.pos_emb.weight.shape[0]
    encoded = text_to_token_ids(start_context, tokenizer).to(device)
    with torch.no_grad():
        token_ids = generate_text_simple(model=model, idx=encoded, max_new_tokens=10, context_size=context_size)
    decoded_text = token_ids_to_text(token_ids, tokenizer)
    print(decoded_text.replace("\n", " "))
    model.train()

In [ ]:
# TODO: implement this exercise
def train_model_simple(model, train_loader, val_loader, optimizer, device,
                        num_epochs, eval_freq, eval_iter, start_context, tokenizer):
    train_losses, val_losses, track_tokens_seen = [], [], []
    tokens_seen, global_step = 0, -1

    for epoch in range(num_epochs):
        model.train()
        for input_batch, target_batch in train_loader:
            optimizer.zero_grad()
            loss = ...   # calc_loss_batch(...)
            loss.backward()
            optimizer.step()
            tokens_seen += input_batch.numel()
            global_step += 1

            if global_step % eval_freq == 0:
                train_loss, val_loss = evaluate_model(model, train_loader, val_loader, device, eval_iter)
                train_losses.append(train_loss)
                val_losses.append(val_loss)
                track_tokens_seen.append(tokens_seen)
                print(f"Ep {epoch+1} (Step {global_step:06d}): "
                      f"Train loss {train_loss:.3f}, Val loss {val_loss:.3f}")

        generate_and_print_sample(model, tokenizer, device, start_context)

    return train_losses, val_losses, track_tokens_seen

torch.manual_seed(123)
pretrain_model = GPTModel(PRETRAIN_CFG)
pretrain_model.to(device)
optimizer = torch.optim.AdamW(pretrain_model.parameters(), lr=0.01, weight_decay=0.1)

train_losses, val_losses, tokens_seen = train_model_simple(
    pretrain_model, pretrain_train_loader, pretrain_val_loader, optimizer, device,
    num_epochs=5, eval_freq=5, eval_iter=1,
    start_context="Every effort moves you", tokenizer=gpt2_tokenizer
)

In [ ]:
%%ipytest -qq

def test_train_losses_recorded():
    assert len(train_losses) > 0
    assert len(train_losses) == len(val_losses) == len(tokens_seen)

def test_loss_decreases_over_training():
    assert train_losses[-1] < train_losses[0]

def test_tokens_seen_is_nondecreasing():
    assert all(tokens_seen[i] <= tokens_seen[i+1] for i in range(len(tokens_seen)-1))

def test_optimizer_actually_updated_weights():
    # a fresh model with the same seed should NOT match the trained one anymore
    torch.manual_seed(123)
    fresh_model = GPTModel(PRETRAIN_CFG)
    trained_param = next(pretrain_model.parameters())
    fresh_param = next(fresh_model.parameters())
    assert not torch.allclose(trained_param, fresh_param)

<button id="claim-ex25" onclick="window.gamifyClaim('ex25', 4, this)" style="padding:6px 14px;border-radius:8px;border:1.5px solid #16a34a;background:#f0fdf4;color:#15803d;font-weight:600;font-size:13px;cursor:pointer;font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Helvetica,Arial,sans-serif">
✅ Mark passed (+4 pts, 4 tests)
</button>

<details ontoggle="window.gamifyReveal(this, 'ex25', 4)"><summary>Show solution (4 tests, −4 pts if opened)</summary>

```python
def train_model_simple(model, train_loader, val_loader, optimizer, device,
                        num_epochs, eval_freq, eval_iter, start_context, tokenizer):
    train_losses, val_losses, track_tokens_seen = [], [], []
    tokens_seen, global_step = 0, -1

    for epoch in range(num_epochs):
        model.train()
        for input_batch, target_batch in train_loader:
            optimizer.zero_grad()
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            loss.backward()
            optimizer.step()
            tokens_seen += input_batch.numel()
            global_step += 1

            if global_step % eval_freq == 0:
                train_loss, val_loss = evaluate_model(model, train_loader, val_loader, device, eval_iter)
                train_losses.append(train_loss)
                val_losses.append(val_loss)
                track_tokens_seen.append(tokens_seen)
                print(f"Ep {epoch+1} (Step {global_step:06d}): "
                      f"Train loss {train_loss:.3f}, Val loss {val_loss:.3f}")

        generate_and_print_sample(model, tokenizer, device, start_context)

    return train_losses, val_losses, track_tokens_seen

torch.manual_seed(123)
pretrain_model = GPTModel(PRETRAIN_CFG)
pretrain_model.to(device)
optimizer = torch.optim.AdamW(pretrain_model.parameters(), lr=0.01, weight_decay=0.1)

train_losses, val_losses, tokens_seen = train_model_simple(
    pretrain_model, pretrain_train_loader, pretrain_val_loader, optimizer, device,
    num_epochs=5, eval_freq=5, eval_iter=1,
    start_context="Every effort moves you", tokenizer=gpt2_tokenizer
)
```

</details>

### 5.3 — Decoding strategies to control randomness

`generate_text_simple` always picks the single highest-probability token (greedy decoding) — completely deterministic, and prone to repetitive loops in practice ("the picture, the picture, the picture..."). Two independent knobs fix this:

- **Temperature scaling**: divide logits by `temperature` before the softmax. `temperature < 1` sharpens the distribution (more confident, closer to greedy); `temperature > 1` flattens it (more random); `temperature == 1` leaves it unchanged.
- **Top-k sampling**: before applying temperature/softmax, zero out (`-inf`) every logit outside the `k` highest — this prevents temperature from ever letting a wildly implausible token get sampled, no matter how flat the distribution gets.

<img src="https://raw.githubusercontent.com/cleophasmashiri/ai-jupter-notebooks/main/llm_from_scratch/images/15-effect-of-temperature-on-the-softmax-distribution.png" alt="Effect of temperature on the softmax distribution" style="max-width:100%;height:auto;display:block;margin:1em auto" width="700"/>

Both knobs sit inside the same autoregressive loop every `generate*` function in this notebook uses — one new token per iteration, fed right back in as input for the next:

<img src="https://raw.githubusercontent.com/cleophasmashiri/ai-jupter-notebooks/main/llm_from_scratch/images/16-autoregressive-generation-loop-with-temperature-to.png" alt="Autoregressive generation loop, with temperature/top-k/eos" style="max-width:100%;height:auto;display:block;margin:1em auto" width="712" height="712"/>

Every generated token becomes part of the input for generating the *next* one — this is what "autoregressive" means concretely, and it's why generation is inherently sequential (each step needs the previous step's output) even though training itself is fully parallel across all positions at once.

### Exercise 4 — softmax_with_temperature

Implement temperature-scaled softmax: divide `logits` by `temperature`, then softmax.

In [ ]:
# TODO: implement this exercise
def softmax_with_temperature(logits, temperature):
    scaled_logits = ...  # logits / temperature
    return torch.softmax(scaled_logits, dim=0)

logits_example = torch.tensor([1.0, 2.0, 3.0])
print("T=1 (unchanged):  ", softmax_with_temperature(logits_example, 1.0))
print("T=0.1 (sharper):  ", softmax_with_temperature(logits_example, 0.1))
print("T=5 (flatter):    ", softmax_with_temperature(logits_example, 5.0))

In [ ]:
%%ipytest -qq

def test_temperature_1_equals_plain_softmax():
    assert torch.allclose(softmax_with_temperature(logits_example, 1.0), torch.softmax(logits_example, dim=0))

def test_low_temperature_sharpens_distribution():
    low_t = softmax_with_temperature(logits_example, 0.1)
    baseline = torch.softmax(logits_example, dim=0)
    assert low_t.max() > baseline.max()

def test_high_temperature_flattens_distribution():
    high_t = softmax_with_temperature(logits_example, 5.0)
    baseline = torch.softmax(logits_example, dim=0)
    assert high_t.max() < baseline.max()

def test_output_always_sums_to_one():
    for t in [0.1, 1.0, 5.0]:
        assert torch.isclose(softmax_with_temperature(logits_example, t).sum(), torch.tensor(1.0), atol=1e-5)

<button id="claim-ex26" onclick="window.gamifyClaim('ex26', 4, this)" style="padding:6px 14px;border-radius:8px;border:1.5px solid #16a34a;background:#f0fdf4;color:#15803d;font-weight:600;font-size:13px;cursor:pointer;font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Helvetica,Arial,sans-serif">
✅ Mark passed (+4 pts, 4 tests)
</button>

<details ontoggle="window.gamifyReveal(this, 'ex26', 4)"><summary>Show solution (4 tests, −4 pts if opened)</summary>

```python
def softmax_with_temperature(logits, temperature):
    scaled_logits = logits / temperature
    return torch.softmax(scaled_logits, dim=0)

logits_example = torch.tensor([1.0, 2.0, 3.0])
print("T=1 (unchanged):  ", softmax_with_temperature(logits_example, 1.0))
print("T=0.1 (sharper):  ", softmax_with_temperature(logits_example, 0.1))
print("T=5 (flatter):    ", softmax_with_temperature(logits_example, 5.0))
```

</details>

### Exercise 5 — generate (temperature + top-k + optional early stopping)

Implement the book's upgraded `generate` function (listing 5.4): same crop-and-forward-pass loop as `generate_text_simple`, but with optional top-k filtering before temperature-scaled sampling, falling back to greedy `argmax` when `temperature == 0.0`, and optional early stopping on an `eos_id` token.

In [ ]:
# TODO: implement this exercise
def generate(model, idx, max_new_tokens, context_size, temperature=0.0, top_k=None, eos_id=None):
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -context_size:]
        with torch.no_grad():
            logits = model(idx_cond)
        logits = logits[:, -1, :]

        if top_k is not None:
            top_logits, _ = torch.topk(logits, top_k)
            min_val = top_logits[:, -1]
            logits = ...  # torch.where(logits < min_val, -inf, logits) -- filter out everything below the k-th largest

        if temperature > 0.0:
            logits = logits / temperature
            probs = torch.softmax(logits, dim=-1)
            idx_next = ...  # torch.multinomial(probs, num_samples=1) -- sample instead of argmax
        else:
            idx_next = torch.argmax(logits, dim=-1, keepdim=True)

        if idx_next == eos_id:
            break

        idx = torch.cat((idx, idx_next), dim=1)
    return idx

torch.manual_seed(123)
start_idx = text_to_token_ids("Every effort moves you", gpt2_tokenizer)
greedy_out = generate(pretrain_model, start_idx, max_new_tokens=10, context_size=PRETRAIN_CFG["context_length"], temperature=0.0)
print("Greedy:", token_ids_to_text(greedy_out, gpt2_tokenizer))

torch.manual_seed(123)
sampled_out = generate(pretrain_model, start_idx, max_new_tokens=10, context_size=PRETRAIN_CFG["context_length"], temperature=1.4, top_k=25)
print("Sampled:", token_ids_to_text(sampled_out, gpt2_tokenizer))

In [ ]:
%%ipytest -qq

def test_generate_output_length():
    torch.manual_seed(123)
    start = text_to_token_ids("Every effort moves you", gpt2_tokenizer)
    out = generate(pretrain_model, start, max_new_tokens=5, context_size=PRETRAIN_CFG["context_length"], temperature=0.0)
    assert out.shape[1] == start.shape[1] + 5

def test_top_k_1_forces_deterministic_greedy_equivalent_choice():
    # sampling with top_k=1 has only one candidate token to draw from every step,
    # so it must exactly match plain greedy argmax regardless of temperature/seed
    torch.manual_seed(1)
    start = text_to_token_ids("Every effort moves you", gpt2_tokenizer)
    greedy = generate(pretrain_model, start, max_new_tokens=5, context_size=PRETRAIN_CFG["context_length"], temperature=0.0)
    torch.manual_seed(2)
    top1 = generate(pretrain_model, start, max_new_tokens=5, context_size=PRETRAIN_CFG["context_length"], temperature=1.0, top_k=1)
    assert torch.equal(greedy, top1)

def test_temperature_zero_is_deterministic_across_runs():
    start = text_to_token_ids("Every effort moves you", gpt2_tokenizer)
    out1 = generate(pretrain_model, start.clone(), max_new_tokens=5, context_size=PRETRAIN_CFG["context_length"], temperature=0.0)
    out2 = generate(pretrain_model, start.clone(), max_new_tokens=5, context_size=PRETRAIN_CFG["context_length"], temperature=0.0)
    assert torch.equal(out1, out2)

def test_generate_respects_eos_id_by_stopping_early():
    # force eos_id to be exactly the model's own first greedy prediction, so generation
    # must stop on step 1. Note `break` runs BEFORE the `torch.cat` line in `generate`,
    # so the eos token itself is never appended -- output length stays unchanged, it
    # does NOT grow by one. That asymmetry (compare to generate_text_simple, which always
    # appends) is exactly what early stopping being a "break," not a special append, means.
    start = text_to_token_ids("Every effort moves you", gpt2_tokenizer)
    with torch.no_grad():
        first_logits = pretrain_model(start)[:, -1, :]
    forced_eos = torch.argmax(first_logits, dim=-1).item()
    out = generate(pretrain_model, start.clone(), max_new_tokens=20, context_size=PRETRAIN_CFG["context_length"],
                    temperature=0.0, eos_id=forced_eos)
    assert out.shape[1] == start.shape[1]

<button id="claim-ex27" onclick="window.gamifyClaim('ex27', 4, this)" style="padding:6px 14px;border-radius:8px;border:1.5px solid #16a34a;background:#f0fdf4;color:#15803d;font-weight:600;font-size:13px;cursor:pointer;font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Helvetica,Arial,sans-serif">
✅ Mark passed (+4 pts, 4 tests)
</button>

<details ontoggle="window.gamifyReveal(this, 'ex27', 4)"><summary>Show solution (4 tests, −4 pts if opened)</summary>

```python
def generate(model, idx, max_new_tokens, context_size, temperature=0.0, top_k=None, eos_id=None):
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -context_size:]
        with torch.no_grad():
            logits = model(idx_cond)
        logits = logits[:, -1, :]

        if top_k is not None:
            top_logits, _ = torch.topk(logits, top_k)
            min_val = top_logits[:, -1]
            logits = torch.where(logits < min_val, torch.tensor(float('-inf')).to(logits.device), logits)

        if temperature > 0.0:
            logits = logits / temperature
            probs = torch.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
        else:
            idx_next = torch.argmax(logits, dim=-1, keepdim=True)

        if idx_next == eos_id:
            break

        idx = torch.cat((idx, idx_next), dim=1)
    return idx

torch.manual_seed(123)
start_idx = text_to_token_ids("Every effort moves you", gpt2_tokenizer)
greedy_out = generate(pretrain_model, start_idx, max_new_tokens=10, context_size=PRETRAIN_CFG["context_length"], temperature=0.0)
print("Greedy:", token_ids_to_text(greedy_out, gpt2_tokenizer))

torch.manual_seed(123)
sampled_out = generate(pretrain_model, start_idx, max_new_tokens=10, context_size=PRETRAIN_CFG["context_length"], temperature=1.4, top_k=25)
print("Sampled:", token_ids_to_text(sampled_out, gpt2_tokenizer))
```

</details>

### 5.4 — Saving and loading model weights

Training is expensive; a model's learned weights (`state_dict()`, a dict mapping parameter names to tensors) should outlive the Python process that trained it. `torch.save`/`torch.load` handle this directly — and since `AdamW` (like most PyTorch optimizers) tracks per-parameter running statistics, saving the optimizer's state too lets you resume training exactly where you left off, not just do inference with the trained weights.

In [ ]:
torch.save(pretrain_model.state_dict(), "model.pth")

restored_model = GPTModel(PRETRAIN_CFG)
restored_model.load_state_dict(torch.load("model.pth", map_location=device, weights_only=True))
restored_model.eval()

# checkpoint including optimizer state, for resuming training later
torch.save({
    "model_state_dict": pretrain_model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
}, "model_and_optimizer.pth")

checkpoint = torch.load("model_and_optimizer.pth", map_location=device, weights_only=True)
restored_model2 = GPTModel(PRETRAIN_CFG)
restored_model2.load_state_dict(checkpoint["model_state_dict"])
restored_optimizer = torch.optim.AdamW(restored_model2.parameters(), lr=0.01, weight_decay=0.1)
restored_optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

# sanity check: restored model produces identical output to the original
verify_input = text_to_token_ids("Every effort moves you", gpt2_tokenizer)
with torch.no_grad():
    original_out = pretrain_model(verify_input)
    restored_out = restored_model(verify_input)
print("Restored weights match:", torch.allclose(original_out, restored_out))

Section 5.5 of the book (not reproduced as an exercise here, since it requires downloading OpenAI's ~500MB TensorFlow GPT-2 checkpoint files) shows how to load OpenAI's *actual* pretrained GPT-2 weights into this exact `GPTModel` class — proof that the architecture built across chapters 3–4 is not just "GPT-shaped" but bit-for-bit the real thing, modulo the weight-tying difference noted in chapter 4. The loading code maps each TensorFlow parameter name to its `GPTModel` equivalent and copies values in with `param = torch.nn.Parameter(torch.tensor(pretrained_weights))`; conceptually it's the same `state_dict()`-loading pattern shown above, just from a different weight format.

### Checkpoint: chapter 5 — pretraining

1. Why is cross-entropy loss computed over `logits.flatten(0, 1)` and `target_batch.flatten()` instead of iterating position by position?

<details><summary>Show answer</summary>

`F.cross_entropy` expects a batch of independent predictions — shape `(N, num_classes)` — and a matching batch of labels, shape `(N,)`. Flattening the `(batch, seq_len, vocab_size)` logits into `(batch*seq_len, vocab_size)` treats every position of every sequence as one independent classification example, which is exactly correct here: next-token prediction at position `t` doesn't need special handling relative to position `t'` — they're both "predict one token from vocab_size classes," just conditioned on different context.

</details>

2. Why does `evaluate_model` wrap its loss computation in `model.eval()` and `torch.no_grad()`?

<details><summary>Show answer</summary>

`model.eval()` disables dropout (so evaluation numbers are stable and reproducible rather than randomly varying run to run) — irrelevant with `PRETRAIN_CFG`'s `drop_rate=0.0` here, but essential once dropout is nonzero. `torch.no_grad()` skips building the autograd graph, which evaluation doesn't need (no `.backward()` call follows) — pure memory/compute savings with zero effect on the numbers.

</details>

3. With `top_k=1`, why does `generate(..., temperature=1.0, top_k=1)` always produce the exact same output as `generate(..., temperature=0.0)` (greedy), regardless of the random seed?

<details><summary>Show answer</summary>

`top_k=1` filters every logit except the single highest one down to `-inf`, so after softmax exactly one token has nonzero probability (1.0) — `torch.multinomial` has nothing to randomly choose between, it's forced to return that one token every time. That's identical to what `torch.argmax` would return directly. Randomness in sampling only shows up once `top_k > 1` gives the sampler multiple real candidates to choose between.

</details>

Chapters 6 and 7 both start from a pretrained `GPTModel` (in practice, one loaded with real GPT-2 weights via the `load_weights_into_gpt` pattern noted above) and fine-tune it for a specific task — everything from here on assumes the model already has *some* competence at language, not the random-init state these chapter-5 exercises trained from scratch.

## Chapter 6 — Fine-tuning for classification

### 6.1 — Different categories of fine-tuning

Chapters 2–5 built and pretrained a general-purpose next-token predictor — a **base model**, competent at completing text plausibly but with no particular task in mind. Chapters 6 and 7 fine-tune copies of that same base model two structurally different ways, and it's worth seeing the contrast up front since chapter 7 will otherwise look like an unexplained departure from this chapter's approach:

<img src="https://raw.githubusercontent.com/cleophasmashiri/ai-jupter-notebooks/main/llm_from_scratch/images/17-two-fine-tuning-strategies-branching-from-the-same.png" alt="Two fine-tuning strategies branching from the same base model" style="max-width:100%;height:auto;display:block;margin:1em auto" width="772" height="272"/>

Classification fine-tuning narrows the model down to one specific, closed-ended judgment (spam vs. not-spam here). Instruction fine-tuning keeps the model's full generative range and instead teaches it *when and how* to use it — following instructions phrased in natural language rather than emitting one of a fixed set of labels. That's why chapter 6 replaces `out_head`, and chapter 7 doesn't.

This chapter builds the classification path: binary classification — is this SMS message spam or not? — using the [UCI SMS Spam Collection](https://archive.ics.uci.edu/dataset/228/sms+spam+collection) dataset, 5,572 real text messages labeled `ham`/`spam`.

The core idea: a pretrained GPT already has rich internal representations of language from next-token-prediction pretraining. Classification fine-tuning **reuses almost the entire model as-is** and only swaps + trains a small piece at the very end — far cheaper than training a classifier from scratch, and it works because the pretrained representations already capture the kind of features (tone, word choice, structure) that happen to be exactly what spam detection needs.

In [ ]:
import urllib.request
import zipfile
import os
import pandas as pd
from pathlib import Path

url = "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"
zip_path = "sms_spam_collection.zip"
extracted_path = "sms_spam_collection"
data_file_path = Path(extracted_path) / "SMSSpamCollection.tsv"

def download_and_unzip_spam_data(url, zip_path, extracted_path, data_file_path):
    if data_file_path.exists():
        print(f"{data_file_path} already exists. Skipping download and extraction.")
        return
    with urllib.request.urlopen(url) as response:
        with open(zip_path, "wb") as out_file:
            out_file.write(response.read())
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(extracted_path)
    original_file_path = Path(extracted_path) / "SMSSpamCollection"
    os.rename(original_file_path, data_file_path)
    print(f"File downloaded and saved as {data_file_path}")

download_and_unzip_spam_data(url, zip_path, extracted_path, data_file_path)

df = pd.read_csv(data_file_path, sep="\t", header=None, names=["Label", "Text"])
print(df["Label"].value_counts())

### 6.2 — Preparing the dataset

Two problems before this is trainable: the classes are wildly imbalanced (4,825 `ham` vs. 747 `spam` — a model could get 87% accuracy by always guessing `ham`), and the labels are strings, not the integers a loss function needs.

<img src="https://raw.githubusercontent.com/cleophasmashiri/ai-jupter-notebooks/main/llm_from_scratch/images/18-sms-spam-dataset-class-balance-before-and-after-un.png" alt="SMS spam dataset class balance, before and after undersampling" style="max-width:100%;height:auto;display:block;margin:1em auto" width="560"/>

### Exercise 1 — Balance the dataset and encode labels

Implement `create_balanced_dataset` (book listing 6.2): undersample the majority class (`ham`) down to the same count as the minority class (`spam`), using `df.sample(n, random_state=123)` for a reproducible random subset. Then map the string labels to integers.

In [ ]:
# TODO: implement this exercise
def create_balanced_dataset(df):
    num_spam = df[df["Label"] == "spam"].shape[0]
    ham_subset = ...  # df[df["Label"] == "ham"].sample(num_spam, random_state=123)
    balanced_df = ...  # pd.concat([ham_subset, df[df["Label"] == "spam"]])
    return balanced_df

balanced_df = create_balanced_dataset(df)
print(balanced_df["Label"].value_counts())

balanced_df["Label"] = ...  # map "ham" -> 0, "spam" -> 1
print(balanced_df["Label"].value_counts())

In [ ]:
%%ipytest -qq

def test_balanced_dataset_has_equal_classes():
    d = create_balanced_dataset(df)
    counts = d["Label"].value_counts()
    assert counts["ham"] == counts["spam"] == 747

def test_balanced_dataset_total_size():
    d = create_balanced_dataset(df)
    assert len(d) == 747 * 2

def test_labels_are_integers_after_mapping():
    assert set(balanced_df["Label"].unique()) == {0, 1}

def test_spam_maps_to_one_ham_maps_to_zero():
    d2 = create_balanced_dataset(df)
    d2["Label"] = d2["Label"].map({"ham": 0, "spam": 1})
    spam_rows = d2[d2["Label"] == 1]
    assert len(spam_rows) == 747

<button id="claim-ex28" onclick="window.gamifyClaim('ex28', 4, this)" style="padding:6px 14px;border-radius:8px;border:1.5px solid #16a34a;background:#f0fdf4;color:#15803d;font-weight:600;font-size:13px;cursor:pointer;font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Helvetica,Arial,sans-serif">
✅ Mark passed (+4 pts, 4 tests)
</button>

<details ontoggle="window.gamifyReveal(this, 'ex28', 4)"><summary>Show solution (4 tests, −4 pts if opened)</summary>

```python
def create_balanced_dataset(df):
    num_spam = df[df["Label"] == "spam"].shape[0]
    ham_subset = df[df["Label"] == "ham"].sample(num_spam, random_state=123)
    balanced_df = pd.concat([ham_subset, df[df["Label"] == "spam"]])
    return balanced_df

balanced_df = create_balanced_dataset(df)
print(balanced_df["Label"].value_counts())

balanced_df["Label"] = balanced_df["Label"].map({"ham": 0, "spam": 1})
print(balanced_df["Label"].value_counts())
```

</details>

### Exercise 2 — Split into train / validation / test

Implement `random_split` (book listing 6.3): shuffle the whole dataframe (`frac=1, random_state=123` — sampling 100% of rows is a standard pandas idiom for "shuffle everything"), then slice it into 70% train / 10% validation / 20% test.

(One deliberate naming deviation from the book below: the global test-split variable is named `df_test`, not `test_df` — this notebook's `%%ipytest` cells wipe any global whose name matches `test_*` before every test run, so any real data accidentally named that way would vanish. `random_split`'s own local variable is still called `test_df`, exactly as in the book, since that's unaffected.)

In [ ]:
# TODO: implement this exercise
def random_split(df, train_frac, validation_frac):
    df = df.sample(frac=1, random_state=123).reset_index(drop=True)
    train_end = ...          # int(len(df) * train_frac)
    validation_end = ...     # train_end + int(len(df) * validation_frac)
    train_df = ...           # df[:train_end]
    validation_df = ...      # df[train_end:validation_end]
    test_df = ...            # df[validation_end:]
    return train_df, validation_df, test_df

train_df, validation_df, df_test = random_split(balanced_df, 0.7, 0.1)
print(len(train_df), len(validation_df), len(df_test))

train_df.to_csv("train.csv", index=None)
validation_df.to_csv("validation.csv", index=None)
df_test.to_csv("test.csv", index=None)

In [ ]:
%%ipytest -qq

def test_split_sizes():
    tr, va, te = random_split(balanced_df, 0.7, 0.1)
    assert len(tr) == int(len(balanced_df) * 0.7)
    assert len(va) == int(len(balanced_df) * 0.1)
    assert len(tr) + len(va) + len(te) == len(balanced_df)

def test_split_is_a_partition_no_overlap():
    tr, va, te = random_split(balanced_df, 0.7, 0.1)
    idx_all = set(tr.index) | set(va.index) | set(te.index)
    assert len(idx_all) == len(tr) + len(va) + len(te)  # no duplicated rows across splits

def test_split_uses_all_rows_shuffled():
    tr, va, te = random_split(balanced_df, 0.7, 0.1)
    combined_labels = pd.concat([tr["Label"], va["Label"], te["Label"]])
    assert sorted(combined_labels.tolist()) == sorted(balanced_df["Label"].tolist())

<button id="claim-ex29" onclick="window.gamifyClaim('ex29', 3, this)" style="padding:6px 14px;border-radius:8px;border:1.5px solid #16a34a;background:#f0fdf4;color:#15803d;font-weight:600;font-size:13px;cursor:pointer;font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Helvetica,Arial,sans-serif">
✅ Mark passed (+3 pts, 3 tests)
</button>

<details ontoggle="window.gamifyReveal(this, 'ex29', 3)"><summary>Show solution (3 tests, −3 pts if opened)</summary>

```python
def random_split(df, train_frac, validation_frac):
    df = df.sample(frac=1, random_state=123).reset_index(drop=True)
    train_end = int(len(df) * train_frac)
    validation_end = train_end + int(len(df) * validation_frac)
    train_df = df[:train_end]
    validation_df = df[train_end:validation_end]
    test_df = df[validation_end:]
    return train_df, validation_df, test_df

train_df, validation_df, df_test = random_split(balanced_df, 0.7, 0.1)
print(len(train_df), len(validation_df), len(df_test))

train_df.to_csv("train.csv", index=None)
validation_df.to_csv("validation.csv", index=None)
df_test.to_csv("test.csv", index=None)
```

</details>

### 6.3 — Creating data loaders

Unlike chapters 2–5's sliding-window dataset, every SMS text here is a short, separate, independent example — so this `Dataset` pads (rather than truncates via sliding windows) every text up to a common length using GPT-2's `<|endoftext|>` token id (`50256`) as the pad token, so a batch of variously-long texts can still be stacked into one rectangular tensor.

### Exercise 3 — SpamDataset

Implement the book's `SpamDataset` (listing 6.4): pre-tokenize every text, determine `max_length` (the longest encoded text, if not given explicitly), then truncate + pad every sequence to exactly that length.

In [ ]:
# TODO: implement this exercise
class SpamDataset(Dataset):
    def __init__(self, csv_file, tokenizer, max_length=None, pad_token_id=50256):
        self.data = pd.read_csv(csv_file)
        self.encoded_texts = [
            tokenizer.encode(text) for text in self.data["Text"]
        ]
        if max_length is None:
            self.max_length = self._longest_encoded_length()
        else:
            self.max_length = max_length

        self.encoded_texts = [
            encoded_text[:self.max_length]
            for encoded_text in self.encoded_texts
        ]
        self.encoded_texts = [
            ...  # pad each encoded_text with pad_token_id up to self.max_length
            for encoded_text in self.encoded_texts
        ]

    def __getitem__(self, index):
        encoded = self.encoded_texts[index]
        label = self.data.iloc[index]["Label"]
        return (
            torch.tensor(encoded, dtype=torch.long),
            torch.tensor(label, dtype=torch.long)
        )

    def __len__(self):
        return len(self.data)

    def _longest_encoded_length(self):
        return max(len(encoded_text) for encoded_text in self.encoded_texts)

spam_tokenizer = tiktoken.get_encoding("gpt2")
train_dataset = SpamDataset("train.csv", spam_tokenizer, max_length=None)
print("max_length:", train_dataset.max_length)
val_dataset = SpamDataset("validation.csv", spam_tokenizer, max_length=train_dataset.max_length)
dataset_test = SpamDataset("test.csv", spam_tokenizer, max_length=train_dataset.max_length)

spam_train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, drop_last=True, num_workers=0)
spam_val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, drop_last=False, num_workers=0)
spam_test_loader = DataLoader(dataset_test, batch_size=8, shuffle=False, drop_last=False, num_workers=0)

x_batch, y_batch = next(iter(spam_train_loader))
print(x_batch.shape, y_batch.shape)

In [ ]:
%%ipytest -qq

def test_train_dataset_max_length_is_longest_text():
    manual_max = max(len(spam_tokenizer.encode(t)) for t in pd.read_csv("train.csv")["Text"])
    assert train_dataset.max_length == manual_max

def test_all_sequences_padded_to_same_length():
    for i in range(len(train_dataset)):
        encoded, _ = train_dataset[i]
        assert encoded.shape[0] == train_dataset.max_length

def test_val_and_test_use_train_max_length_not_their_own():
    assert val_dataset.max_length == train_dataset.max_length
    assert dataset_test.max_length == train_dataset.max_length

def test_padding_uses_endoftext_token_id():
    for i in range(5):
        encoded, _ = train_dataset[i]
        raw_len = len(spam_tokenizer.encode(train_dataset.data.iloc[i]["Text"])[:train_dataset.max_length])
        if raw_len < train_dataset.max_length:
            assert (encoded[raw_len:] == 50256).all()

def test_batch_shapes():
    x, y = next(iter(spam_train_loader))
    assert x.shape == (8, train_dataset.max_length)
    assert y.shape == (8,)

<button id="claim-ex30" onclick="window.gamifyClaim('ex30', 5, this)" style="padding:6px 14px;border-radius:8px;border:1.5px solid #16a34a;background:#f0fdf4;color:#15803d;font-weight:600;font-size:13px;cursor:pointer;font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Helvetica,Arial,sans-serif">
✅ Mark passed (+5 pts, 5 tests)
</button>

<details ontoggle="window.gamifyReveal(this, 'ex30', 5)"><summary>Show solution (5 tests, −5 pts if opened)</summary>

```python
class SpamDataset(Dataset):
    def __init__(self, csv_file, tokenizer, max_length=None, pad_token_id=50256):
        self.data = pd.read_csv(csv_file)
        self.encoded_texts = [
            tokenizer.encode(text) for text in self.data["Text"]
        ]
        if max_length is None:
            self.max_length = self._longest_encoded_length()
        else:
            self.max_length = max_length

        self.encoded_texts = [
            encoded_text[:self.max_length]
            for encoded_text in self.encoded_texts
        ]
        self.encoded_texts = [
            encoded_text + [pad_token_id] * (self.max_length - len(encoded_text))
            for encoded_text in self.encoded_texts
        ]

    def __getitem__(self, index):
        encoded = self.encoded_texts[index]
        label = self.data.iloc[index]["Label"]
        return (
            torch.tensor(encoded, dtype=torch.long),
            torch.tensor(label, dtype=torch.long)
        )

    def __len__(self):
        return len(self.data)

    def _longest_encoded_length(self):
        return max(len(encoded_text) for encoded_text in self.encoded_texts)

spam_tokenizer = tiktoken.get_encoding("gpt2")
train_dataset = SpamDataset("train.csv", spam_tokenizer, max_length=None)
print("max_length:", train_dataset.max_length)
val_dataset = SpamDataset("validation.csv", spam_tokenizer, max_length=train_dataset.max_length)
dataset_test = SpamDataset("test.csv", spam_tokenizer, max_length=train_dataset.max_length)

spam_train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, drop_last=True, num_workers=0)
spam_val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, drop_last=False, num_workers=0)
spam_test_loader = DataLoader(dataset_test, batch_size=8, shuffle=False, drop_last=False, num_workers=0)

x_batch, y_batch = next(iter(spam_train_loader))
print(x_batch.shape, y_batch.shape)
```

</details>

### 6.4 & 6.5 — Initializing and modifying a pretrained model for fine-tuning

Here's the actual repurposing step. Starting from a pretrained `GPTModel` (in the book, loaded with real GPT-2 weights — this notebook uses `PRETRAIN_CFG`'s architecture with a fresh random init, since we're not downloading GPT-2's weights here, but the *procedure* is identical either way):

1. **Freeze everything** (`requires_grad = False` on every parameter) — nothing updates by default.
2. **Replace `out_head`**: swap the `(emb_dim -> vocab_size)` next-token-logit layer for a fresh `(emb_dim -> num_classes)` layer. New `nn.Parameter`s always default to `requires_grad=True`, so this one layer is trainable again.
3. **Unfreeze a little more**: the book also unfreezes the *last* transformer block and the final `LayerNorm` — empirically this improves results over training the classification head alone, since it lets the top of the network adapt its representations specifically for classification, not just next-token prediction.

<img src="https://raw.githubusercontent.com/cleophasmashiri/ai-jupter-notebooks/main/llm_from_scratch/images/19-which-layers-are-frozen-vs-trainable-during-classi.png" alt="Which layers are frozen vs. trainable during classification fine-tuning" style="max-width:100%;height:auto;display:block;margin:1em auto" width="812" height="272"/>

Gray = frozen (`requires_grad=False`, untouched by the optimizer); green = trainable. Most of the model's *language* competence, built during pretraining, is preserved as-is — training only touches the small part of the network closest to the task-specific decision.

### Exercise 4 — Turn a GPTModel into a classifier

Implement these three steps against a given `model`.

In [ ]:
# TODO: implement this exercise
CLS_CFG = dict(GPT_CONFIG_124M)
CLS_CFG.update({
    "context_length": train_dataset.max_length,
    "emb_dim": 12, "n_heads": 3, "n_layers": 2, "drop_rate": 0.0,
})

torch.manual_seed(123)
cls_model = GPTModel(CLS_CFG)

# Step 1: freeze every existing parameter
for param in cls_model.parameters():
    param.requires_grad = ...  # False

# Step 2: replace the output head with a fresh, trainable classification layer
torch.manual_seed(123)
num_classes = 2
cls_model.out_head = ...  # nn.Linear(CLS_CFG["emb_dim"], num_classes)

# Step 3: unfreeze the last transformer block and the final LayerNorm
for param in cls_model.trf_blocks[-1].parameters():
    param.requires_grad = ...  # True
for param in cls_model.final_norm.parameters():
    param.requires_grad = ...  # True

trainable_params = sum(p.numel() for p in cls_model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in cls_model.parameters())
print(f"{trainable_params:,} trainable out of {total_params:,} total parameters")

In [ ]:
%%ipytest -qq

def test_out_head_replaced_with_two_class_output():
    assert cls_model.out_head.out_features == 2
    assert cls_model.out_head.in_features == CLS_CFG["emb_dim"]

def test_most_of_model_is_frozen():
    frozen = sum(1 for p in cls_model.tok_emb.parameters() if not p.requires_grad)
    assert frozen > 0  # token embedding must stay frozen

def test_out_head_is_trainable():
    assert all(p.requires_grad for p in cls_model.out_head.parameters())

def test_last_block_and_final_norm_are_trainable():
    assert all(p.requires_grad for p in cls_model.trf_blocks[-1].parameters())
    assert all(p.requires_grad for p in cls_model.final_norm.parameters())

def test_earlier_blocks_remain_frozen():
    assert all(not p.requires_grad for p in cls_model.trf_blocks[0].parameters())

def test_classifier_still_produces_per_class_logits_for_a_batch():
    x, _ = next(iter(spam_train_loader))
    with torch.no_grad():
        out = cls_model(x)
    assert out.shape == (8, train_dataset.max_length, 2)

<button id="claim-ex31" onclick="window.gamifyClaim('ex31', 6, this)" style="padding:6px 14px;border-radius:8px;border:1.5px solid #16a34a;background:#f0fdf4;color:#15803d;font-weight:600;font-size:13px;cursor:pointer;font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Helvetica,Arial,sans-serif">
✅ Mark passed (+6 pts, 6 tests)
</button>

<details ontoggle="window.gamifyReveal(this, 'ex31', 6)"><summary>Show solution (6 tests, −6 pts if opened)</summary>

```python
CLS_CFG = dict(GPT_CONFIG_124M)
CLS_CFG.update({
    "context_length": train_dataset.max_length,
    "emb_dim": 12, "n_heads": 3, "n_layers": 2, "drop_rate": 0.0,
})

torch.manual_seed(123)
cls_model = GPTModel(CLS_CFG)

for param in cls_model.parameters():
    param.requires_grad = False

torch.manual_seed(123)
num_classes = 2
cls_model.out_head = nn.Linear(CLS_CFG["emb_dim"], num_classes)

for param in cls_model.trf_blocks[-1].parameters():
    param.requires_grad = True
for param in cls_model.final_norm.parameters():
    param.requires_grad = True

trainable_params = sum(p.numel() for p in cls_model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in cls_model.parameters())
print(f"{trainable_params:,} trainable out of {total_params:,} total parameters")
```

</details>

### 6.6 — Calculating the classification loss and accuracy

Unlike next-token prediction, the classification target — spam or not — describes the *entire* input text, not each individual token. The book handles this by only using the logits at the **last** token position (`model(input_batch)[:, -1, :]`): by the last position, causal attention has let that token attend to every earlier token in the text, so its representation has "seen" the whole message. Everything from `calc_loss_batch` and `calc_accuracy_loader` onward operates on just that one `(batch, num_classes)` slice.

<img src="https://raw.githubusercontent.com/cleophasmashiri/ai-jupter-notebooks/main/llm_from_scratch/images/20-classification-pipeline-last-token-logits.png" alt="Classification pipeline: last-token logits" style="max-width:100%;height:auto;display:block;margin:1em auto" width="1092" height="132"/>

Every position *before* the last one still computes a next-token-style prediction internally (that's just how the Transformer works — it can't turn that off), but the classification head only ever reads the final position's output; nothing else is used for the class decision.

### Exercise 5 — calc_accuracy_loader and calc_loss_batch (classification variants)

Both reuse the "only the last position" trick above. `calc_loss_batch` is otherwise identical in shape to the chapter 5 version — cross-entropy loss — just against 2 classes instead of `vocab_size`.

In [ ]:
# TODO: implement this exercise
def calc_accuracy_loader(data_loader, model, device, num_batches=None):
    model.eval()
    correct_predictions, num_examples = 0, 0
    if num_batches is None:
        num_batches = len(data_loader)
    else:
        num_batches = min(num_batches, len(data_loader))
    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i < num_batches:
            input_batch, target_batch = input_batch.to(device), target_batch.to(device)
            with torch.no_grad():
                logits = ...  # model(input_batch)[:, -1, :]  -- last-token logits only
            predicted_labels = ...  # torch.argmax(logits, dim=-1)
            num_examples += predicted_labels.shape[0]
            correct_predictions += (predicted_labels == target_batch).sum().item()
        else:
            break
    return correct_predictions / num_examples

def calc_loss_batch(input_batch, target_batch, model, device):
    input_batch, target_batch = input_batch.to(device), target_batch.to(device)
    logits = ...  # model(input_batch)[:, -1, :]  -- last-token logits only
    loss = F.cross_entropy(logits, target_batch)
    return loss

def calc_loss_loader(data_loader, model, device, num_batches=None):
    total_loss = 0.
    if len(data_loader) == 0:
        return float("nan")
    elif num_batches is None:
        num_batches = len(data_loader)
    else:
        num_batches = min(num_batches, len(data_loader))
    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i < num_batches:
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            total_loss += loss.item()
        else:
            break
    return total_loss / num_batches

train_accuracy = calc_accuracy_loader(spam_train_loader, cls_model, device, num_batches=5)
print(f"Initial (untrained) train accuracy: {train_accuracy*100:.2f}%")

In [ ]:
%%ipytest -qq

def test_calc_accuracy_returns_fraction_between_0_and_1():
    acc = calc_accuracy_loader(spam_train_loader, cls_model, device, num_batches=5)
    assert 0.0 <= acc <= 1.0

def test_calc_accuracy_uses_last_token_logits_only():
    x, y = next(iter(spam_train_loader))
    with torch.no_grad():
        expected_preds = torch.argmax(cls_model(x)[:, -1, :], dim=-1)
    assert (expected_preds == y).float().mean().item() == calc_accuracy_loader(
        DataLoader(list(zip(x, y)), batch_size=8), cls_model, device
    )

def test_calc_loss_batch_shape_and_type():
    x, y = next(iter(spam_train_loader))
    loss = calc_loss_batch(x, y, cls_model, device)
    assert loss.dim() == 0
    assert loss.item() > 0

def test_calc_loss_batch_matches_manual_2class_cross_entropy():
    x, y = next(iter(spam_train_loader))
    with torch.no_grad():
        logits = cls_model(x)[:, -1, :]
    expected = F.cross_entropy(logits, y)
    assert torch.isclose(calc_loss_batch(x, y, cls_model, device), expected)

<button id="claim-ex32" onclick="window.gamifyClaim('ex32', 4, this)" style="padding:6px 14px;border-radius:8px;border:1.5px solid #16a34a;background:#f0fdf4;color:#15803d;font-weight:600;font-size:13px;cursor:pointer;font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Helvetica,Arial,sans-serif">
✅ Mark passed (+4 pts, 4 tests)
</button>

<details ontoggle="window.gamifyReveal(this, 'ex32', 4)"><summary>Show solution (4 tests, −4 pts if opened)</summary>

```python
def calc_accuracy_loader(data_loader, model, device, num_batches=None):
    model.eval()
    correct_predictions, num_examples = 0, 0
    if num_batches is None:
        num_batches = len(data_loader)
    else:
        num_batches = min(num_batches, len(data_loader))
    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i < num_batches:
            input_batch, target_batch = input_batch.to(device), target_batch.to(device)
            with torch.no_grad():
                logits = model(input_batch)[:, -1, :]
            predicted_labels = torch.argmax(logits, dim=-1)
            num_examples += predicted_labels.shape[0]
            correct_predictions += (predicted_labels == target_batch).sum().item()
        else:
            break
    return correct_predictions / num_examples

def calc_loss_batch(input_batch, target_batch, model, device):
    input_batch, target_batch = input_batch.to(device), target_batch.to(device)
    logits = model(input_batch)[:, -1, :]
    loss = F.cross_entropy(logits, target_batch)
    return loss

def calc_loss_loader(data_loader, model, device, num_batches=None):
    total_loss = 0.
    if len(data_loader) == 0:
        return float("nan")
    elif num_batches is None:
        num_batches = len(data_loader)
    else:
        num_batches = min(num_batches, len(data_loader))
    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i < num_batches:
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            total_loss += loss.item()
        else:
            break
    return total_loss / num_batches

train_accuracy = calc_accuracy_loader(spam_train_loader, cls_model, device, num_batches=5)
print(f"Initial (untrained) train accuracy: {train_accuracy*100:.2f}%")
```

</details>

### Exercise 6 — train_classifier_simple

Nearly identical in shape to chapter 5's `train_model_simple` — same four-line training step — but tracks accuracy (via `calc_accuracy_loader`) once per epoch in addition to loss, and reports `examples_seen` (text messages) instead of `tokens_seen`, since "how many texts has the classifier trained on" is the more meaningful unit here.

In [ ]:
def evaluate_model(model, train_loader, val_loader, device, eval_iter):
    model.eval()
    with torch.no_grad():
        train_loss = calc_loss_loader(train_loader, model, device, num_batches=eval_iter)
        val_loss = calc_loss_loader(val_loader, model, device, num_batches=eval_iter)
    model.train()
    return train_loss, val_loss

In [ ]:
# TODO: implement this exercise
def train_classifier_simple(model, train_loader, val_loader, optimizer, device,
                             num_epochs, eval_freq, eval_iter):
    train_losses, val_losses, train_accs, val_accs = [], [], [], []
    examples_seen, global_step = 0, -1

    for epoch in range(num_epochs):
        model.train()
        for input_batch, target_batch in train_loader:
            optimizer.zero_grad()
            loss = ...  # calc_loss_batch(...)
            loss.backward()
            optimizer.step()
            examples_seen += input_batch.shape[0]
            global_step += 1

            if global_step % eval_freq == 0:
                train_loss, val_loss = evaluate_model(model, train_loader, val_loader, device, eval_iter)
                train_losses.append(train_loss)
                val_losses.append(val_loss)
                print(f"Ep {epoch+1} (Step {global_step:06d}): "
                      f"Train loss {train_loss:.3f}, Val loss {val_loss:.3f}")

        train_accuracy = calc_accuracy_loader(train_loader, model, device, num_batches=eval_iter)
        val_accuracy = calc_accuracy_loader(val_loader, model, device, num_batches=eval_iter)
        print(f"Training accuracy: {train_accuracy*100:.2f}% | Validation accuracy: {val_accuracy*100:.2f}%")
        train_accs.append(train_accuracy)
        val_accs.append(val_accuracy)

    return train_losses, val_losses, train_accs, val_accs, examples_seen

torch.manual_seed(123)
optimizer = torch.optim.AdamW(cls_model.parameters(), lr=5e-4, weight_decay=0.1)
train_losses, val_losses, train_accs, val_accs, examples_seen = train_classifier_simple(
    cls_model, spam_train_loader, spam_val_loader, optimizer, device,
    num_epochs=5, eval_freq=5, eval_iter=5
)

In [ ]:
%%ipytest -qq

def test_classifier_loss_decreases():
    assert train_losses[-1] < train_losses[0]

def test_classifier_accuracy_improves():
    assert val_accs[-1] > val_accs[0]

def test_examples_seen_matches_epochs_times_dataset_size():
    assert examples_seen == 5 * len(spam_train_loader) * spam_train_loader.batch_size

def test_final_accuracy_beats_random_guessing_baseline():
    # balanced 2-class dataset -> random guessing baseline is 0.5
    assert val_accs[-1] > 0.5

<button id="claim-ex33" onclick="window.gamifyClaim('ex33', 4, this)" style="padding:6px 14px;border-radius:8px;border:1.5px solid #16a34a;background:#f0fdf4;color:#15803d;font-weight:600;font-size:13px;cursor:pointer;font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Helvetica,Arial,sans-serif">
✅ Mark passed (+4 pts, 4 tests)
</button>

<details ontoggle="window.gamifyReveal(this, 'ex33', 4)"><summary>Show solution (4 tests, −4 pts if opened)</summary>

```python
def train_classifier_simple(model, train_loader, val_loader, optimizer, device,
                             num_epochs, eval_freq, eval_iter):
    train_losses, val_losses, train_accs, val_accs = [], [], [], []
    examples_seen, global_step = 0, -1

    for epoch in range(num_epochs):
        model.train()
        for input_batch, target_batch in train_loader:
            optimizer.zero_grad()
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            loss.backward()
            optimizer.step()
            examples_seen += input_batch.shape[0]
            global_step += 1

            if global_step % eval_freq == 0:
                train_loss, val_loss = evaluate_model(model, train_loader, val_loader, device, eval_iter)
                train_losses.append(train_loss)
                val_losses.append(val_loss)
                print(f"Ep {epoch+1} (Step {global_step:06d}): "
                      f"Train loss {train_loss:.3f}, Val loss {val_loss:.3f}")

        train_accuracy = calc_accuracy_loader(train_loader, model, device, num_batches=eval_iter)
        val_accuracy = calc_accuracy_loader(val_loader, model, device, num_batches=eval_iter)
        print(f"Training accuracy: {train_accuracy*100:.2f}% | Validation accuracy: {val_accuracy*100:.2f}%")
        train_accs.append(train_accuracy)
        val_accs.append(val_accuracy)

    return train_losses, val_losses, train_accs, val_accs, examples_seen

torch.manual_seed(123)
optimizer = torch.optim.AdamW(cls_model.parameters(), lr=5e-4, weight_decay=0.1)
train_losses, val_losses, train_accs, val_accs, examples_seen = train_classifier_simple(
    cls_model, spam_train_loader, spam_val_loader, optimizer, device,
    num_epochs=5, eval_freq=5, eval_iter=5
)
```

</details>

### Exercise 7 — classify_review

The inference-time function: tokenize a raw string, truncate/pad it to the same `max_length` the model was trained with, run it through the model, and read off the predicted class from the last-token logits — exactly `calc_accuracy_loader`'s per-example logic, applied to one new piece of text.

In [ ]:
# TODO: implement this exercise
def classify_review(text, model, tokenizer, device, max_length, pad_token_id=50256):
    model.eval()
    input_ids = tokenizer.encode(text)
    supported_context_length = model.pos_emb.weight.shape[0]
    input_ids = input_ids[:min(max_length, supported_context_length)]
    input_ids = ...  # pad input_ids with pad_token_id up to length max_length

    input_tensor = torch.tensor(input_ids, device=device).unsqueeze(0)
    with torch.no_grad():
        logits = model(input_tensor)[:, -1, :]
    predicted_label = ...  # torch.argmax(logits, dim=-1).item()

    return "spam" if predicted_label == 1 else "not spam"

spam_text = (
    "You are a winner you have been specially selected to receive $1000 cash "
    "or a $2000 award."
)
ham_text = "Hey, just wanted to check if we're still on for lunch today?"
print(classify_review(spam_text, cls_model, spam_tokenizer, device, max_length=train_dataset.max_length))
print(classify_review(ham_text, cls_model, spam_tokenizer, device, max_length=train_dataset.max_length))

In [ ]:
%%ipytest -qq

def test_classify_review_returns_valid_label():
    result = classify_review(spam_text, cls_model, spam_tokenizer, device, max_length=train_dataset.max_length)
    assert result in ("spam", "not spam")

def test_classify_review_pads_short_input_to_max_length():
    import inspect
    # a short text must not raise despite being far shorter than max_length
    result = classify_review("ok", cls_model, spam_tokenizer, device, max_length=train_dataset.max_length)
    assert result in ("spam", "not spam")

def test_classify_review_truncates_input_exceeding_max_length():
    long_text = "buy now " * 500  # far longer than any training example
    result = classify_review(long_text, cls_model, spam_tokenizer, device, max_length=train_dataset.max_length)
    assert result in ("spam", "not spam")

def test_classify_review_matches_manual_forward_pass():
    ids = spam_tokenizer.encode(spam_text)[:train_dataset.max_length]
    ids = ids + [50256] * (train_dataset.max_length - len(ids))
    with torch.no_grad():
        expected = torch.argmax(cls_model(torch.tensor(ids).unsqueeze(0))[:, -1, :], dim=-1).item()
    expected_label = "spam" if expected == 1 else "not spam"
    assert classify_review(spam_text, cls_model, spam_tokenizer, device, max_length=train_dataset.max_length) == expected_label

<button id="claim-ex34" onclick="window.gamifyClaim('ex34', 4, this)" style="padding:6px 14px;border-radius:8px;border:1.5px solid #16a34a;background:#f0fdf4;color:#15803d;font-weight:600;font-size:13px;cursor:pointer;font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Helvetica,Arial,sans-serif">
✅ Mark passed (+4 pts, 4 tests)
</button>

<details ontoggle="window.gamifyReveal(this, 'ex34', 4)"><summary>Show solution (4 tests, −4 pts if opened)</summary>

```python
def classify_review(text, model, tokenizer, device, max_length, pad_token_id=50256):
    model.eval()
    input_ids = tokenizer.encode(text)
    supported_context_length = model.pos_emb.weight.shape[0]
    input_ids = input_ids[:min(max_length, supported_context_length)]
    input_ids = input_ids + [pad_token_id] * (max_length - len(input_ids))

    input_tensor = torch.tensor(input_ids, device=device).unsqueeze(0)
    with torch.no_grad():
        logits = model(input_tensor)[:, -1, :]
    predicted_label = torch.argmax(logits, dim=-1).item()

    return "spam" if predicted_label == 1 else "not spam"

spam_text = (
    "You are a winner you have been specially selected to receive $1000 cash "
    "or a $2000 award."
)
ham_text = "Hey, just wanted to check if we're still on for lunch today?"
print(classify_review(spam_text, cls_model, spam_tokenizer, device, max_length=train_dataset.max_length))
print(classify_review(ham_text, cls_model, spam_tokenizer, device, max_length=train_dataset.max_length))
```

</details>

### Checkpoint: chapter 6 — classification fine-tuning

1. Why use the logits at the *last* token position for classification, rather than averaging logits across all positions, or using the first token?

<details><summary>Show answer</summary>

Causal attention means position `t`'s representation can only depend on positions `≤ t` — it never sees the future. The *last* position is the only one that has attended to the entire input text; every earlier position has an incomplete view. Averaging across positions would dilute that full-context representation with several partial-context ones for no benefit.

</details>

2. Why freeze most of the model instead of fine-tuning every parameter?

<details><summary>Show answer</summary>

Two reasons, both mentioned in the book: (1) the lower/earlier layers of a pretrained LLM tend to capture general-purpose language structure that's already useful for almost any downstream task — there's little to gain and some risk of "catastrophic forgetting" by disturbing them; (2) it's dramatically cheaper — far fewer gradients to compute and optimizer states to track. Fine-tuning just the last block + norm + a small new head captures most of the benefit at a fraction of the cost.

</details>

3. If you swapped `nn.Linear(emb_dim, 2)` for `nn.Linear(emb_dim, 5)` and re-ran everything unchanged, what would break, and what wouldn't?

<details><summary>Show answer</summary>

Nothing architecturally breaks — `calc_accuracy_loader`, `calc_loss_batch`, and `classify_review`'s `argmax` logic all generalize to any number of classes without modification, since none of them hardcode "2" anywhere. What *would* need to change: the balanced-dataset construction (currently 2-class specific), and `classify_review`'s final `"spam" if ... == 1 else "not spam"` line, which hardcodes the two label names.

</details>

## Chapter 7 — Fine-tuning to follow instructions

Chapter 6 fine-tuned a pretrained GPT for one narrow, fixed task (spam/not-spam). This chapter fine-tunes the same kind of model for something much more general: given an *instruction* (arbitrary natural language, phrased however a user phrases it), produce an appropriate free-text *response* — the technique behind turning a raw next-token predictor into something that behaves like ChatGPT. The dataset is 1,100 instruction/input/response triples the book provides directly.

Unlike chapter 6 (a new classification head bolted onto a frozen backbone), instruction fine-tuning keeps the **entire original architecture** — `out_head` still predicts next-token logits over the full vocabulary, exactly as in chapter 5. What changes is *what* the model is trained to predict next: given a formatted instruction, predict the tokens of a good response, one at a time, same next-token cross-entropy loss as pretraining, just on a different (much smaller, much more curated) dataset.

In [ ]:
import json
import os
import urllib.request

def download_and_load_file(file_path, url):
    if not os.path.exists(file_path):
        with urllib.request.urlopen(url) as response:
            text_data = response.read().decode("utf-8")
        with open(file_path, "w", encoding="utf-8") as file:
            file.write(text_data)
    else:
        with open(file_path, "r", encoding="utf-8") as file:
            text_data = file.read()
    with open(file_path, "r") as file:
        data = json.load(file)
    return data

file_path = "instruction-data.json"
url = (
    "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch"
    "/main/ch07/01_main-chapter-code/instruction-data.json"
)
instruction_data = download_and_load_file(file_path, url)
print("Number of entries:", len(instruction_data))
print(instruction_data[50])

### 7.2 — Preparing a dataset for supervised instruction fine-tuning

Every entry has `instruction`, an optional `input` (extra context/data the instruction operates on — not every instruction needs one), and `output` (the target response). Before any of this reaches the tokenizer, it needs to become one formatted string the model can learn "instruction in, response out" from. The book uses the **Alpaca prompt style** — a fixed template wrapping the instruction (and input, if present) in section headers the model can learn to recognize as structural markers.

<img src="https://raw.githubusercontent.com/cleophasmashiri/ai-jupter-notebooks/main/llm_from_scratch/images/21-alpaca-style-instruction-prompt-template.png" alt="Alpaca-style instruction prompt template" style="max-width:100%;height:auto;display:block;margin:1em auto" width="632" height="442"/>

The book briefly contrasts this with the alternative **Phi-3 style** (a terser `<|user|>...<|assistant|>...` chat-turn format) — different wrapping, same underlying idea: a fixed, learnable template that marks where the instruction ends and the response the model should generate begins. This notebook uses Alpaca throughout, matching the book's main running example.

### Exercise 1 — format_input

Implement `format_input` (book listing 7.2): build the fixed instruction-description preamble + `### Instruction:` section, then append a `### Input:` section **only if** `entry["input"]` is non-empty.

In [ ]:
# TODO: implement this exercise
def format_input(entry):
    instruction_text = (
        f"Below is an instruction that describes a task. "
        f"Write a response that appropriately completes the request."
        f"\n\n### Instruction:\n{entry['instruction']}"
    )
    input_text = ...  # f"\n\n### Input:\n{entry['input']}" if entry["input"] is non-empty, else ""
    return instruction_text + input_text

model_input = format_input(instruction_data[50])
desired_response = f"\n\n### Response:\n{instruction_data[50]['output']}"
print(model_input + desired_response)

In [ ]:
%%ipytest -qq

def test_format_input_includes_instruction():
    out = format_input(instruction_data[50])
    assert "### Instruction:" in out
    assert instruction_data[50]["instruction"] in out

def test_format_input_includes_input_section_when_present():
    entry = next(e for e in instruction_data if e["input"])
    out = format_input(entry)
    assert "### Input:" in out
    assert entry["input"] in out

def test_format_input_omits_input_section_when_absent():
    entry = next(e for e in instruction_data if not e["input"])
    out = format_input(entry)
    assert "### Input:" not in out

def test_format_input_matches_book_example():
    out = format_input(instruction_data[50])
    assert out == (
        "Below is an instruction that describes a task. Write a response that "
        "appropriately completes the request.\n\n### Instruction:\nIdentify the "
        "correct spelling of the following word.\n\n### Input:\nOcassion"
    )

<button id="claim-ex35" onclick="window.gamifyClaim('ex35', 4, this)" style="padding:6px 14px;border-radius:8px;border:1.5px solid #16a34a;background:#f0fdf4;color:#15803d;font-weight:600;font-size:13px;cursor:pointer;font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Helvetica,Arial,sans-serif">
✅ Mark passed (+4 pts, 4 tests)
</button>

<details ontoggle="window.gamifyReveal(this, 'ex35', 4)"><summary>Show solution (4 tests, −4 pts if opened)</summary>

```python
def format_input(entry):
    instruction_text = (
        f"Below is an instruction that describes a task. "
        f"Write a response that appropriately completes the request."
        f"\n\n### Instruction:\n{entry['instruction']}"
    )
    input_text = f"\n\n### Input:\n{entry['input']}" if entry["input"] else ""
    return instruction_text + input_text

model_input = format_input(instruction_data[50])
desired_response = f"\n\n### Response:\n{instruction_data[50]['output']}"
print(model_input + desired_response)
```

</details>

### Exercise 2 — Train / test / validation split

Straightforward slicing (no shuffling this time — the book keeps the dataset's given order): 85% train, 10% test, 5% validation. Note the book's own ordering — the test slice comes out *before* `val_data`.

(As in chapter 6, the test-split variables here are named `portion_test`/`data_test` rather than `test_portion`/`test_data` — this notebook's `%%ipytest` cells delete any global matching `test_*` before each run, so real data can't use that prefix and survive.)

In [ ]:
# TODO: implement this exercise
train_portion = ...  # int(len(instruction_data) * 0.85)
portion_test = ...   # int(len(instruction_data) * 0.1)
val_portion = ...    # remainder: len(instruction_data) - train_portion - portion_test

train_data = instruction_data[:train_portion]
data_test = instruction_data[train_portion:train_portion + portion_test]
val_data = instruction_data[train_portion + portion_test:]

print("Training set length:", len(train_data))
print("Validation set length:", len(val_data))
print("Test set length:", len(data_test))

In [ ]:
%%ipytest -qq

def test_split_sizes():
    assert len(train_data) == 935
    assert len(data_test) == 110
    assert len(val_data) == 55

def test_split_covers_every_entry_in_original_order():
    assert train_data + data_test + val_data == instruction_data

def test_split_ratios_approximate_85_10_5():
    total = len(instruction_data)
    assert abs(len(train_data) / total - 0.85) < 0.01
    assert abs(len(data_test) / total - 0.10) < 0.01

<button id="claim-ex36" onclick="window.gamifyClaim('ex36', 3, this)" style="padding:6px 14px;border-radius:8px;border:1.5px solid #16a34a;background:#f0fdf4;color:#15803d;font-weight:600;font-size:13px;cursor:pointer;font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Helvetica,Arial,sans-serif">
✅ Mark passed (+3 pts, 3 tests)
</button>

<details ontoggle="window.gamifyReveal(this, 'ex36', 3)"><summary>Show solution (3 tests, −3 pts if opened)</summary>

```python
train_portion = int(len(instruction_data) * 0.85)
portion_test = int(len(instruction_data) * 0.1)
val_portion = len(instruction_data) - train_portion - portion_test

train_data = instruction_data[:train_portion]
data_test = instruction_data[train_portion:train_portion + portion_test]
val_data = instruction_data[train_portion + portion_test:]

print("Training set length:", len(train_data))
print("Validation set length:", len(val_data))
print("Test set length:", len(data_test))
```

</details>

### 7.3 — Organizing data into training batches

Chapter 6's `SpamDataset` padded every example to one fixed `max_length`, computed up front. Instruction-tuning batches are trickier for two reasons the book walks through incrementally:

1. **Targets need the usual next-token shift**, same as pretraining — but now *within* one already-formatted `instruction + response` string, not via a sliding window.
2. **Padding tokens must not contribute to the loss.** If padding were left as an ordinary training target, the model would waste capacity learning "predict `<|endoftext|>` after `<|endoftext|>`" — harmless in itself, but it dilutes the gradient signal that should be entirely about generating good responses. The fix: replace every padding position's target (**except the first** one, which stays as a legitimate "this is where the response ends" signal) with `ignore_index=-100`, a sentinel `F.cross_entropy` is built to skip automatically.

### Exercise 3 — InstructionDataset

Pre-tokenize every entry as `format_input(entry) + "\n\n### Response:\n" + entry["output"]`, one flat token sequence per example (no fixed-length padding yet — that happens per-batch, in the collate function below, so different batches can use different lengths instead of one global max).

In [ ]:
# TODO: implement this exercise
class InstructionDataset(Dataset):
    def __init__(self, data, tokenizer):
        self.data = data
        self.encoded_texts = []
        for entry in data:
            instruction_plus_input = format_input(entry)
            response_text = ...  # f"\n\n### Response:\n{entry['output']}"
            full_text = instruction_plus_input + response_text
            self.encoded_texts.append(tokenizer.encode(full_text))

    def __getitem__(self, index):
        return self.encoded_texts[index]

    def __len__(self):
        return len(self.data)

instr_tokenizer = tiktoken.get_encoding("gpt2")
instr_train_dataset = InstructionDataset(train_data, instr_tokenizer)
print(len(instr_train_dataset))
print(instr_train_dataset[0][:20])

In [ ]:
%%ipytest -qq

def test_dataset_length_matches_train_data():
    assert len(instr_train_dataset) == len(train_data)

def test_each_item_is_a_plain_token_id_list():
    item = instr_train_dataset[0]
    assert isinstance(item, list)
    assert all(isinstance(t, int) for t in item)

def test_encoded_text_includes_full_prompt_and_response():
    expected_text = format_input(train_data[0]) + f"\n\n### Response:\n{train_data[0]['output']}"
    assert instr_train_dataset[0] == instr_tokenizer.encode(expected_text)

def test_items_are_not_padded_uniformly():
    lengths = {len(instr_train_dataset[i]) for i in range(20)}
    assert len(lengths) > 1  # real instruction/response pairs vary in length

<button id="claim-ex37" onclick="window.gamifyClaim('ex37', 4, this)" style="padding:6px 14px;border-radius:8px;border:1.5px solid #16a34a;background:#f0fdf4;color:#15803d;font-weight:600;font-size:13px;cursor:pointer;font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Helvetica,Arial,sans-serif">
✅ Mark passed (+4 pts, 4 tests)
</button>

<details ontoggle="window.gamifyReveal(this, 'ex37', 4)"><summary>Show solution (4 tests, −4 pts if opened)</summary>

```python
class InstructionDataset(Dataset):
    def __init__(self, data, tokenizer):
        self.data = data
        self.encoded_texts = []
        for entry in data:
            instruction_plus_input = format_input(entry)
            response_text = f"\n\n### Response:\n{entry['output']}"
            full_text = instruction_plus_input + response_text
            self.encoded_texts.append(tokenizer.encode(full_text))

    def __getitem__(self, index):
        return self.encoded_texts[index]

    def __len__(self):
        return len(self.data)

instr_tokenizer = tiktoken.get_encoding("gpt2")
instr_train_dataset = InstructionDataset(train_data, instr_tokenizer)
print(len(instr_train_dataset))
print(instr_train_dataset[0][:20])
```

</details>

### Exercise 4 — custom_collate_fn

The most involved function in this notebook — read it slowly. For each item in a batch: append one `pad_token_id` (guaranteeing every sequence has at least one, so there's always an end-of-text signal to learn), pad every sequence up to the batch's longest, then build `inputs`/`targets` via the usual shift-by-one (`padded[:-1]` / `padded[1:]`), and finally mask out every padding target **except the first** with `ignore_index`.

Traced through the book's own 3-item worked example (`[0,1,2,3,4]`, `[5,6]`, `[7,8,9]`, `pad_token_id=50256`):

<img src="https://raw.githubusercontent.com/cleophasmashiri/ai-jupter-notebooks/main/llm_from_scratch/images/22-custom-collate-fn-padding-shifting-and-masking.png" alt="custom_collate_fn: padding, shifting, and masking" style="max-width:100%;height:auto;display:block;margin:1em auto" width="772" height="482"/>

`F.cross_entropy(..., ignore_index=-100)` (chapter 7 exercise 5, below) skips every `-100` position entirely when computing the loss — those positions contribute zero gradient, so the model never gets pushed to "learn" anything from pure batching artifacts.

In [ ]:
# TODO: implement this exercise
def custom_collate_fn(batch, pad_token_id=50256, ignore_index=-100, allowed_max_length=None, device="cpu"):
    batch_max_length = max(len(item) + 1 for item in batch)
    inputs_lst, targets_lst = [], []

    for item in batch:
        new_item = item.copy()
        new_item += [pad_token_id]
        padded = new_item + [pad_token_id] * (batch_max_length - len(new_item))

        inputs = torch.tensor(padded[:-1])
        targets = ...  # torch.tensor(padded[1:])  -- shifted by one, same trick as every chapter since 2

        mask = targets == pad_token_id
        indices = torch.nonzero(mask).squeeze()
        if indices.numel() > 1:
            targets[indices[1:]] = ...  # ignore_index  -- mask every pad target except the first

        if allowed_max_length is not None:
            inputs = inputs[:allowed_max_length]
            targets = targets[:allowed_max_length]

        inputs_lst.append(inputs)
        targets_lst.append(targets)

    inputs_tensor = torch.stack(inputs_lst).to(device)
    targets_tensor = torch.stack(targets_lst).to(device)
    return inputs_tensor, targets_tensor

demo_batch = ([0, 1, 2, 3, 4], [5, 6], [7, 8, 9])
demo_inputs, demo_targets = custom_collate_fn(demo_batch)
print(demo_inputs)
print(demo_targets)

In [ ]:
%%ipytest -qq

def test_collate_pads_to_longest_plus_shift():
    demo_batch = ([0, 1, 2, 3, 4], [5, 6], [7, 8, 9])
    inputs, targets = custom_collate_fn(demo_batch)
    assert inputs.shape == (3, 5)  # batch_max_length (6, from the len-5 item +1) minus 1
    assert targets.shape == (3, 5)

def test_collate_targets_are_inputs_shifted_by_one():
    # a single item with no extra padding beyond its own +1 pad_token_id: no masking kicks in,
    # so the shift-by-one relationship is exact and easy to check directly
    demo_batch = ([0, 1, 2, 3, 4],)
    inputs, targets = custom_collate_fn(demo_batch)
    assert inputs[0].tolist() == [0, 1, 2, 3, 4]
    assert targets[0].tolist() == [1, 2, 3, 4, 50256]
    assert inputs[0, 1:].tolist() == targets[0, :-1].tolist()

def test_collate_masks_all_but_first_padding_target():
    # batched alongside a much longer item, the short item ([5, 6]) picks up real padding
    demo_batch = ([0, 1, 2, 3, 4], [5, 6])
    inputs, targets = custom_collate_fn(demo_batch)
    short_item_targets = targets[1]
    pad_positions = (short_item_targets == -100).nonzero().squeeze(-1)
    endoftext_positions = (short_item_targets == 50256).nonzero().squeeze(-1)
    assert endoftext_positions.numel() == 1  # exactly one un-masked pad token (the first) survives
    assert pad_positions.numel() > 0

def test_collate_respects_allowed_max_length():
    demo_batch = ([0, 1, 2, 3, 4, 5, 6, 7, 8, 9],)
    inputs, targets = custom_collate_fn(demo_batch, allowed_max_length=5)
    assert inputs.shape[1] == 5
    assert targets.shape[1] == 5

def test_collate_matches_book_worked_example():
    demo_batch = ([0, 1, 2, 3, 4], [5, 6], [7, 8, 9])
    inputs, targets = custom_collate_fn(demo_batch)
    assert inputs[0].tolist() == [0, 1, 2, 3, 4]
    assert targets[0].tolist() == [1, 2, 3, 4, 50256]
    assert inputs[1].tolist() == [5, 6, 50256, 50256, 50256]
    assert targets[1].tolist() == [6, 50256, -100, -100, -100]

<button id="claim-ex38" onclick="window.gamifyClaim('ex38', 5, this)" style="padding:6px 14px;border-radius:8px;border:1.5px solid #16a34a;background:#f0fdf4;color:#15803d;font-weight:600;font-size:13px;cursor:pointer;font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Helvetica,Arial,sans-serif">
✅ Mark passed (+5 pts, 5 tests)
</button>

<details ontoggle="window.gamifyReveal(this, 'ex38', 5)"><summary>Show solution (5 tests, −5 pts if opened)</summary>

```python
def custom_collate_fn(batch, pad_token_id=50256, ignore_index=-100, allowed_max_length=None, device="cpu"):
    batch_max_length = max(len(item) + 1 for item in batch)
    inputs_lst, targets_lst = [], []

    for item in batch:
        new_item = item.copy()
        new_item += [pad_token_id]
        padded = new_item + [pad_token_id] * (batch_max_length - len(new_item))

        inputs = torch.tensor(padded[:-1])
        targets = torch.tensor(padded[1:])

        mask = targets == pad_token_id
        indices = torch.nonzero(mask).squeeze()
        if indices.numel() > 1:
            targets[indices[1:]] = ignore_index

        if allowed_max_length is not None:
            inputs = inputs[:allowed_max_length]
            targets = targets[:allowed_max_length]

        inputs_lst.append(inputs)
        targets_lst.append(targets)

    inputs_tensor = torch.stack(inputs_lst).to(device)
    targets_tensor = torch.stack(targets_lst).to(device)
    return inputs_tensor, targets_tensor

demo_batch = ([0, 1, 2, 3, 4], [5, 6], [7, 8, 9])
demo_inputs, demo_targets = custom_collate_fn(demo_batch)
print(demo_inputs)
print(demo_targets)
```

</details>

### Building the data loaders

`functools.partial` bakes in `custom_collate_fn`'s extra arguments (`device`, `allowed_max_length=1024`, GPT-2's own context limit) so `DataLoader` can call it with just `batch`, its own expected signature.

In [ ]:
from functools import partial

device = "cpu"
customized_collate_fn = partial(custom_collate_fn, device=device, allowed_max_length=1024)

instr_train_loader = DataLoader(
    instr_train_dataset, batch_size=2, collate_fn=customized_collate_fn,
    shuffle=True, drop_last=True, num_workers=0
)
instr_val_dataset = InstructionDataset(val_data, instr_tokenizer)
instr_val_loader = DataLoader(
    instr_val_dataset, batch_size=2, collate_fn=customized_collate_fn,
    shuffle=False, drop_last=False, num_workers=0
)

instr_x, instr_y = next(iter(instr_train_loader))
print(instr_x.shape, instr_y.shape)

### 7.4 & 7.5 — Loading a pretrained LLM and fine-tuning it

This is where the book loads real pretrained GPT-2 weights (section 7.5) and fine-tunes with the exact `train_model_simple` from chapter 5 — no new training-loop code at all, only `calc_loss_batch` needs one small change from the chapter 5 version.

### Exercise 5 — calc_loss_batch with ignore_index

Adapt chapter 5's `calc_loss_batch`: identical except `F.cross_entropy` needs `ignore_index=-100` so the padding positions `custom_collate_fn` masked out don't contribute to the loss (without this argument, `-100` would be treated as a real, very-out-of-range class label and crash).

In [ ]:
# TODO: implement this exercise
def calc_loss_batch(input_batch, target_batch, model, device):
    input_batch, target_batch = input_batch.to(device), target_batch.to(device)
    logits = model(input_batch)
    loss = ...  # F.cross_entropy(logits.flatten(0, 1), target_batch.flatten(), ignore_index=-100)
    return loss

INSTR_CFG = dict(GPT_CONFIG_124M)
INSTR_CFG.update({"context_length": 1024, "emb_dim": 12, "n_heads": 3, "n_layers": 2, "drop_rate": 0.0})

torch.manual_seed(123)
instr_model = GPTModel(INSTR_CFG)
first_loss = calc_loss_batch(instr_x, instr_y, instr_model, device)
print(first_loss)

In [ ]:
%%ipytest -qq

def test_calc_loss_batch_ignores_masked_positions():
    torch.manual_seed(0)
    m = GPTModel(INSTR_CFG)
    x, y = next(iter(instr_train_loader))
    loss_with_mask = calc_loss_batch(x, y, m, device)
    # replacing every ignored position with a real (wrong) label must change the loss,
    # proving ignore_index positions were previously excluded
    y_no_mask = y.clone()
    y_no_mask[y_no_mask == -100] = 0
    logits = m(x)
    loss_without_mask = F.cross_entropy(logits.flatten(0, 1), y_no_mask.flatten())
    assert not torch.isclose(loss_with_mask, loss_without_mask)

def test_calc_loss_batch_matches_manual_ignore_index_cross_entropy():
    torch.manual_seed(0)
    m = GPTModel(INSTR_CFG)
    x, y = next(iter(instr_train_loader))
    with torch.no_grad():
        logits = m(x)
    expected = F.cross_entropy(logits.flatten(0, 1), y.flatten(), ignore_index=-100)
    assert torch.isclose(calc_loss_batch(x, y, m, device), expected)

def test_a_few_training_steps_reduce_loss():
    torch.manual_seed(0)
    m = GPTModel(INSTR_CFG)
    opt = torch.optim.AdamW(m.parameters(), lr=5e-4, weight_decay=0.1)
    losses = []
    for i, (x, y) in enumerate(instr_train_loader):
        if i >= 8:
            break
        opt.zero_grad()
        loss = calc_loss_batch(x, y, m, device)
        loss.backward()
        opt.step()
        losses.append(loss.item())
    assert losses[-1] < losses[0]

<button id="claim-ex39" onclick="window.gamifyClaim('ex39', 3, this)" style="padding:6px 14px;border-radius:8px;border:1.5px solid #16a34a;background:#f0fdf4;color:#15803d;font-weight:600;font-size:13px;cursor:pointer;font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Helvetica,Arial,sans-serif">
✅ Mark passed (+3 pts, 3 tests)
</button>

<details ontoggle="window.gamifyReveal(this, 'ex39', 3)"><summary>Show solution (3 tests, −3 pts if opened)</summary>

```python
def calc_loss_batch(input_batch, target_batch, model, device):
    input_batch, target_batch = input_batch.to(device), target_batch.to(device)
    logits = model(input_batch)
    loss = F.cross_entropy(logits.flatten(0, 1), target_batch.flatten(), ignore_index=-100)
    return loss

INSTR_CFG = dict(GPT_CONFIG_124M)
INSTR_CFG.update({"context_length": 1024, "emb_dim": 12, "n_heads": 3, "n_layers": 2, "drop_rate": 0.0})

torch.manual_seed(123)
instr_model = GPTModel(INSTR_CFG)
first_loss = calc_loss_batch(instr_x, instr_y, instr_model, device)
print(first_loss)
```

</details>

From here, fine-tuning is a direct call to chapter 5's `train_model_simple(model, train_loader, val_loader, optimizer, device, num_epochs, eval_freq, eval_iter, start_context, tokenizer)` — every piece (the training step, `evaluate_model`, `generate_and_print_sample`) is unchanged; only what's *in* `train_loader`/`val_loader` is different (instruction data instead of raw pretraining text), which is the entire point of having built these pieces as reusable, task-agnostic functions back in chapter 5.

### 7.7 — Extracting and evaluating responses

Generation (via `generate`, chapter 5) produces `format_input(entry) + " generated continuation..."` — the prompt followed by whatever the model predicted next. Since training targets everything *after* `### Response:\n`, evaluating the response means stripping the echoed-back prompt off the front of whatever `generate_text_simple`/`generate` returned.

### Exercise 6 — Extract the generated response

Given `generated_text` (the full decoded output, prompt included) and `input_text` (the formatted prompt that was fed in), isolate just the newly generated response text.

In [ ]:
# TODO: implement this exercise
torch.manual_seed(123)
input_text = format_input(val_data[0])
token_ids = generate_text_simple(
    model=instr_model,
    idx=text_to_token_ids(input_text, gpt2_tokenizer),
    max_new_tokens=15,
    context_size=INSTR_CFG["context_length"],
)
generated_text = token_ids_to_text(token_ids, gpt2_tokenizer)

response_text = ...  # generated_text, with the leading `input_text` prompt stripped off and whitespace trimmed
print(response_text)

In [ ]:
%%ipytest -qq

def test_response_text_excludes_the_prompt():
    assert input_text not in response_text

def test_response_text_is_suffix_of_generated_text():
    assert generated_text.endswith(response_text) or generated_text.strip().endswith(response_text)

def test_response_extraction_matches_manual_slicing():
    expected = generated_text[len(input_text):].strip()
    assert response_text == expected

def test_full_generated_text_starts_with_the_prompt():
    # sanity check on the setup itself: generation always echoes the prompt verbatim first
    assert generated_text.startswith(input_text)

<button id="claim-ex40" onclick="window.gamifyClaim('ex40', 4, this)" style="padding:6px 14px;border-radius:8px;border:1.5px solid #16a34a;background:#f0fdf4;color:#15803d;font-weight:600;font-size:13px;cursor:pointer;font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Helvetica,Arial,sans-serif">
✅ Mark passed (+4 pts, 4 tests)
</button>

<details ontoggle="window.gamifyReveal(this, 'ex40', 4)"><summary>Show solution (4 tests, −4 pts if opened)</summary>

```python
torch.manual_seed(123)
input_text = format_input(val_data[0])
token_ids = generate_text_simple(
    model=instr_model,
    idx=text_to_token_ids(input_text, gpt2_tokenizer),
    max_new_tokens=15,
    context_size=INSTR_CFG["context_length"],
)
generated_text = token_ids_to_text(token_ids, gpt2_tokenizer)

response_text = generated_text[len(input_text):].strip()
print(response_text)
```

</details>

The book's final evaluation step (section 7.7) scores generated responses using a separate, larger LLM (Llama 3 via Ollama, run locally) as an automated judge — comparing each fine-tuned model response against the dataset's reference `output` and producing a 0–100 quality score. That step isn't reproduced here since it depends on external infrastructure (a running Ollama server) outside this notebook's scope; `response_text` above is exactly the string such a judge would be shown.

### Checkpoint: chapter 7 — instruction fine-tuning

1. Why does `custom_collate_fn` keep the *first* padding token's target as a real label (`pad_token_id`, i.e. `<|endoftext|>`) instead of masking every padding position with `ignore_index`?

<details><summary>Show answer</summary>

The model needs to learn *when to stop generating* — predicting `<|endoftext|>` right after the legitimate response content is a real, useful thing to train it to do (this is exactly what lets `generate`'s `eos_id` early-stopping, from chapter 5, work at inference time). Every padding token *after* that first one is pure batching artifact with no signal to learn from, so those get masked; the first one is the actual "stop here" label.

</details>

2. Why does instruction fine-tuning keep `GPTModel`'s original `out_head` (predicting over the full `vocab_size`), while chapter 6 replaced it with a small classification head?

<details><summary>Show answer</summary>

The task shape is different. Classification has one fixed, closed set of output labels (spam/not-spam) entirely unrelated to "what word comes next," so the vocabulary-sized `out_head` is actively the wrong shape and gets swapped out. Instruction-following is still fundamentally next-token prediction — the model needs to generate open-ended text, one token from the full vocabulary at a time — so the original architecture is *already* the right shape; only the training data changes.

</details>

3. `format_input`'s prompt template is fixed and identical for every example. Why does that matter for fine-tuning to work?

<details><summary>Show answer</summary>

The model has to learn to recognize `### Response:\n` as "now generate the answer," and it can only learn that reliably if the marker consistently means the same thing across the whole training set. A fixed, consistent template turns prompt formatting into a learnable structural signal the model can key off of; inconsistent formatting across examples would blur that signal and make it harder for the model to learn when generation should start.

</details>

This is the last of the numbered chapters — from chapter 2's `re.split()` on a single short story to a model that follows arbitrary natural-language instructions, every step reused something built one or two chapters earlier. That reuse (not any single clever trick) is the actual throughline of the book.

Three appendices follow, each treated here as its own chapter: **Appendix A** (a PyTorch primer — worth doing even now, retroactively, since every mechanic it names — tensors, autograd, `nn.Module`, `Dataset`/`DataLoader`, the training loop — you've already been using since chapter 2, just without it being named explicitly), **Appendix D** (production training-loop upgrades: learning rate warmup, cosine decay, gradient clipping), and **Appendix E** (LoRA — fine-tuning orders of magnitude fewer parameters than chapters 6–7 did).

## Appendix A — Introduction to PyTorch

Every mechanism used from chapter 2 onward — tensors, `nn.Module`, autograd, `Dataset`/`DataLoader`, the training loop shape — is PyTorch, used without much explanation of PyTorch itself along the way. This appendix is the primer the book puts *before* chapter 2 (skip it there if you already know deep learning frameworks); here, after having already used all of it, it works just as well as a "here's what you were actually doing" retrospective. Everything in this section uses a tiny toy dataset, deliberately unrelated to GPT, so the PyTorch mechanics stand on their own.

### A.1 — What is PyTorch?

The book frames PyTorch as three components: (1) a **tensor library** (like NumPy, but tensors can live on a GPU and track gradients), (2) an **automatic differentiation engine** (autograd — computes gradients of any computation you run, automatically), and (3) **deep learning utilities** (`nn.Module`, `Dataset`/`DataLoader`, optimizers, and more, built on top of the first two). Every chapter 2–7 exercise used all three; this appendix names each one explicitly.

### A.2 — Understanding tensors

A **tensor** is PyTorch's data container: a scalar is a 0-D tensor, a vector 1-D, a matrix 2-D, and anything higher-dimensional is just "an *n*-D tensor" (no special name past 2-D). Every input, weight, and activation in this entire notebook has been a tensor.

### Exercise 1 — Tensor shape, reshape, transpose, matmul

Given `tensor2d = torch.tensor([[1,2,3],[4,5,6]])`, compute its shape, a reshaped `(3,2)` view two different ways (`.reshape()` and `.view()` — same result, different memory-layout guarantees: `.view()` requires the underlying data to already be contiguous and raises if it isn't, `.reshape()` copies if needed and always works), its transpose, and a matrix product with itself two ways (`.matmul()` and `@`).

In [ ]:
# TODO: implement this exercise
import torch

tensor2d = torch.tensor([[1, 2, 3], [4, 5, 6]])

t_shape = ...       # tensor2d.shape
t_reshaped = ...     # tensor2d.reshape(3, 2)
t_viewed = ...        # tensor2d.view(3, 2)
t_transposed = ...    # tensor2d.T
t_matmul = ...        # tensor2d.matmul(tensor2d.T)
t_at = ...             # tensor2d @ tensor2d.T

print(t_shape)
print(t_reshaped)
print(t_transposed)
print(t_matmul)

In [ ]:
%%ipytest -qq

def test_shape():
    assert tuple(t_shape) == (2, 3)

def test_reshape_and_view_agree():
    assert torch.equal(t_reshaped, t_viewed)
    assert t_reshaped.shape == (3, 2)
    assert torch.equal(t_reshaped, torch.tensor([[1, 2], [3, 4], [5, 6]]))

def test_transpose():
    assert torch.equal(t_transposed, torch.tensor([[1, 4], [2, 5], [3, 6]]))

def test_matmul_and_at_agree():
    assert torch.equal(t_matmul, t_at)
    assert torch.equal(t_matmul, torch.tensor([[14, 32], [32, 77]]))

<button id="claim-ex41" onclick="window.gamifyClaim('ex41', 4, this)" style="padding:6px 14px;border-radius:8px;border:1.5px solid #16a34a;background:#f0fdf4;color:#15803d;font-weight:600;font-size:13px;cursor:pointer;font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Helvetica,Arial,sans-serif">
✅ Mark passed (+4 pts, 4 tests)
</button>

<details ontoggle="window.gamifyReveal(this, 'ex41', 4)"><summary>Show solution (4 tests, −4 pts if opened)</summary>

```python
import torch

tensor2d = torch.tensor([[1, 2, 3], [4, 5, 6]])

t_shape = tensor2d.shape
t_reshaped = tensor2d.reshape(3, 2)
t_viewed = tensor2d.view(3, 2)
t_transposed = tensor2d.T
t_matmul = tensor2d.matmul(tensor2d.T)
t_at = tensor2d @ tensor2d.T

print(t_shape)
print(t_reshaped)
print(t_transposed)
print(t_matmul)
```

</details>

### A.3 & A.4 — Computation graphs and automatic differentiation

Every operation PyTorch runs on a tensor with `requires_grad=True` gets recorded into a **computation graph** — a record of exactly how each intermediate value was computed from its inputs. `.backward()` walks that graph in reverse (the chain rule, applied automatically) to compute the gradient of some final scalar (usually a loss) with respect to every tensor that fed into it. This is the mechanism behind every single `loss.backward()` call in chapters 5–7 — none of them needed a hand-derived gradient formula, because autograd built and differentiated the graph automatically.

<img src="https://raw.githubusercontent.com/cleophasmashiri/ai-jupter-notebooks/main/llm_from_scratch/images/23-autograd-s-computation-graph-forward-pass-and-back.png" alt="Autograd's computation graph: forward pass and backward gradient flow" style="max-width:100%;height:auto;display:block;margin:1em auto" width="652" height="342"/>

### Exercise 2 — Autograd on a logistic regression forward pass

The book's own worked example: `z = x1*w1 + b`, `a = sigmoid(z)`, `loss = binary_cross_entropy(a, y)`. Compute `∂loss/∂w1` and `∂loss/∂b` two ways — via `torch.autograd.grad()` directly, and via `loss.backward()` + `.grad` — and confirm they agree.

In [ ]:
# TODO: implement this exercise
import torch.nn.functional as F
from torch.autograd import grad

y = torch.tensor([1.0])
x1 = torch.tensor([1.1])
w1 = torch.tensor([2.2], requires_grad=True)
b = torch.tensor([0.0], requires_grad=True)

z = x1 * w1 + b
a = torch.sigmoid(z)
loss = F.binary_cross_entropy(a, y)

grad_L_w1 = ...  # grad(loss, w1, retain_graph=True) -- retain_graph=True: we reuse this graph below
grad_L_b = ...   # grad(loss, b, retain_graph=True)

loss.backward()
backward_w1_grad = ...  # w1.grad
backward_b_grad = ...   # b.grad

print(grad_L_w1, grad_L_b)
print(backward_w1_grad, backward_b_grad)

In [ ]:
%%ipytest -qq

def test_manual_grad_matches_book_values():
    assert torch.isclose(grad_L_w1[0], torch.tensor(-0.0898), atol=1e-4)
    assert torch.isclose(grad_L_b[0], torch.tensor(-0.0817), atol=1e-4)

def test_backward_matches_manual_grad():
    assert torch.isclose(backward_w1_grad, grad_L_w1[0])
    assert torch.isclose(backward_b_grad, grad_L_b[0])

def test_grad_is_a_tuple_backward_grad_is_a_tensor():
    assert isinstance(grad_L_w1, tuple)
    assert isinstance(backward_w1_grad, torch.Tensor)

<button id="claim-ex42" onclick="window.gamifyClaim('ex42', 3, this)" style="padding:6px 14px;border-radius:8px;border:1.5px solid #16a34a;background:#f0fdf4;color:#15803d;font-weight:600;font-size:13px;cursor:pointer;font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Helvetica,Arial,sans-serif">
✅ Mark passed (+3 pts, 3 tests)
</button>

<details ontoggle="window.gamifyReveal(this, 'ex42', 3)"><summary>Show solution (3 tests, −3 pts if opened)</summary>

```python
import torch.nn.functional as F
from torch.autograd import grad

y = torch.tensor([1.0])
x1 = torch.tensor([1.1])
w1 = torch.tensor([2.2], requires_grad=True)
b = torch.tensor([0.0], requires_grad=True)

z = x1 * w1 + b
a = torch.sigmoid(z)
loss = F.binary_cross_entropy(a, y)

grad_L_w1 = grad(loss, w1, retain_graph=True)
grad_L_b = grad(loss, b, retain_graph=True)

loss.backward()
backward_w1_grad = w1.grad
backward_b_grad = b.grad

print(grad_L_w1, grad_L_b)
print(backward_w1_grad, backward_b_grad)
```

</details>

### A.5 — Implementing multilayer neural networks

`nn.Module` is the base class for every learnable component in PyTorch — subclass it, define layers in `__init__`, define how data flows through them in `forward()`, and PyTorch handles parameter tracking, `.to(device)`, `state_dict()`, and gradient bookkeeping for free. This is the exact pattern `SelfAttention_v1`, `GPTModel`, and everything in between were built from.

### Exercise 3 — NeuralNetwork (a plain multilayer perceptron)

Implement the book's toy classifier (listing A.4): two hidden layers (`num_inputs → 30 → 20 → num_outputs`), `ReLU` between them, no activation on the final layer (the book's convention — losses like `cross_entropy` apply softmax internally, so raw logits are what a model should return).

In [ ]:
# TODO: implement this exercise
import torch.nn as nn

class NeuralNetwork(nn.Module):
    def __init__(self, num_inputs, num_outputs):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(num_inputs, 30),
            nn.ReLU(),
            nn.Linear(30, 20),
            nn.ReLU(),
            nn.Linear(20, num_outputs),
        )

    def forward(self, x):
        logits = ...  # self.layers(x)
        return logits

toy_model = NeuralNetwork(50, 3)
num_params = sum(p.numel() for p in toy_model.parameters() if p.requires_grad)
print("Total trainable parameters:", num_params)

torch.manual_seed(123)
toy_X = torch.rand((1, 50))
toy_out = toy_model(toy_X)
print(toy_out)

In [ ]:
%%ipytest -qq

def test_neural_network_param_count():
    m = NeuralNetwork(50, 3)
    n = sum(p.numel() for p in m.parameters() if p.requires_grad)
    assert n == 2213

def test_neural_network_output_shape():
    m = NeuralNetwork(50, 3)
    x = torch.rand(4, 50)
    assert m(x).shape == (4, 3)

def test_neural_network_no_activation_on_output_layer():
    # last Sequential entry must be the final Linear, not an activation
    assert isinstance(toy_model.layers[-1], nn.Linear)
    assert toy_model.layers[-1].out_features == 3

def test_no_grad_context_gives_identical_values_without_graph():
    with torch.no_grad():
        out_no_grad = toy_model(toy_X)
    assert torch.allclose(toy_out, out_no_grad)
    assert toy_out.requires_grad and not out_no_grad.requires_grad

<button id="claim-ex43" onclick="window.gamifyClaim('ex43', 4, this)" style="padding:6px 14px;border-radius:8px;border:1.5px solid #16a34a;background:#f0fdf4;color:#15803d;font-weight:600;font-size:13px;cursor:pointer;font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Helvetica,Arial,sans-serif">
✅ Mark passed (+4 pts, 4 tests)
</button>

<details ontoggle="window.gamifyReveal(this, 'ex43', 4)"><summary>Show solution (4 tests, −4 pts if opened)</summary>

```python
import torch.nn as nn

class NeuralNetwork(nn.Module):
    def __init__(self, num_inputs, num_outputs):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(num_inputs, 30),
            nn.ReLU(),
            nn.Linear(30, 20),
            nn.ReLU(),
            nn.Linear(20, num_outputs),
        )

    def forward(self, x):
        logits = self.layers(x)
        return logits

toy_model = NeuralNetwork(50, 3)
num_params = sum(p.numel() for p in toy_model.parameters() if p.requires_grad)
print("Total trainable parameters:", num_params)

torch.manual_seed(123)
toy_X = torch.rand((1, 50))
toy_out = toy_model(toy_X)
print(toy_out)
```

</details>

### A.6 — Setting up efficient data loaders

Same `Dataset`/`DataLoader` split used throughout this notebook (`GPTDatasetV1` in chapter 2, `SpamDataset` in chapter 6, `InstructionDataset` in chapter 7): a custom `Dataset` defines *how one example is fetched* (`__getitem__`, `__len__`); `DataLoader` wraps it to handle shuffling, batching, and (optionally) parallel loading (`num_workers`) — logic you'd otherwise have to write by hand, every time, for every dataset shape.

<img src="https://raw.githubusercontent.com/cleophasmashiri/ai-jupter-notebooks/main/llm_from_scratch/images/24-the-dataset-dataloader-pattern.png" alt="The Dataset/DataLoader pattern" style="max-width:100%;height:auto;display:block;margin:1em auto" width="772" height="142"/>

### Exercise 4 — ToyDataset

Implement the book's `ToyDataset` (listing A.6) for a small 5-example, 2-feature toy classification set, then build `DataLoader`s from it.

In [ ]:
# TODO: implement this exercise
from torch.utils.data import Dataset, DataLoader

X_train = torch.tensor([
    [-1.2, 3.1], [-0.9, 2.9], [-0.5, 2.6], [2.3, -1.1], [2.7, -1.5]
])
y_train = torch.tensor([0, 0, 0, 1, 1])
X_test = torch.tensor([[-0.8, 2.8], [2.6, -1.6]])
y_test = torch.tensor([0, 1])

class ToyDataset(Dataset):
    def __init__(self, X, y):
        self.features = X
        self.labels = y

    def __getitem__(self, index):
        one_x = ...  # self.features[index]
        one_y = ...  # self.labels[index]
        return one_x, one_y

    def __len__(self):
        return ...  # self.labels.shape[0]

toy_train_ds = ToyDataset(X_train, y_train)
toy_test_ds = ToyDataset(X_test, y_test)
print(len(toy_train_ds))

toy_train_loader = DataLoader(dataset=toy_train_ds, batch_size=2, shuffle=True, num_workers=0)
toy_train_loader_dropped = DataLoader(dataset=toy_train_ds, batch_size=2, shuffle=True, num_workers=0, drop_last=True)
toy_test_loader = DataLoader(dataset=toy_test_ds, batch_size=2, shuffle=False, num_workers=0)

for idx, (x, y) in enumerate(toy_train_loader):
    print(f"Batch {idx+1}:", x, y)

In [ ]:
%%ipytest -qq

def test_toy_dataset_length():
    assert len(toy_train_ds) == 5
    assert len(toy_test_ds) == 2

def test_toy_dataset_getitem():
    x, y = toy_train_ds[0]
    assert torch.equal(x, X_train[0])
    assert torch.equal(y, y_train[0])

def test_dataloader_without_drop_last_has_uneven_final_batch():
    # 5 examples, batch_size=2 -> batches of size 2, 2, 1 (not evenly divisible)
    batch_sizes = [len(y) for _, y in toy_train_loader]
    assert sorted(batch_sizes) == [1, 2, 2]

def test_dataloader_with_drop_last_drops_the_uneven_batch():
    batch_sizes = [len(y) for _, y in toy_train_loader_dropped]
    assert all(bs == 2 for bs in batch_sizes)
    assert len(batch_sizes) == 2  # last, smaller batch dropped entirely

def test_dataloader_visits_every_example_exactly_once_per_epoch():
    seen_labels = []
    for _, y in toy_train_loader:
        seen_labels.extend(y.tolist())
    assert sorted(seen_labels) == sorted(y_train.tolist())

<button id="claim-ex44" onclick="window.gamifyClaim('ex44', 5, this)" style="padding:6px 14px;border-radius:8px;border:1.5px solid #16a34a;background:#f0fdf4;color:#15803d;font-weight:600;font-size:13px;cursor:pointer;font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Helvetica,Arial,sans-serif">
✅ Mark passed (+5 pts, 5 tests)
</button>

<details ontoggle="window.gamifyReveal(this, 'ex44', 5)"><summary>Show solution (5 tests, −5 pts if opened)</summary>

```python
from torch.utils.data import Dataset, DataLoader

X_train = torch.tensor([
    [-1.2, 3.1], [-0.9, 2.9], [-0.5, 2.6], [2.3, -1.1], [2.7, -1.5]
])
y_train = torch.tensor([0, 0, 0, 1, 1])
X_test = torch.tensor([[-0.8, 2.8], [2.6, -1.6]])
y_test = torch.tensor([0, 1])

class ToyDataset(Dataset):
    def __init__(self, X, y):
        self.features = X
        self.labels = y

    def __getitem__(self, index):
        one_x = self.features[index]
        one_y = self.labels[index]
        return one_x, one_y

    def __len__(self):
        return self.labels.shape[0]

toy_train_ds = ToyDataset(X_train, y_train)
toy_test_ds = ToyDataset(X_test, y_test)
print(len(toy_train_ds))

toy_train_loader = DataLoader(dataset=toy_train_ds, batch_size=2, shuffle=True, num_workers=0)
toy_train_loader_dropped = DataLoader(dataset=toy_train_ds, batch_size=2, shuffle=True, num_workers=0, drop_last=True)
toy_test_loader = DataLoader(dataset=toy_test_ds, batch_size=2, shuffle=False, num_workers=0)

for idx, (x, y) in enumerate(toy_train_loader):
    print(f"Batch {idx+1}:", x, y)
```

</details>

### A.7 — A typical training loop

The same four-line core (`zero_grad` → forward → `backward` → `step`) as every training loop since chapter 5 — `train_model_simple` (ch. 5), `train_classifier_simple` (ch. 6), and the reused chapter 5 loop for instruction tuning (ch. 7) are all this pattern with task-specific data and a task-specific loss swapped in.

### Exercise 5 — Train NeuralNetwork on the toy dataset, and compute_accuracy

Train for 3 epochs with plain SGD, then implement `compute_accuracy` (book listing A.10): iterate a `DataLoader`, compare `argmax(logits)` against true labels, and return the fraction correct.

In [ ]:
# TODO: implement this exercise
torch.manual_seed(123)
toy_model = NeuralNetwork(num_inputs=2, num_outputs=2)
optimizer = torch.optim.SGD(toy_model.parameters(), lr=0.5)

num_epochs = 3
toy_losses = []
for epoch in range(num_epochs):
    toy_model.train()
    for batch_idx, (features, labels) in enumerate(toy_train_loader):
        logits = toy_model(features)
        loss = F.cross_entropy(logits, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        toy_losses.append(loss.item())
    toy_model.eval()

def compute_accuracy(model, dataloader):
    model = model.eval()
    correct = 0.0
    total_examples = 0
    for idx, (features, labels) in enumerate(dataloader):
        with torch.no_grad():
            logits = model(features)
        predictions = ...  # torch.argmax(logits, dim=1)
        compare = ...       # predictions == labels
        correct += torch.sum(compare)
        total_examples += len(compare)
    return (correct / total_examples).item()

print("Final training loss:", toy_losses[-1])
print("Train accuracy:", compute_accuracy(toy_model, toy_train_loader))
print("Test accuracy:", compute_accuracy(toy_model, toy_test_loader))

In [ ]:
%%ipytest -qq

def test_toy_training_loss_converges_near_zero():
    assert toy_losses[-1] < 0.1

def test_toy_training_loss_generally_decreases():
    assert toy_losses[-1] < toy_losses[0]

def test_compute_accuracy_is_a_fraction():
    acc = compute_accuracy(toy_model, toy_train_loader)
    assert 0.0 <= acc <= 1.0

def test_compute_accuracy_matches_manual_computation():
    correct, total = 0, 0
    for x, y in toy_test_loader:
        with torch.no_grad():
            preds = torch.argmax(toy_model(x), dim=1)
        correct += (preds == y).sum().item()
        total += len(y)
    assert compute_accuracy(toy_model, toy_test_loader) == correct / total

def test_toy_model_achieves_perfect_accuracy_on_this_trivially_separable_toy_set():
    # the toy dataset's two classes are linearly separable by a wide margin,
    # so after 3 epochs of training the book's own result is 100% on both splits
    assert compute_accuracy(toy_model, toy_train_loader) == 1.0
    assert compute_accuracy(toy_model, toy_test_loader) == 1.0

<button id="claim-ex45" onclick="window.gamifyClaim('ex45', 5, this)" style="padding:6px 14px;border-radius:8px;border:1.5px solid #16a34a;background:#f0fdf4;color:#15803d;font-weight:600;font-size:13px;cursor:pointer;font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Helvetica,Arial,sans-serif">
✅ Mark passed (+5 pts, 5 tests)
</button>

<details ontoggle="window.gamifyReveal(this, 'ex45', 5)"><summary>Show solution (5 tests, −5 pts if opened)</summary>

```python
torch.manual_seed(123)
toy_model = NeuralNetwork(num_inputs=2, num_outputs=2)
optimizer = torch.optim.SGD(toy_model.parameters(), lr=0.5)

num_epochs = 3
toy_losses = []
for epoch in range(num_epochs):
    toy_model.train()
    for batch_idx, (features, labels) in enumerate(toy_train_loader):
        logits = toy_model(features)
        loss = F.cross_entropy(logits, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        toy_losses.append(loss.item())
    toy_model.eval()

def compute_accuracy(model, dataloader):
    model = model.eval()
    correct = 0.0
    total_examples = 0
    for idx, (features, labels) in enumerate(dataloader):
        with torch.no_grad():
            logits = model(features)
        predictions = torch.argmax(logits, dim=1)
        compare = predictions == labels
        correct += torch.sum(compare)
        total_examples += len(compare)
    return (correct / total_examples).item()

print("Final training loss:", toy_losses[-1])
print("Train accuracy:", compute_accuracy(toy_model, toy_train_loader))
print("Test accuracy:", compute_accuracy(toy_model, toy_test_loader))
```

</details>

### A.8 & A.9 — Saving/loading models, and GPUs

Both already appeared with GPT-specific examples: `torch.save(model.state_dict(), "model.pth")` / `model.load_state_dict(torch.load(...))` was exercised in chapter 5's checkpoint section, and `device = "cuda" if torch.cuda.is_available() else "cpu"` + `.to(device)` appears throughout chapters 5–7's `calc_loss_batch` functions. Nothing GPT-specific about either — they're general PyTorch patterns, which is exactly why they didn't need re-explaining when they first showed up.

The one genuinely new idea in A.9: **multi-GPU training** splits either the *data* (each GPU holds a full model copy, processes a different mini-batch, gradients are averaged/synced across GPUs — `DistributedDataParallel`) or, for models too large for one GPU's memory, the *model itself* (different layers on different GPUs). This notebook's configs are deliberately tiny and CPU-friendly, so multi-GPU training isn't exercised here — but nothing about `GPTModel`'s architecture would need to change to scale it up; only the training *infrastructure* around it would.

### Checkpoint: Appendix A — PyTorch fundamentals

1. What's the practical difference between `torch.autograd.grad(loss, w1)` and `loss.backward()` followed by `w1.grad`?

<details><summary>Show answer</summary>

Mathematically identical results — both use the same underlying autograd engine and chain-rule machinery. `grad()` returns the gradient(s) for specifically the tensor(s) you name, without touching any `.grad` attributes; `.backward()` computes gradients for *every* leaf tensor with `requires_grad=True` in the graph and accumulates them into each one's `.grad` attribute. Training loops universally use `.backward()` + `optimizer.step()` (which reads every parameter's `.grad`); `grad()` is more of a debugging/inspection tool, which is exactly how the book introduces it — to make the mechanism concrete before switching to the form real code actually uses.

</details>

2. Why does `NeuralNetwork.forward()` return raw logits instead of applying `softmax` internally?

<details><summary>Show answer</summary>

`F.cross_entropy` (and PyTorch's other common classification losses) apply `softmax` (or `log_softmax`) *internally*, combined with the loss computation in one numerically-stable operation. Applying softmax yourself first and then calling `cross_entropy` on the result would double-apply it, both hurting numerical stability and producing an incorrect loss. `GPTModel.out_head` follows the exact same convention — chapter 4/5's `calc_loss_batch` calls `F.cross_entropy` directly on raw logits, and `generate_text_simple` only applies `softmax` explicitly at *inference* time, when actual probabilities (not just relative ordering) are needed for sampling.

</details>

3. `SpamDataset` (ch. 6) and `InstructionDataset` (ch. 7) both subclass `Dataset` just like `ToyDataset` here — what's actually different between them?

<details><summary>Show answer</summary>

Only `__getitem__`/`__init__` internals — what "one example" means and how it's fetched/encoded. `ToyDataset` returns pre-existing tensors by index; `SpamDataset` tokenizes and pads text at construction time; `InstructionDataset` formats and tokenizes instruction/response pairs. `DataLoader` doesn't care about any of that — it only needs `__getitem__` and `__len__` to exist, which is exactly what makes the `Dataset`/`DataLoader` split reusable across such different data shapes.

</details>

## Appendix D — Adding bells and whistles to the training loop

`train_model_simple` (chapter 5) is deliberately minimal — enough to prove the model learns, not tuned for stability at scale. This appendix adds three techniques real LLM pretraining runs use in practice: **learning rate warmup**, **cosine decay**, and **gradient clipping**. All three modify *when* and *how much* each optimizer step moves the weights — none change the model architecture or the loss function at all.

The book re-initializes a fresh model and re-imports chapter 5's plumbing to keep this appendix self-contained; this notebook does the same explicitly below, since chapters 6 and 7 have since redefined `calc_loss_batch` for their own tasks (last-token-only for classification, `ignore_index`-masked for instruction data) — Appendix D needs the plain, full-sequence pretraining versions back.

In [ ]:
# Re-establish the plain pretraining versions of these functions (chapters 6 and 7
# redefined calc_loss_batch for their own tasks since chapter 5 originally defined it).
def calc_loss_batch(input_batch, target_batch, model, device):
    input_batch, target_batch = input_batch.to(device), target_batch.to(device)
    logits = model(input_batch)
    return F.cross_entropy(logits.flatten(0, 1), target_batch.flatten())

def calc_loss_loader(data_loader, model, device, num_batches=None):
    total_loss = 0.
    if len(data_loader) == 0:
        return float("nan")
    num_batches = len(data_loader) if num_batches is None else min(num_batches, len(data_loader))
    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i < num_batches:
            total_loss += calc_loss_batch(input_batch, target_batch, model, device).item()
        else:
            break
    return total_loss / num_batches

def evaluate_model(model, train_loader, val_loader, device, eval_iter):
    model.eval()
    with torch.no_grad():
        train_loss = calc_loss_loader(train_loader, model, device, num_batches=eval_iter)
        val_loss = calc_loss_loader(val_loader, model, device, num_batches=eval_iter)
    model.train()
    return train_loss, val_loss

def generate_and_print_sample(model, tokenizer, device, start_context):
    model.eval()
    context_size = model.pos_emb.weight.shape[0]
    encoded = text_to_token_ids(start_context, tokenizer).to(device)
    with torch.no_grad():
        token_ids = generate_text_simple(model=model, idx=encoded, max_new_tokens=10, context_size=context_size)
    print(token_ids_to_text(token_ids, tokenizer).replace("\n", " "))
    model.train()

# pretrain_train_loader / pretrain_val_loader / PRETRAIN_CFG / gpt2_tokenizer
# are still the objects built back in chapter 5 -- reused as-is here.
print(len(pretrain_train_loader), "training batches available")

### D.1 & D.2 — Learning rate warmup and cosine decay

**Warmup**: start the learning rate very low (`initial_lr`) and ramp it *linearly* up to the target (`peak_lr`) over the first `warmup_steps` steps, instead of using the full learning rate from step 0. Large weight updates on a randomly-initialized model are exactly when training is most likely to destabilize (huge gradients, loss spikes) — warmup avoids ever taking a large step before the model has had a chance to settle into a reasonable region of parameter space.

**Cosine decay**: *after* warmup, instead of holding the learning rate constant, decay it smoothly along a half-cosine curve down to a small `min_lr` by the end of training. Smaller steps late in training reduce the risk of overshooting a good minimum once the model is already close to one.

<img src="https://raw.githubusercontent.com/cleophasmashiri/ai-jupter-notebooks/main/llm_from_scratch/images/25-learning-rate-schedule-warmup-then-cosine-decay.png" alt="Learning rate schedule: warmup then cosine decay" style="max-width:100%;height:auto;display:block;margin:1em auto" width="712" height="162"/>

<img src="https://raw.githubusercontent.com/cleophasmashiri/ai-jupter-notebooks/main/llm_from_scratch/images/26-learning-rate-warmup-followed-by-cosine-decay-plot.png" alt="Learning rate warmup followed by cosine decay, plotted over training steps" style="max-width:100%;height:auto;display:block;margin:1em auto" width="620"/>

### Exercise 1 — Learning rate warmup

Compute the learning rate at each training step: linearly interpolate from `initial_lr` to `peak_lr` over `warmup_steps` steps, then hold at `peak_lr` for every step after.

In [ ]:
# TODO: implement this exercise
n_epochs = 15
initial_lr = 0.0001
peak_lr = 0.01

total_steps = len(pretrain_train_loader) * n_epochs
warmup_steps = int(0.2 * total_steps)
print("warmup_steps:", warmup_steps)

lr_increment = (peak_lr - initial_lr) / warmup_steps

track_lrs = []
global_step = -1
for epoch in range(n_epochs):
    for input_batch, target_batch in pretrain_train_loader:
        global_step += 1
        if global_step < warmup_steps:
            lr = ...  # initial_lr + global_step * lr_increment  -- linear ramp
        else:
            lr = ...  # peak_lr  -- held constant after warmup
        track_lrs.append(lr)

print(track_lrs[:5])
print(track_lrs[warmup_steps])

In [ ]:
%%ipytest -qq

def test_warmup_starts_at_initial_lr():
    assert abs(track_lrs[0] - initial_lr) < 1e-9

def test_warmup_reaches_peak_lr_exactly_at_warmup_steps():
    assert abs(track_lrs[warmup_steps] - peak_lr) < 1e-9

def test_warmup_is_monotonically_increasing():
    assert all(track_lrs[i] <= track_lrs[i + 1] + 1e-12 for i in range(warmup_steps))

def test_lr_constant_after_warmup():
    assert all(abs(lr - peak_lr) < 1e-9 for lr in track_lrs[warmup_steps:])

<button id="claim-ex46" onclick="window.gamifyClaim('ex46', 4, this)" style="padding:6px 14px;border-radius:8px;border:1.5px solid #16a34a;background:#f0fdf4;color:#15803d;font-weight:600;font-size:13px;cursor:pointer;font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Helvetica,Arial,sans-serif">
✅ Mark passed (+4 pts, 4 tests)
</button>

<details ontoggle="window.gamifyReveal(this, 'ex46', 4)"><summary>Show solution (4 tests, −4 pts if opened)</summary>

```python
n_epochs = 15
initial_lr = 0.0001
peak_lr = 0.01

total_steps = len(pretrain_train_loader) * n_epochs
warmup_steps = int(0.2 * total_steps)
print("warmup_steps:", warmup_steps)

lr_increment = (peak_lr - initial_lr) / warmup_steps

track_lrs = []
global_step = -1
for epoch in range(n_epochs):
    for input_batch, target_batch in pretrain_train_loader:
        global_step += 1
        if global_step < warmup_steps:
            lr = initial_lr + global_step * lr_increment
        else:
            lr = peak_lr
        track_lrs.append(lr)

print(track_lrs[:5])
print(track_lrs[warmup_steps])
```

</details>

### Exercise 2 — Cosine decay after warmup

Extend the same loop: once past `warmup_steps`, replace the constant `peak_lr` with a half-cosine decay down to `min_lr` by the final training step.

In [ ]:
# TODO: implement this exercise
import math

min_lr = 0.1 * initial_lr
total_training_steps = total_steps

track_lrs_cosine = []
global_step = -1
for epoch in range(n_epochs):
    for input_batch, target_batch in pretrain_train_loader:
        global_step += 1
        if global_step < warmup_steps:
            lr = initial_lr + global_step * lr_increment
        else:
            progress = ...  # (global_step - warmup_steps) / (total_training_steps - warmup_steps)
            lr = ...          # min_lr + (peak_lr - min_lr) * 0.5 * (1 + cos(pi * progress))
        track_lrs_cosine.append(lr)

print(track_lrs_cosine[warmup_steps])
print(track_lrs_cosine[-1])

In [ ]:
%%ipytest -qq

def test_cosine_peak_at_warmup_boundary():
    assert abs(track_lrs_cosine[warmup_steps] - peak_lr) < 1e-6

def test_cosine_decays_to_near_min_lr_by_the_end():
    assert abs(track_lrs_cosine[-1] - min_lr) < 1e-3

def test_cosine_is_monotonically_decreasing_after_warmup():
    tail = track_lrs_cosine[warmup_steps:]
    assert all(tail[i] >= tail[i + 1] - 1e-9 for i in range(len(tail) - 1))

def test_cosine_matches_warmup_during_warmup_phase():
    assert track_lrs_cosine[:warmup_steps] == track_lrs[:warmup_steps]

<button id="claim-ex47" onclick="window.gamifyClaim('ex47', 4, this)" style="padding:6px 14px;border-radius:8px;border:1.5px solid #16a34a;background:#f0fdf4;color:#15803d;font-weight:600;font-size:13px;cursor:pointer;font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Helvetica,Arial,sans-serif">
✅ Mark passed (+4 pts, 4 tests)
</button>

<details ontoggle="window.gamifyReveal(this, 'ex47', 4)"><summary>Show solution (4 tests, −4 pts if opened)</summary>

```python
import math

min_lr = 0.1 * initial_lr
total_training_steps = total_steps

track_lrs_cosine = []
global_step = -1
for epoch in range(n_epochs):
    for input_batch, target_batch in pretrain_train_loader:
        global_step += 1
        if global_step < warmup_steps:
            lr = initial_lr + global_step * lr_increment
        else:
            progress = (global_step - warmup_steps) / (total_training_steps - warmup_steps)
            lr = min_lr + (peak_lr - min_lr) * 0.5 * (1 + math.cos(math.pi * progress))
        track_lrs_cosine.append(lr)

print(track_lrs_cosine[warmup_steps])
print(track_lrs_cosine[-1])
```

</details>

### D.3 — Gradient clipping

Occasionally a batch produces an unusually large gradient — a large update in a bad direction can knock the model out of a good region of parameter space it had been converging toward ("exploding gradients"). **Gradient clipping** rescales the *entire* gradient (treating all parameters' gradients together as one big vector) so its L2 norm never exceeds `max_norm`, without changing its *direction* — large gradients get shrunk, small ones pass through unchanged.

### Exercise 3 — Gradient clipping

Implement `find_highest_gradient` (book listing, scans every parameter's `.grad` for the single largest value), use it to inspect gradients before and after `torch.nn.utils.clip_grad_norm_`, and confirm clipping shrinks the maximum.

In [ ]:
# TODO: implement this exercise
def find_highest_gradient(model):
    max_grad = None
    for param in model.parameters():
        if param.grad is not None:
            grad_values = param.grad.data.flatten()
            max_grad_param = grad_values.max()
            if max_grad is None or max_grad_param > max_grad:
                max_grad = ...  # max_grad_param
    return max_grad

torch.manual_seed(123)
clip_demo_model = GPTModel(PRETRAIN_CFG)
demo_x, demo_y = next(iter(pretrain_train_loader))
loss = calc_loss_batch(demo_x, demo_y, clip_demo_model, "cpu")
loss.backward()

grad_before = find_highest_gradient(clip_demo_model)
print("before clipping:", grad_before)

torch.nn.utils.clip_grad_norm_(clip_demo_model.parameters(), max_norm=1.0)
grad_after = ...  # find_highest_gradient(clip_demo_model)
print("after clipping:", grad_after)

In [ ]:
%%ipytest -qq

def test_gradients_exist_before_clipping():
    assert grad_before is not None
    assert grad_before.item() > 0

def test_clipping_does_not_increase_the_max_gradient():
    assert grad_after <= grad_before

def test_clip_grad_norm_actually_rescales_when_norm_exceeds_max():
    torch.manual_seed(123)
    m = GPTModel(PRETRAIN_CFG)
    x, y = next(iter(pretrain_train_loader))
    loss = calc_loss_batch(x, y, m, "cpu")
    loss.backward()
    total_norm_before = torch.sqrt(sum((p.grad.data ** 2).sum() for p in m.parameters() if p.grad is not None))
    torch.nn.utils.clip_grad_norm_(m.parameters(), max_norm=1.0)
    total_norm_after = torch.sqrt(sum((p.grad.data ** 2).sum() for p in m.parameters() if p.grad is not None))
    if total_norm_before > 1.0:
        assert torch.isclose(total_norm_after, torch.tensor(1.0), atol=1e-3)

<button id="claim-ex48" onclick="window.gamifyClaim('ex48', 3, this)" style="padding:6px 14px;border-radius:8px;border:1.5px solid #16a34a;background:#f0fdf4;color:#15803d;font-weight:600;font-size:13px;cursor:pointer;font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Helvetica,Arial,sans-serif">
✅ Mark passed (+3 pts, 3 tests)
</button>

<details ontoggle="window.gamifyReveal(this, 'ex48', 3)"><summary>Show solution (3 tests, −3 pts if opened)</summary>

```python
def find_highest_gradient(model):
    max_grad = None
    for param in model.parameters():
        if param.grad is not None:
            grad_values = param.grad.data.flatten()
            max_grad_param = grad_values.max()
            if max_grad is None or max_grad_param > max_grad:
                max_grad = max_grad_param
    return max_grad

torch.manual_seed(123)
clip_demo_model = GPTModel(PRETRAIN_CFG)
demo_x, demo_y = next(iter(pretrain_train_loader))
loss = calc_loss_batch(demo_x, demo_y, clip_demo_model, "cpu")
loss.backward()

grad_before = find_highest_gradient(clip_demo_model)
print("before clipping:", grad_before)

torch.nn.utils.clip_grad_norm_(clip_demo_model.parameters(), max_norm=1.0)
grad_after = find_highest_gradient(clip_demo_model)
print("after clipping:", grad_after)
```

</details>

### D.4 — The modified training function

### Exercise 4 — train_model

Combine warmup, cosine decay, and gradient clipping into one training function — otherwise identical to chapter 5's `train_model_simple`. Gradient clipping is applied only *after* warmup (`global_step > warmup_steps`) — during warmup, steps are already deliberately small, so clipping isn't needed yet.

In [ ]:
# TODO: implement this exercise
def train_model(model, train_loader, val_loader, optimizer, device,
                 n_epochs, eval_freq, eval_iter, start_context, tokenizer,
                 warmup_steps, initial_lr=3e-05, min_lr=1e-6):
    train_losses, val_losses, track_tokens_seen, track_lrs = [], [], [], []
    tokens_seen, global_step = 0, -1
    peak_lr = optimizer.param_groups[0]["lr"]
    total_training_steps = len(train_loader) * n_epochs
    lr_increment = (peak_lr - initial_lr) / warmup_steps

    for epoch in range(n_epochs):
        model.train()
        for input_batch, target_batch in train_loader:
            optimizer.zero_grad()
            global_step += 1

            if global_step < warmup_steps:
                lr = initial_lr + global_step * lr_increment
            else:
                progress = (global_step - warmup_steps) / (total_training_steps - warmup_steps)
                lr = min_lr + (peak_lr - min_lr) * 0.5 * (1 + math.cos(math.pi * progress))
            for param_group in optimizer.param_groups:
                param_group["lr"] = lr
            track_lrs.append(lr)

            loss = calc_loss_batch(input_batch, target_batch, model, device)
            loss.backward()
            if global_step > warmup_steps:
                ...  # torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            tokens_seen += input_batch.numel()

            if global_step % eval_freq == 0:
                train_loss, val_loss = evaluate_model(model, train_loader, val_loader, device, eval_iter)
                train_losses.append(train_loss)
                val_losses.append(val_loss)
                track_tokens_seen.append(tokens_seen)
                print(f"Ep {epoch+1} (Iter {global_step:06d}): "
                      f"Train loss {train_loss:.3f}, Val loss {val_loss:.3f}")
        generate_and_print_sample(model, tokenizer, device, start_context)

    return train_losses, val_losses, track_tokens_seen, track_lrs

torch.manual_seed(123)
bells_cfg = dict(PRETRAIN_CFG)
bells_model = GPTModel(bells_cfg)
bells_optimizer = torch.optim.AdamW(bells_model.parameters(), lr=5e-4, weight_decay=0.1)

bells_train_losses, bells_val_losses, bells_tokens_seen, bells_track_lrs = train_model(
    bells_model, pretrain_train_loader, pretrain_val_loader, bells_optimizer, "cpu",
    n_epochs=2, eval_freq=5, eval_iter=1, start_context="Every effort moves you",
    tokenizer=gpt2_tokenizer, warmup_steps=3
)

In [ ]:
%%ipytest -qq

def test_bells_loss_decreases():
    assert bells_train_losses[-1] < bells_train_losses[0]

def test_bells_lr_schedule_has_expected_shape():
    assert len(bells_track_lrs) == len(pretrain_train_loader) * 2
    # ramps up through warmup...
    assert bells_track_lrs[0] < bells_track_lrs[3]
    # ...then decays afterward, ending near min_lr rather than staying at peak
    assert bells_track_lrs[-1] < max(bells_track_lrs)

def test_bells_gradient_clipping_kicks_in_after_warmup_only():
    # sanity check purely on the control-flow condition used above
    warmup_steps = 3
    assert not (2 > warmup_steps)  # step 2: still warming up, no clipping
    assert (5 > warmup_steps)      # step 5: past warmup, clipping applies

<button id="claim-ex49" onclick="window.gamifyClaim('ex49', 3, this)" style="padding:6px 14px;border-radius:8px;border:1.5px solid #16a34a;background:#f0fdf4;color:#15803d;font-weight:600;font-size:13px;cursor:pointer;font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Helvetica,Arial,sans-serif">
✅ Mark passed (+3 pts, 3 tests)
</button>

<details ontoggle="window.gamifyReveal(this, 'ex49', 3)"><summary>Show solution (3 tests, −3 pts if opened)</summary>

```python
def train_model(model, train_loader, val_loader, optimizer, device,
                 n_epochs, eval_freq, eval_iter, start_context, tokenizer,
                 warmup_steps, initial_lr=3e-05, min_lr=1e-6):
    train_losses, val_losses, track_tokens_seen, track_lrs = [], [], [], []
    tokens_seen, global_step = 0, -1
    peak_lr = optimizer.param_groups[0]["lr"]
    total_training_steps = len(train_loader) * n_epochs
    lr_increment = (peak_lr - initial_lr) / warmup_steps

    for epoch in range(n_epochs):
        model.train()
        for input_batch, target_batch in train_loader:
            optimizer.zero_grad()
            global_step += 1

            if global_step < warmup_steps:
                lr = initial_lr + global_step * lr_increment
            else:
                progress = (global_step - warmup_steps) / (total_training_steps - warmup_steps)
                lr = min_lr + (peak_lr - min_lr) * 0.5 * (1 + math.cos(math.pi * progress))
            for param_group in optimizer.param_groups:
                param_group["lr"] = lr
            track_lrs.append(lr)

            loss = calc_loss_batch(input_batch, target_batch, model, device)
            loss.backward()
            if global_step > warmup_steps:
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            tokens_seen += input_batch.numel()

            if global_step % eval_freq == 0:
                train_loss, val_loss = evaluate_model(model, train_loader, val_loader, device, eval_iter)
                train_losses.append(train_loss)
                val_losses.append(val_loss)
                track_tokens_seen.append(tokens_seen)
                print(f"Ep {epoch+1} (Iter {global_step:06d}): "
                      f"Train loss {train_loss:.3f}, Val loss {val_loss:.3f}")
        generate_and_print_sample(model, tokenizer, device, start_context)

    return train_losses, val_losses, track_tokens_seen, track_lrs

torch.manual_seed(123)
bells_cfg = dict(PRETRAIN_CFG)
bells_model = GPTModel(bells_cfg)
bells_optimizer = torch.optim.AdamW(bells_model.parameters(), lr=5e-4, weight_decay=0.1)

bells_train_losses, bells_val_losses, bells_tokens_seen, bells_track_lrs = train_model(
    bells_model, pretrain_train_loader, pretrain_val_loader, bells_optimizer, "cpu",
    n_epochs=2, eval_freq=5, eval_iter=1, start_context="Every effort moves you",
    tokenizer=gpt2_tokenizer, warmup_steps=3
)
```

</details>

### Checkpoint: Appendix D — training loop bells and whistles

1. Why apply warmup *before* cosine decay, rather than starting the cosine decay from step 0?

<details><summary>Show answer</summary>

Cosine decay's curve is smooth and gradual by design — starting it at step 0 would mean the very first, most destabilizing updates (on a randomly-initialized model) happen at close to the *peak* learning rate, since cosine decay only decreases slowly near its start. Warmup exists specifically to keep those earliest updates small; cosine decay's job is to *taper off* training later, a completely different concern operating on a different part of training.

</details>

2. Why does gradient clipping rescale the gradient's magnitude but not its direction?

<details><summary>Show answer</summary>

`clip_grad_norm_` divides the whole gradient vector by a scalar factor (`max_norm / current_norm`) whenever the norm exceeds `max_norm` — scalar multiplication changes a vector's length, never its direction. The point isn't "these gradients are pointing the wrong way," it's "this step is too large" — clipping keeps the model moving the same way the gradient says to, just more cautiously.

</details>

## Appendix E — Parameter-efficient fine-tuning with LoRA

Chapter 6 froze most of the model and trained only a handful of layers — still, every trainable layer there stored and updated a *full* weight matrix. **LoRA** (Low-Rank Adaptation) goes further: instead of updating a layer's weight matrix `W` directly, it learns a much smaller *approximation* of the update, and never touches `W` at all. The book demonstrates this against chapter 6's classification setup, but notes it applies equally to chapter 7's instruction fine-tuning — the technique is orthogonal to *what* task you fine-tune for.

### E.1 — Introduction to LoRA

In ordinary fine-tuning, training learns a full update matrix `ΔW` (same shape as `W`) and applies `W_new = W + ΔW`. LoRA instead learns two much *smaller* matrices, `A` (shape `in_dim × rank`) and `B` (shape `rank × out_dim`), and approximates `ΔW ≈ A @ B`. `rank` is a small number (the book uses 16) — far smaller than `in_dim`/`out_dim` — so `A` and `B` together hold vastly fewer parameters than a full `ΔW` would.

<img src="https://raw.githubusercontent.com/cleophasmashiri/ai-jupter-notebooks/main/llm_from_scratch/images/27-regular-fine-tuning-vs-lora-s-low-rank-weight-upda.png" alt="Regular fine-tuning vs. LoRA's low-rank weight update" style="max-width:100%;height:auto;display:block;margin:1em auto" width="772" height="332"/>

Because `W` is never modified, the same pretrained model can be reused for many different fine-tuning targets just by swapping in different (small, cheap-to-store) `A`/`B` pairs — no need to keep a full separately-fine-tuned copy of the model per task.

### Exercise 1 — LoRALayer

Implement the book's `LoRALayer` (listing E.5): `A` initialized with PyTorch's standard `kaiming_uniform_` scheme (the same one `nn.Linear` itself uses), `B` initialized to all **zeros**. `forward` computes `alpha * (x @ A @ B)`.

In [ ]:
# TODO: implement this exercise
import math

class LoRALayer(nn.Module):
    def __init__(self, in_dim, out_dim, rank, alpha):
        super().__init__()
        self.A = nn.Parameter(torch.empty(in_dim, rank))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = ...  # nn.Parameter(torch.zeros(rank, out_dim))
        self.alpha = alpha

    def forward(self, x):
        x = ...  # self.alpha * (x @ self.A @ self.B)
        return x

torch.manual_seed(0)
lora_demo = LoRALayer(in_dim=4, out_dim=6, rank=2, alpha=8)
lora_x = torch.randn(3, 4)
lora_out = lora_demo(lora_x)
print(lora_out)

In [ ]:
%%ipytest -qq

def test_lora_layer_output_is_zero_at_init():
    # B starts at all zeros, so A @ B is the zero matrix regardless of A's values
    assert torch.allclose(lora_out, torch.zeros_like(lora_out))

def test_lora_layer_shapes():
    m = LoRALayer(4, 6, rank=2, alpha=8)
    assert m.A.shape == (4, 2)
    assert m.B.shape == (2, 6)
    assert m(torch.randn(5, 4)).shape == (5, 6)

def test_lora_A_is_not_all_zeros_but_B_is():
    m = LoRALayer(4, 6, rank=2, alpha=8)
    assert not torch.allclose(m.A, torch.zeros_like(m.A))
    assert torch.equal(m.B, torch.zeros_like(m.B))

def test_lora_output_nonzero_once_B_is_trained():
    m = LoRALayer(4, 6, rank=2, alpha=8)
    with torch.no_grad():
        m.B += 0.1
    out = m(torch.randn(3, 4))
    assert not torch.allclose(out, torch.zeros_like(out))

<button id="claim-ex50" onclick="window.gamifyClaim('ex50', 4, this)" style="padding:6px 14px;border-radius:8px;border:1.5px solid #16a34a;background:#f0fdf4;color:#15803d;font-weight:600;font-size:13px;cursor:pointer;font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Helvetica,Arial,sans-serif">
✅ Mark passed (+4 pts, 4 tests)
</button>

<details ontoggle="window.gamifyReveal(this, 'ex50', 4)"><summary>Show solution (4 tests, −4 pts if opened)</summary>

```python
import math

class LoRALayer(nn.Module):
    def __init__(self, in_dim, out_dim, rank, alpha):
        super().__init__()
        self.A = nn.Parameter(torch.empty(in_dim, rank))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(rank, out_dim))
        self.alpha = alpha

    def forward(self, x):
        x = self.alpha * (x @ self.A @ self.B)
        return x

torch.manual_seed(0)
lora_demo = LoRALayer(in_dim=4, out_dim=6, rank=2, alpha=8)
lora_x = torch.randn(3, 4)
lora_out = lora_demo(lora_x)
print(lora_out)
```

</details>

Since `B` starts at zero, `LoRALayer` starts out contributing *exactly nothing* — this matters a lot for the next step: swapping it into a pretrained model must not change that model's behavior before any LoRA-specific training happens.

### Exercise 2 — LinearWithLoRA

Wrap an existing `nn.Linear` together with a `LoRALayer`: `forward` returns `self.linear(x) + self.lora(x)` — the original layer's output, plus LoRA's (initially zero) correction.

In [ ]:
# TODO: implement this exercise
class LinearWithLoRA(nn.Module):
    def __init__(self, linear, rank, alpha):
        super().__init__()
        self.linear = linear
        self.lora = LoRALayer(linear.in_features, linear.out_features, rank, alpha)

    def forward(self, x):
        return ...  # self.linear(x) + self.lora(x)

torch.manual_seed(0)
plain_linear = nn.Linear(4, 6)
wrapped_linear = LinearWithLoRA(plain_linear, rank=2, alpha=8)

lora_test_x = torch.randn(3, 4)
print(torch.allclose(wrapped_linear(lora_test_x), plain_linear(lora_test_x)))

In [ ]:
%%ipytest -qq

def test_linear_with_lora_matches_plain_linear_at_init():
    torch.manual_seed(0)
    lin = nn.Linear(4, 6)
    wrapped = LinearWithLoRA(lin, rank=2, alpha=8)
    x = torch.randn(5, 4)
    assert torch.allclose(wrapped(x), lin(x))

def test_linear_with_lora_diverges_once_B_is_perturbed():
    torch.manual_seed(0)
    lin = nn.Linear(4, 6)
    wrapped = LinearWithLoRA(lin, rank=2, alpha=8)
    x = torch.randn(5, 4)
    with torch.no_grad():
        wrapped.lora.B += 0.1
    assert not torch.allclose(wrapped(x), lin(x))

def test_linear_with_lora_preserves_original_linear_unchanged():
    torch.manual_seed(0)
    lin = nn.Linear(4, 6)
    original_weight = lin.weight.clone()
    wrapped = LinearWithLoRA(lin, rank=2, alpha=8)
    with torch.no_grad():
        wrapped.lora.B += 0.1
    assert torch.equal(wrapped.linear.weight, original_weight)

<button id="claim-ex51" onclick="window.gamifyClaim('ex51', 3, this)" style="padding:6px 14px;border-radius:8px;border:1.5px solid #16a34a;background:#f0fdf4;color:#15803d;font-weight:600;font-size:13px;cursor:pointer;font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Helvetica,Arial,sans-serif">
✅ Mark passed (+3 pts, 3 tests)
</button>

<details ontoggle="window.gamifyReveal(this, 'ex51', 3)"><summary>Show solution (3 tests, −3 pts if opened)</summary>

```python
class LinearWithLoRA(nn.Module):
    def __init__(self, linear, rank, alpha):
        super().__init__()
        self.linear = linear
        self.lora = LoRALayer(linear.in_features, linear.out_features, rank, alpha)

    def forward(self, x):
        return self.linear(x) + self.lora(x)

torch.manual_seed(0)
plain_linear = nn.Linear(4, 6)
wrapped_linear = LinearWithLoRA(plain_linear, rank=2, alpha=8)

lora_test_x = torch.randn(3, 4)
print(torch.allclose(wrapped_linear(lora_test_x), plain_linear(lora_test_x)))
```

</details>

### E.2–E.4 — Applying LoRA to the GPT model

The last step: swap every `nn.Linear` in an existing model (attention projections, feed-forward layers, the output head — everywhere) for a `LinearWithLoRA` wrapping it, recursively. The book applies this to a pretrained, GPT-2-weight-loaded model from chapter 6; this notebook applies the identical function to `GPTModel(TEST_CFG)` from chapter 4 to keep it fast, but nothing about `replace_linear_with_lora` cares which weights the model started with.

### Exercise 3 — replace_linear_with_lora

Recursively walk a model's children (`named_children()`); wherever a child is an `nn.Linear`, replace it in-place (`setattr`) with a `LinearWithLoRA` wrapping it; otherwise, recurse into that child (since it may itself contain `nn.Linear` layers nested further down).

In [ ]:
# TODO: implement this exercise
def replace_linear_with_lora(model, rank, alpha):
    for name, module in model.named_children():
        if isinstance(module, nn.Linear):
            setattr(model, name, ...)  # LinearWithLoRA(module, rank, alpha)
        else:
            replace_linear_with_lora(module, rank, alpha)  # recurse into non-Linear children

torch.manual_seed(123)
lora_gpt = GPTModel(TEST_CFG)

total_before = sum(p.numel() for p in lora_gpt.parameters() if p.requires_grad)
print(f"Trainable parameters before freezing: {total_before:,}")

for param in lora_gpt.parameters():
    param.requires_grad = False
total_frozen = sum(p.numel() for p in lora_gpt.parameters() if p.requires_grad)
print(f"Trainable parameters after freezing: {total_frozen:,}")

replace_linear_with_lora(lora_gpt, rank=4, alpha=8)
total_lora = sum(p.numel() for p in lora_gpt.parameters() if p.requires_grad)
print(f"Trainable LoRA parameters: {total_lora:,}")

In [ ]:
%%ipytest -qq

def test_freezing_zeroes_out_trainable_params():
    assert total_frozen == 0
    assert total_before > 0

def test_lora_adds_back_far_fewer_trainable_params_than_the_original_model():
    assert total_lora > 0
    assert total_lora < total_before

def test_no_bare_linear_layers_remain_unwrapped():
    for name, module in lora_gpt.named_modules():
        if isinstance(module, nn.Linear):
            assert name.endswith(".linear"), f"found an un-wrapped nn.Linear at {name}"

def test_model_still_produces_correctly_shaped_output_after_lora_injection():
    x = torch.randint(0, TEST_CFG["vocab_size"], (2, TEST_CFG["context_length"]))
    out = lora_gpt(x)
    assert out.shape == (2, TEST_CFG["context_length"], TEST_CFG["vocab_size"])

def test_lora_parameters_are_the_only_trainable_ones():
    trainable_names = [name for name, p in lora_gpt.named_parameters() if p.requires_grad]
    assert all(".lora." in name for name in trainable_names)

<button id="claim-ex52" onclick="window.gamifyClaim('ex52', 5, this)" style="padding:6px 14px;border-radius:8px;border:1.5px solid #16a34a;background:#f0fdf4;color:#15803d;font-weight:600;font-size:13px;cursor:pointer;font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Helvetica,Arial,sans-serif">
✅ Mark passed (+5 pts, 5 tests)
</button>

<details ontoggle="window.gamifyReveal(this, 'ex52', 5)"><summary>Show solution (5 tests, −5 pts if opened)</summary>

```python
def replace_linear_with_lora(model, rank, alpha):
    for name, module in model.named_children():
        if isinstance(module, nn.Linear):
            setattr(model, name, LinearWithLoRA(module, rank, alpha))
        else:
            replace_linear_with_lora(module, rank, alpha)

torch.manual_seed(123)
lora_gpt = GPTModel(TEST_CFG)

total_before = sum(p.numel() for p in lora_gpt.parameters() if p.requires_grad)
print(f"Trainable parameters before freezing: {total_before:,}")

for param in lora_gpt.parameters():
    param.requires_grad = False
total_frozen = sum(p.numel() for p in lora_gpt.parameters() if p.requires_grad)
print(f"Trainable parameters after freezing: {total_frozen:,}")

replace_linear_with_lora(lora_gpt, rank=4, alpha=8)
total_lora = sum(p.numel() for p in lora_gpt.parameters() if p.requires_grad)
print(f"Trainable LoRA parameters: {total_lora:,}")
```

</details>

This is the same freeze-then-selectively-unfreeze idea chapter 6 used (`requires_grad=False` on everything, then re-enable a small trainable slice) — LoRA is just a far more aggressive version of it: instead of choosing *which existing layers* to leave trainable, it adds a *new*, deliberately tiny, trainable piece alongside every layer, while every original weight — the entire pretrained model — stays frozen and untouched throughout fine-tuning.

### Checkpoint: Appendix E — LoRA

1. Why is `B` initialized to all zeros, but `A` initialized with the usual random `kaiming_uniform_` scheme — why not both zero, or both random?

<details><summary>Show answer</summary>

`A @ B` is the zero matrix whenever *either* `A` or `B` is all zeros — so initializing just one of them to zero is enough to guarantee `LinearWithLoRA` behaves identically to the original `nn.Linear` at the very start of training (test `test_linear_with_lora_matches_plain_linear_at_init` above confirms exactly this). Making both zero would also achieve that, but would leave `A`'s gradient trivially zero too during the first step (the gradient w.r.t. `A` involves a factor of `B`, which would be zero) — stalling learning at the very start. Randomly initializing `A` (matching how `nn.Linear` itself initializes its weights) avoids that, while zero-initializing `B` alone is still sufficient to guarantee a clean, behavior-preserving start.

</details>

2. `replace_linear_with_lora` recurses into `model.named_children()` rather than `model.named_modules()` (which would already give every nested layer in one flat list) — why walk it recursively by hand instead?

<details><summary>Show answer</summary>

`setattr(model, name, ...)` only works correctly on a module's *direct* children — you need a reference to the *parent* module to actually replace one of its attributes. `named_modules()` gives you every nested module flattened out, but not "and here's its direct parent, to call `setattr` on." Recursing through `named_children()` naturally keeps track of the correct parent at every level, so each `nn.Linear` gets replaced on the actual module that owns it as an attribute.

</details>

3. Chapter 6 fine-tuned by unfreezing the last transformer block, the final norm, and a new output head. Could you combine that approach with LoRA — e.g. LoRA-wrap only the attention layers, while still directly fine-tuning the output head?

<details><summary>Show answer</summary>

Yes — the two techniques aren't mutually exclusive, and mixing them is common in practice. `replace_linear_with_lora` recurses over the *whole* model here for simplicity, but nothing about it requires that; calling it on just `model.trf_blocks` (leaving `model.out_head` as an ordinary, directly-trainable `nn.Linear`) would LoRA-adapt the transformer layers while still fully fine-tuning the task-specific head — exactly the kind of per-layer strategy real fine-tuning setups tune based on their specific task and compute budget.

</details>

## Where to go from here

**Re-derive it from memory.** Each chapter's classes build on the last (`MultiHeadAttention` → `TransformerBlock` → `GPTModel` → fine-tuned variants) — try rewriting them in a plain `.py` file without looking back. That dependency chain is also why running this notebook top-to-bottom matters: chapter 7's exercises assume chapter 4's `GPTModel` and chapter 5's `generate_text_simple` are already defined in your kernel.

**Load real GPT-2 weights.** Every exercise above used small custom configs (`TEST_CFG`, `PRETRAIN_CFG`, `CLS_CFG`, `INSTR_CFG`) so cells run in seconds. The architecture is identical to real GPT-2 — section 5.5 of the book (and its companion code at [rasbt/LLMs-from-scratch](https://github.com/rasbt/LLMs-from-scratch)) shows `load_weights_into_gpt`, which copies OpenAI's actual released weights into this exact `GPTModel` class. Swap in `GPT_CONFIG_124M` (or the 355M/774M/1558M variants) with real weights, and every fine-tuning exercise above (chapters 6 and 7) applies unchanged — you're just starting from a model that already knows language, instead of one initialized from noise.

**Read the two papers this book distills.** *Attention Is All You Need* (Vaswani et al., 2017) for the original Transformer architecture chapter 3 derives; the GPT-2 and GPT-3 papers for the decoder-only, pre-norm variant implemented in chapter 4 onward.

**Combine what appendices D and E each added independently.** Both extend chapter 5's training loop without touching `GPTModel`'s architecture at all — appendix D changes *how* the optimizer steps (warmup, decay, clipping), appendix E changes *what's trainable* (a low-rank side path instead of the full weights). Nothing stops using both together: apply `replace_linear_with_lora` (appendix E) to a model, then fine-tune it with `train_model` (appendix D) instead of a plain training loop — production LoRA fine-tuning runs typically do exactly this.

If you've also worked through [`gpt_dev_tutorial.ipynb`](https://colab.research.google.com/github/cleophasmashiri/ai-jupter-notebooks/blob/main/gpt_dev_tutorial.ipynb), compare the two `GPTModel`/`BigramLanguageModel` implementations side by side — same architecture, arrived at from two different angles (Karpathy's minimal from-scratch derivation vs. this book's production-shaped, chapter-by-chapter software engineering). Recognizing that they're the same thing underneath is the real payoff of doing both.

### FAQ / common pitfalls

- **"My `GPTDatasetV1`/`InstructionDataset` batch has the wrong shape."** Check `max_length`/`allowed_max_length` first — a mismatch between what a dataset was built with and what a model's `context_length` expects is the single most common source of shape errors across chapters 2, 6, and 7.
- **"Loss is `nan` almost immediately."** In chapter 7 especially, check that `ignore_index=-100` was actually passed to `F.cross_entropy` — without it, `-100` gets treated as a real (wildly out-of-range) class index and blows up the loss.
- **"My causal attention weights aren't actually causal — token 0 attends to token 5."** Almost always a `torch.tril`/`torch.triu` mixup, or `diagonal=1` vs. the default `diagonal=0` — re-read the `masked_fill_` line in `CausalAttention`/`MultiHeadAttention` (chapter 3) carefully; `diagonal=1` is what excludes the diagonal itself (a token is always allowed to attend to itself).
- **"Fine-tuning (chapters 6/7) doesn't seem to be learning anything."** Check `requires_grad` — chapter 6 deliberately freezes most of the model, so if accuracy never moves, verify the *intended* trainable parameters (`out_head`, last block, final norm) actually have `requires_grad=True`, since a typo there silently trains nothing.
- **"Text generation repeats itself in a loop."** Expected from `generate_text_simple`'s greedy decoding (chapter 4) — always picking `argmax` has no mechanism to escape a locally-repetitive pattern once it starts. Switch to chapter 5's `generate` with `temperature > 0` and a `top_k`, which is exactly the problem those two knobs exist to fix.
- **"Why does everything reuse `calc_loss_batch`/`calc_loss_loader`/`train_model_simple` with only tiny tweaks across chapters 5, 6, and 7?"** That's deliberate on the book's part, not incidental — it's demonstrating that pretraining and fine-tuning are the *same* underlying loop (`zero_grad` → forward → loss → `backward` → `step`), just pointed at different data and, in chapter 6, a different loss target. Recognizing what's shared vs. what's task-specific is most of what there is to learn about the fine-tuning chapters.